<a href="https://colab.research.google.com/github/FrancoR72/GC-MS_compounds_first_alignment/blob/main/Copia_di_Copia_di_Untitled8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# CELLA 0 — CARICAMENTO DEL FILE GC-MS E SCELTA DELLA MODALITÀ
#
# Eseguire questa cella per PRIMA (è la prima cosa che compare
# lanciando "Esegui tutte"). Permette di:
#
#   1. caricare subito il file GC-MS di origine (con i fogli
#      GCMS_Areas, Compound_Mapping, metodo_analitico), che le
#      celle successive troveranno da sole in /content senza
#      richiederlo di nuovo;
#
#   2. scegliere la modalità "headspace" oppure "sample" tramite un
#      menu a tendina QUI, prima di lanciare l'esecuzione — la
#      cella 4 userà automaticamente questa scelta, senza più
#      fermarsi ad aspettare un input a metà notebook.
# =====================================================================

# @markdown ### Modalità di riferimento della concentrazione stimata
composition_choice = "sample"  # @param ["headspace", "sample"]

COMPOSITION_MODE = composition_choice

print("Modalità selezionata:", COMPOSITION_MODE)
if COMPOSITION_MODE == "headspace":
    print("-> Headspace in itself")
else:
    print("-> The solid/liquid sample that generated the headspace")

print()

from pathlib import Path
from google.colab import files

WORKING_DIRECTORY = Path("/content")

print("Caricare ora il file GC-MS di origine")
print(
    "(deve contenere i fogli GCMS_Areas, Compound_Mapping, "
    "metodo_analitico):"
)

uploaded = files.upload()

for filename in uploaded.keys():
    print("File caricato:", filename)

Modalità selezionata: sample
-> The solid/liquid sample that generated the headspace

Caricare ora il file GC-MS di origine
(deve contenere i fogli GCMS_Areas, Compound_Mapping, metodo_analitico):


Saving GCMS_Areas.xlsx to GCMS_Areas.xlsx
File caricato: GCMS_Areas.xlsx


In [2]:
# @title
# =====================================================================
# COSTRUZIONE AUTOMATICA DELLA CACHE SENSORIALE
# Versione corretta per una singola cella Google Colab
#
# INPUT:
#   file .xlsx contenente il foglio Compound_Mapping oppure un unico
#   foglio con almeno:
#       IUPAC_name
#       CAS
#
#   Colonna facoltativa:
#       Compound_role
#
# OUTPUT:
#   Sensory_Cache.xlsx
#
# Il codice:
#   1. legge la chiave API dai Secrets di Google Colab;
#   2. carica l'elenco dei composti;
#   3. verifica formalmente i numeri CAS;
#   4. controlla l'identità tramite PubChem;
#   5. ricerca online soglie olfattive in aria e descrittori;
#   6. converte le soglie in µg/m³;
#   7. crea un file Excel con dati, fonti e record da revisionare.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DELLE LIBRERIE
# =====================================================================

!pip -q install --upgrade openai pydantic xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
import json
import time
import getpass
from datetime import date
from itertools import zip_longest
from typing import Optional, List

import numpy as np
import pandas as pd
import requests

from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import files, userdata


# =====================================================================
# 2. PARAMETRI MODIFICABILI
# =====================================================================

# Modello OpenAI.
# gpt-5.6 è adatto alla ricerca web complessa.
MODEL = "gpt-5.6"

# Nome preferenziale del foglio di input.
INPUT_SHEET_PREFERRED = "Compound_Mapping"

# Nome del file prodotto.
OUTPUT_FILE = "Sensory_Cache.xlsx"

# Intestazioni richieste nel file Excel.
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

# Intestazione facoltativa per distinguere analiti e standard interno.
ROLE_COLUMN = "Compound_role"

# Pausa fra una ricerca e la successiva.
PAUSE_SECONDS = 1.0

# Numero massimo di tentativi per ogni composto.
MAX_RETRIES = 2

# Volume molare approssimato a 25 °C e 1 atm.
MOLAR_VOLUME_L_MOL = 24.45

# Famiglie sensoriali ammesse.
ALLOWED_FAMILIES = [
    "Cacao/Cioccolato",
    "Tostato",
    "Caffè",
    "Frutta secca",
    "Caramellato",
    "Dolce",
    "Fruttato",
    "Floreale",
    "Verde/Erbaceo",
    "Speziato",
    "Legnoso",
    "Affumicato",
    "Terroso",
    "Fungino",
    "Lattico",
    "Grasso/Ceroso",
    "Fermentato",
    "Solforato",
    "Animale",
    "Chimico/Solvente",
    "Fenolico/Medicinale",
    "Altro",
    "Non classificabile"
]


# =====================================================================
# 3. LETTURA DELLA CHIAVE API
# =====================================================================

# Prima prova a leggere il Secret di Colab.
try:
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

# Se il Secret non è disponibile, consente comunque l'inserimento manuale.
if not api_key:
    print(
        "Il Secret OPENAI_API_KEY non è stato trovato o non è accessibile."
    )
    api_key = getpass.getpass(
        "Inserire la chiave API OpenAI. "
        "La chiave non sarà visualizzata: "
    )

if not api_key or not api_key.strip():
    raise ValueError(
        "La chiave API OpenAI non è disponibile."
    )

client = OpenAI(api_key=api_key.strip())

print("Chiave API caricata correttamente.")


# =====================================================================
# 4. STRUTTURA DELLA RISPOSTA DEL MODELLO
# =====================================================================

class SensoryRecord(BaseModel):

    cas_input: str
    iupac_input: str

    identity_match: str = Field(
        description=(
            "Valori consentiti: confirmed, probable, "
            "conflicting, not_found"
        )
    )

    common_name: Optional[str] = None
    molecular_formula: Optional[str] = None
    molecular_weight_g_mol: Optional[float] = None

    threshold_air_found: bool = False

    threshold_air_value: Optional[float] = None
    threshold_air_unit_original: Optional[str] = None

    threshold_air_min_original: Optional[float] = None
    threshold_air_max_original: Optional[float] = None

    threshold_type: Optional[str] = Field(
        default=None,
        description=(
            "Detection, recognition, unspecified oppure null"
        )
    )

    threshold_medium: Optional[str] = Field(
        default=None,
        description="air oppure null"
    )

    threshold_conditions: Optional[str] = None

    odor_descriptors: List[str] = Field(
        default_factory=list
    )

    flavor_descriptors: List[str] = Field(
        default_factory=list
    )

    sensory_families: List[str] = Field(
        default_factory=list
    )

    threshold_source_name: Optional[str] = None
    threshold_source_url: Optional[str] = None
    threshold_reference: Optional[str] = None

    descriptor_source_names: List[str] = Field(
        default_factory=list
    )

    descriptor_source_urls: List[str] = Field(
        default_factory=list
    )

    confidence: str = Field(
        description="Valori consentiti: high, medium, low"
    )

    review_required: bool = False
    notes: Optional[str] = None


# =====================================================================
# 5. FUNZIONI DI PULIZIA E CONTROLLO
# =====================================================================

def clean_text(value):
    """
    Elimina spazi iniziali e finali mantenendo i valori mancanti.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return np.nan

    return text


def normalize_cas(value):
    """
    Normalizza la scrittura del numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    value = re.sub(r"\s+", "", value)

    return value


def validate_cas(cas_number):
    """
    Controlla:
    - formato del CAS;
    - cifra finale di controllo.
    """

    if pd.isna(cas_number):
        return False

    cas_number = str(cas_number).strip()

    if not re.fullmatch(
        r"\d{2,7}-\d{2}-\d",
        cas_number
    ):
        return False

    digits = cas_number.replace("-", "")

    body = digits[:-1]
    expected_check_digit = int(digits[-1])

    calculated_sum = sum(
        position * int(digit)
        for position, digit
        in enumerate(reversed(body), start=1)
    )

    calculated_check_digit = calculated_sum % 10

    return calculated_check_digit == expected_check_digit


def normalize_role(value):
    """
    Normalizza il ruolo dell'analita.
    """

    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def select_input_sheet(filename):
    """
    Seleziona:
    1. Compound_Mapping, se presente;
    2. l'unico foglio disponibile;
    3. genera errore se esistono più fogli e nessuno ha il nome atteso.
    """

    excel_file = pd.ExcelFile(filename)

    if INPUT_SHEET_PREFERRED in excel_file.sheet_names:
        return INPUT_SHEET_PREFERRED

    if len(excel_file.sheet_names) == 1:
        selected = excel_file.sheet_names[0]

        print(
            f"Il foglio '{INPUT_SHEET_PREFERRED}' non è presente. "
            f"Verrà utilizzato l'unico foglio disponibile: "
            f"'{selected}'."
        )

        return selected

    raise ValueError(
        f"Il foglio '{INPUT_SHEET_PREFERRED}' non è presente.\n"
        f"Fogli disponibili: {excel_file.sheet_names}"
    )


def safe_join(values, separator="; "):
    """
    Unisce una lista di stringhe eliminando valori vuoti e duplicati.
    """

    if not values:
        return ""

    cleaned = []

    for value in values:

        if value is None:
            continue

        text = str(value).strip()

        if text and text not in cleaned:
            cleaned.append(text)

    return separator.join(cleaned)


# =====================================================================
# 6. CONTROLLO DELL'IDENTITÀ MEDIANTE PUBCHEM
# =====================================================================

def get_pubchem_identity(cas_number):
    """
    Recupera da PubChem:
    - CID;
    - nome IUPAC;
    - formula molecolare;
    - massa molecolare.

    Se il CAS non viene trovato restituisce un dizionario vuoto.
    """

    encoded_cas = requests.utils.quote(
        str(cas_number),
        safe=""
    )

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded_cas}/property/"
        "IUPACName,MolecularFormula,MolecularWeight/JSON"
    )

    try:

        response = requests.get(
            url,
            timeout=30
        )

        if response.status_code == 404:
            return {}

        response.raise_for_status()

        payload = response.json()

        properties = (
            payload
            .get("PropertyTable", {})
            .get("Properties", [])
        )

        if not properties:
            return {}

        record = properties[0]

        cid = record.get("CID")

        return {
            "PubChem_CID": cid,
            "PubChem_IUPAC_name": record.get("IUPACName"),
            "PubChem_formula": record.get("MolecularFormula"),
            "PubChem_MW": record.get("MolecularWeight"),
            "PubChem_URL": (
                f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}"
                if cid is not None
                else None
            ),
            "PubChem_error": None
        }

    except Exception as error:

        return {
            "PubChem_CID": None,
            "PubChem_IUPAC_name": None,
            "PubChem_formula": None,
            "PubChem_MW": None,
            "PubChem_URL": None,
            "PubChem_error": str(error)
        }


# =====================================================================
# 7. NORMALIZZAZIONE DELLE UNITÀ
# =====================================================================

def normalize_unit_string(unit):
    """
    Normalizza varianti come:

    ng/L_air
    ng/L (air)
    ng/L air
    ng/lair
    µg/m³
    """

    if unit is None or pd.isna(unit):
        return None

    unit_clean = str(unit).lower().strip()

    replacements = {
        "μ": "µ",
        "³": "3",
        "_": "",
        "(": "",
        ")": "",
        "[": "",
        "]": "",
        "{": "",
        "}": "",
        " ": "",
        "litres": "l",
        "litre": "l",
        "liters": "l",
        "liter": "l",
        "cubicmetre": "m3",
        "cubicmeter": "m3",
        "m^3": "m3"
    }

    for old, new in replacements.items():
        unit_clean = unit_clean.replace(old, new)

    return unit_clean


def threshold_to_ug_m3(
    value,
    unit,
    molecular_weight=None
):
    """
    Converte una soglia olfattiva in aria in µg/m³.

    Conversioni supportate:
    - µg/m³
    - mg/m³
    - ng/m³
    - pg/m³
    - ng/L aria
    - µg/L aria
    - pg/L aria
    - ppb / ppbv
    - ppm / ppmv
    - ppt / pptv

    Per ppb, ppm e ppt serve la massa molecolare.
    """

    if value is None or unit is None:
        return np.nan

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if not np.isfinite(value):
        return np.nan

    unit_clean = normalize_unit_string(unit)

    if unit_clean is None:
        return np.nan

    # µg/m³
    if unit_clean in {
        "µg/m3",
        "ug/m3",
        "microgram/m3",
        "micrograms/m3"
    }:
        return value

    # mg/m³
    if unit_clean in {
        "mg/m3",
        "milligram/m3",
        "milligrams/m3"
    }:
        return value * 1000.0

    # ng/m³
    if unit_clean in {
        "ng/m3",
        "nanogram/m3",
        "nanograms/m3"
    }:
        return value / 1000.0

    # pg/m³
    if unit_clean in {
        "pg/m3",
        "picogram/m3",
        "picograms/m3"
    }:
        return value / 1_000_000.0

    # 1 ng/L = 1 µg/m³
    if unit_clean in {
        "ng/l",
        "ng/lair",
        "ng/l-air",
        "nanogram/l",
        "nanograms/l"
    }:
        return value

    # 1 µg/L = 1000 µg/m³
    if unit_clean in {
        "µg/l",
        "ug/l",
        "µg/lair",
        "ug/lair",
        "microgram/l",
        "micrograms/l"
    }:
        return value * 1000.0

    # 1 pg/L = 0,001 µg/m³
    if unit_clean in {
        "pg/l",
        "pg/lair",
        "picogram/l",
        "picograms/l"
    }:
        return value / 1000.0

    # Per le unità volumetriche è necessaria la massa molecolare.
    try:
        mw = float(molecular_weight)
    except (TypeError, ValueError):
        mw = np.nan

    if pd.isna(mw) or mw <= 0:
        return np.nan

    # ppbv → µg/m³
    if unit_clean in {
        "ppb",
        "ppbv",
        "partperbillion",
        "partsperbillion"
    }:
        return value * mw / MOLAR_VOLUME_L_MOL

    # ppmv → µg/m³
    if unit_clean in {
        "ppm",
        "ppmv",
        "partpermillion",
        "partspermillion"
    }:
        return value * mw * 1000.0 / MOLAR_VOLUME_L_MOL

    # pptv → µg/m³
    if unit_clean in {
        "ppt",
        "pptv",
        "partpertrillion",
        "partspertrillion"
    }:
        return value * mw / (
            MOLAR_VOLUME_L_MOL * 1000.0
        )

    return np.nan


# =====================================================================
# 8. FUNZIONE DI RICERCA ONLINE
# =====================================================================

def search_compound_online(
    cas_number,
    iupac_name,
    pubchem_data
):
    """
    Ricerca:
    - soglia olfattiva esplicitamente riferita all'aria;
    - descrittori olfattivi;
    - descrittori flavour;
    - famiglie sensoriali;
    - fonti.

    Restituisce un oggetto SensoryRecord.
    """

    pubchem_context = json.dumps(
        pubchem_data,
        ensure_ascii=False,
        indent=2
    )

    allowed_families_text = ", ".join(
        ALLOWED_FAMILIES
    )

    system_prompt = f"""
Sei un ricercatore esperto in chimica degli aromi,
HS-SPME-GC-MS, gascromatografia-olfattometria e soglie olfattive.

Devi compilare una scheda strutturata e documentabile per un composto.

FONTI DA PRIVILEGIARE:
1. pubblicazioni scientifiche originali;
2. compilazioni di van Gemert o Leffingwell;
3. The Good Scents Company;
4. PubChem;
5. FlavorDB e altre banche dati scientifiche pertinenti.

REGOLE OBBLIGATORIE:

- Verifica che il CAS e il nome fornito corrispondano alla stessa sostanza.
- Distingui chiaramente soglia in aria, acqua, olio, solvente
  e matrice alimentare.
- Accetta nel campo threshold_air esclusivamente un valore
  dichiarato esplicitamente come soglia olfattiva in aria.
- Non trasformare una soglia in acqua in una soglia in aria.
- Non usare come soglia la concentrazione impiegata per una
  valutazione descrittiva in solvente.
- Distingui detection threshold e recognition threshold.
- Conserva l'unità originale come riportata dalla fonte.
- Se esistono più valori in aria, scegli un valore rappresentativo
  soltanto se la scelta è difendibile.
- Quando possibile, registra anche minimo e massimo.
- Se la soglia in aria non è reperibile, imposta:
      threshold_air_found = false
      threshold_air_value = null
      threshold_air_unit_original = null
- Non inventare valori, fonti o URL.
- Gli URL devono essere reali e riferirsi alle pagine consultate.
- I descrittori devono essere brevi termini inglesi.
- Separa odor descriptors da flavor descriptors.
- Le famiglie sensoriali devono essere scelte esclusivamente da:
  {allowed_families_text}
- Imposta review_required = true quando:
    * le fonti sono discordanti;
    * l'identificazione è incerta;
    * il dato riguarda uno stereoisomero non specificato;
    * non è chiaro se la soglia sia in aria;
    * la fonte non è sufficientemente documentata.
"""

    user_prompt = f"""
COMPOSTO DA STUDIARE

Nome IUPAC fornito:
{iupac_name}

CAS fornito:
{cas_number}

Informazioni preliminari ottenute da PubChem:
{pubchem_context}

Ricercare sul web:

1. soglia olfattiva esplicitamente misurata in aria;
2. tipo di soglia: detection, recognition o unspecified;
3. condizioni sperimentali, se disponibili;
4. descrittori olfattivi;
5. descrittori aromatici o flavour;
6. famiglie sensoriali;
7. fonti e riferimenti.

Non inserire soglie in acqua nel campo threshold_air.
"""

    response = client.responses.parse(
        model=MODEL,
        tools=[
            {
                "type": "web_search",
                "filters": {
                    "allowed_domains": [
                        "pubmed.ncbi.nlm.nih.gov",
                        "pmc.ncbi.nlm.nih.gov",
                        "pubchem.ncbi.nlm.nih.gov",
                        "thegoodscentscompany.com",
                        "leffingwell.com",
                        "sciencedirect.com",
                        "acs.org",
                        "springer.com",
                        "wiley.com",
                        "tandfonline.com"
                    ]
                }
            }
        ],
        input=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        text_format=SensoryRecord
    )

    if response.output_parsed is None:
        raise ValueError(
            "La risposta API non contiene un record strutturato."
        )

    return response.output_parsed


# =====================================================================
# 9. RECUPERO DEL FILE DI INPUT (GIÀ CARICATO, OPPURE RICHIESTA)
# =====================================================================

from pathlib import Path

WORKING_DIRECTORY = Path("/content")


def find_existing_input_file(directory):
    """
    Cerca in /content un file Excel che contenga già il foglio
    Compound_Mapping (per esempio caricato dalla Cella 0), per
    evitare di richiederlo di nuovo.
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            if INPUT_SHEET_PREFERRED in excel_file.sheet_names:
                candidates.append(filepath)

        except Exception:
            continue

    if len(candidates) == 0:
        return None

    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    return candidates[0]


print()
print("=" * 72)
print("CARICAMENTO DEL FILE")
print("=" * 72)

existing_file = find_existing_input_file(WORKING_DIRECTORY)

if existing_file is not None:

    input_filename = str(existing_file)

    print(
        f"Trovato un file già presente in /content con il foglio "
        f"'{INPUT_SHEET_PREFERRED}': {existing_file.name}"
    )
    print("Verrà utilizzato questo file, senza richiederne un nuovo caricamento.")

else:

    print(
        "Nessun file con il foglio "
        f"'{INPUT_SHEET_PREFERRED}' trovato in /content."
    )
    print(
        "Caricare il file Excel contenente IUPAC_name e CAS."
    )

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "È necessario caricare un solo file Excel."
        )

    input_filename = next(iter(uploaded))

sheet_name = select_input_sheet(
    input_filename
)

print()
print("File utilizzato:", input_filename)
print("Foglio utilizzato:", sheet_name)

mapping_df = pd.read_excel(
    input_filename,
    sheet_name=sheet_name,
    dtype={CAS_COLUMN: str}
)


# =====================================================================
# 10. CONTROLLO DELLA STRUTTURA DEL FILE
# =====================================================================

required_columns = [
    IUPAC_COLUMN,
    CAS_COLUMN
]

missing_columns = [
    column
    for column in required_columns
    if column not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        "Mancano le colonne obbligatorie: "
        + ", ".join(missing_columns)
    )

mapping_df[IUPAC_COLUMN] = (
    mapping_df[IUPAC_COLUMN]
    .apply(clean_text)
)

mapping_df[CAS_COLUMN] = (
    mapping_df[CAS_COLUMN]
    .apply(normalize_cas)
)

mapping_df["CAS_valid"] = (
    mapping_df[CAS_COLUMN]
    .apply(validate_cas)
)


# =====================================================================
# 11. ESCLUSIONE DELLO STANDARD INTERNO
# =====================================================================

if ROLE_COLUMN in mapping_df.columns:

    normalized_roles = (
        mapping_df[ROLE_COLUMN]
        .apply(normalize_role)
    )

    internal_standard_labels = {
        "internal standard",
        "internalstandard",
        "standard interno",
        "internal std",
        "is"
    }

    analytes_df = mapping_df.loc[
        ~normalized_roles.isin(
            internal_standard_labels
        )
    ].copy()

else:
    analytes_df = mapping_df.copy()


# Elimina righe prive di identificativi.
analytes_df = analytes_df.loc[
    analytes_df[IUPAC_COLUMN].notna()
    & analytes_df[CAS_COLUMN].notna()
].copy()

# Elimina CAS duplicati.
analytes_df = analytes_df.drop_duplicates(
    subset=[CAS_COLUMN],
    keep="first"
).reset_index(drop=True)

if analytes_df.empty:
    raise ValueError(
        "Non sono presenti composti validi da elaborare."
    )

print()
print(
    f"Composti da elaborare: {len(analytes_df)}"
)

# =====================================================================
# 11B. RECUPERO DI UNA SENSORY_CACHE ESISTENTE (FACOLTATIVO)
#
# Se è già disponibile una Sensory_Cache da un'esecuzione precedente
# (anche rinominata da Colab in "Sensory_Cache (n).xlsx"), la ricerca
# online viene limitata ai soli composti non ancora presenti in
# quella cache. Il file viene poi aggiornato con i nuovi composti, e
# scaricato di nuovo solo se ne sono stati aggiunti.
# =====================================================================

print()
print("=" * 72)
print("RICERCA DI UNA SENSORY_CACHE GIÀ ESISTENTE")
print("=" * 72)

existing_cache_df = None


def find_existing_sensory_cache(directory):
    """
    Cerca in /content un file Sensory_Cache*.xlsx contenente il
    foglio Sensory_Cache con una colonna CAS.
    """

    candidates = []

    for filepath in directory.glob("Sensory_Cache*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            if "Sensory_Cache" not in excel_file.sheet_names:
                continue

            candidate_df = pd.read_excel(
                filepath, sheet_name="Sensory_Cache"
            )

            if CAS_COLUMN not in candidate_df.columns:
                continue

            candidates.append((filepath, candidate_df))

        except Exception:
            continue

    if len(candidates) == 0:
        return None, None

    candidates = sorted(
        candidates,
        key=lambda item: item[0].stat().st_mtime,
        reverse=True
    )

    return candidates[0]

from pathlib import Path

WORKING_DIRECTORY = Path("/content")

existing_cache_path, existing_cache_df = find_existing_sensory_cache(
    WORKING_DIRECTORY
)

if existing_cache_df is not None:

    print(
        "Trovata una Sensory_Cache già presente in /content: "
        f"{existing_cache_path.name}"
    )

else:

    print("Nessuna Sensory_Cache trovata automaticamente in /content.")
    print(
        "Se disponi già di una Sensory_Cache (n).xlsx da "
        "un'esecuzione precedente, caricala ora per limitare la "
        "ricerca ai soli composti nuovi. Se non la carichi (premi "
        "'Annulla'/'Cancel upload'), verranno elaborati tutti i "
        "composti."
    )

    uploaded_cache = files.upload()

    if len(uploaded_cache) > 0:

        cache_filename = next(iter(uploaded_cache))

        try:
            candidate_df = pd.read_excel(
                cache_filename, sheet_name="Sensory_Cache"
            )

            if CAS_COLUMN not in candidate_df.columns:
                raise ValueError(
                    f"Il file caricato non contiene la colonna "
                    f"'{CAS_COLUMN}' nel foglio 'Sensory_Cache'."
                )

            existing_cache_df = candidate_df

            print("Sensory_Cache caricata:", cache_filename)

        except Exception as error:
            print(
                f"Impossibile leggere il file caricato come "
                f"Sensory_Cache valida: {error}"
            )
            print("Si procede senza cache preesistente.")
            existing_cache_df = None

    else:
        print("Nessun file caricato: si procede senza cache preesistente.")


if existing_cache_df is not None:

    existing_cache_df[CAS_COLUMN] = (
        existing_cache_df[CAS_COLUMN]
        .apply(normalize_cas)
    )

    already_cached_cas = set(
        existing_cache_df[CAS_COLUMN].dropna()
    )

    total_before_filter = len(analytes_df)

    analytes_df = analytes_df.loc[
        ~analytes_df[CAS_COLUMN].isin(already_cached_cas)
    ].reset_index(drop=True)

    print()
    print(f"Composti totali nel file di input: {total_before_filter}")
    print(
        "Già presenti nella Sensory_Cache: "
        f"{total_before_filter - len(analytes_df)}"
    )
    print(f"Da elaborare in questa esecuzione: {len(analytes_df)}")

else:
    print()
    print(f"Composti da elaborare: {len(analytes_df)}")

# =====================================================================
# 12. TABELLE VUOTE PER I RISULTATI
# =====================================================================

records = []
threshold_source_rows = []
descriptor_source_rows = []


# =====================================================================
# 13. ELABORAZIONE DEI COMPOSTI
# =====================================================================

for index, row in analytes_df.iterrows():

    cas_number = row[CAS_COLUMN]
    iupac_name = row[IUPAC_COLUMN]
    cas_valid = bool(row["CAS_valid"])

    print()
    print("=" * 72)
    print(
        f"[{index + 1}/{len(analytes_df)}] "
        f"{iupac_name} — {cas_number}"
    )

    # ---------------------------------------------------------------
    # CAS non valido
    # ---------------------------------------------------------------

    if not cas_valid:

        print(
            "CAS formalmente non valido: "
            "la ricerca online non verrà eseguita."
        )

        records.append(
            {
                "IUPAC_name_input": iupac_name,
                "CAS": cas_number,
                "CAS_valid": False,
                "Identity_match": "conflicting",
                "Common_name": "",
                "Molecular_formula": "",
                "Molecular_weight_g_mol": np.nan,
                "Threshold_air_found": False,
                "Odor_threshold_air_value_original": np.nan,
                "Odor_threshold_air_unit_original": "",
                "Odor_threshold_air_min_original": np.nan,
                "Odor_threshold_air_max_original": np.nan,
                "Odor_threshold_air_ug_m3": np.nan,
                "Threshold_conversion_status": "Non eseguita",
                "Threshold_type": "",
                "Threshold_conditions": "",
                "Odor_descriptors": "",
                "Flavor_descriptors": "",
                "Sensory_family": "",
                "Threshold_source_name": "",
                "Threshold_source_url": "",
                "Threshold_reference": "",
                "Descriptor_source_names": "",
                "Descriptor_source_urls": "",
                "Confidence": "low",
                "Review_required": True,
                "Notes": "Numero CAS formalmente non valido",
                "PubChem_CID": np.nan,
                "PubChem_URL": "",
                "PubChem_error": "",
                "Retrieved_on": str(date.today())
            }
        )

        continue

    # ---------------------------------------------------------------
    # Controllo identità su PubChem
    # ---------------------------------------------------------------

    pubchem_data = get_pubchem_identity(
        cas_number
    )

    sensory = None
    last_error = None

    # ---------------------------------------------------------------
    # Ricerca API con tentativi automatici
    # ---------------------------------------------------------------

    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            if attempt > 1:
                print(
                    f"Nuovo tentativo API "
                    f"({attempt}/{MAX_RETRIES})..."
                )

            sensory = search_compound_online(
                cas_number=cas_number,
                iupac_name=iupac_name,
                pubchem_data=pubchem_data
            )

            break

        except Exception as error:

            last_error = error

            print(
                f"Errore API al tentativo "
                f"{attempt}/{MAX_RETRIES}: {error}"
            )

            if attempt < MAX_RETRIES:
                time.sleep(3)

    # ---------------------------------------------------------------
    # Errore definitivo
    # ---------------------------------------------------------------

    if sensory is None:

        records.append(
            {
                "IUPAC_name_input": iupac_name,
                "CAS": cas_number,
                "CAS_valid": True,
                "Identity_match": "not_found",
                "Common_name": "",
                "Molecular_formula": (
                    pubchem_data.get("PubChem_formula") or ""
                ),
                "Molecular_weight_g_mol": (
                    pubchem_data.get("PubChem_MW")
                ),
                "Threshold_air_found": False,
                "Odor_threshold_air_value_original": np.nan,
                "Odor_threshold_air_unit_original": "",
                "Odor_threshold_air_min_original": np.nan,
                "Odor_threshold_air_max_original": np.nan,
                "Odor_threshold_air_ug_m3": np.nan,
                "Threshold_conversion_status": "Non eseguita",
                "Threshold_type": "",
                "Threshold_conditions": "",
                "Odor_descriptors": "",
                "Flavor_descriptors": "",
                "Sensory_family": "",
                "Threshold_source_name": "",
                "Threshold_source_url": "",
                "Threshold_reference": "",
                "Descriptor_source_names": "",
                "Descriptor_source_urls": "",
                "Confidence": "low",
                "Review_required": True,
                "Notes": (
                    "Errore durante la ricerca online: "
                    f"{last_error}"
                ),
                "PubChem_CID": pubchem_data.get(
                    "PubChem_CID"
                ),
                "PubChem_URL": (
                    pubchem_data.get("PubChem_URL") or ""
                ),
                "PubChem_error": (
                    pubchem_data.get("PubChem_error") or ""
                ),
                "Retrieved_on": str(date.today())
            }
        )

        time.sleep(PAUSE_SECONDS)
        continue

    # ---------------------------------------------------------------
    # Massa molecolare
    # ---------------------------------------------------------------

    molecular_weight = (
        sensory.molecular_weight_g_mol
    )

    if molecular_weight is None:
        molecular_weight = pubchem_data.get(
            "PubChem_MW"
        )

    try:
        molecular_weight = float(
            molecular_weight
        )
    except (TypeError, ValueError):
        molecular_weight = np.nan

    # ---------------------------------------------------------------
    # Conversione soglia
    # ---------------------------------------------------------------

    threshold_ug_m3 = threshold_to_ug_m3(
        value=sensory.threshold_air_value,
        unit=sensory.threshold_air_unit_original,
        molecular_weight=molecular_weight
    )

    if not sensory.threshold_air_found:

        conversion_status = (
            "Soglia in aria non disponibile"
        )

    elif pd.notna(threshold_ug_m3):

        conversion_status = (
            "Conversione eseguita"
        )

    else:

        conversion_status = (
            "Unità non riconosciuta o massa molecolare mancante"
        )

        sensory.review_required = True

    # ---------------------------------------------------------------
    # Preparazione del record principale
    # ---------------------------------------------------------------

    odor_descriptors = safe_join(
        sensory.odor_descriptors
    )

    flavor_descriptors = safe_join(
        sensory.flavor_descriptors
    )

    sensory_families = safe_join(
        sensory.sensory_families,
        separator=" | "
    )

    descriptor_names = safe_join(
        sensory.descriptor_source_names
    )

    descriptor_urls = safe_join(
        sensory.descriptor_source_urls
    )

    records.append(
        {
            "IUPAC_name_input": iupac_name,
            "CAS": cas_number,
            "CAS_valid": True,

            "Identity_match": sensory.identity_match,
            "Common_name": sensory.common_name or "",

            "Molecular_formula": (
                sensory.molecular_formula
                or pubchem_data.get("PubChem_formula")
                or ""
            ),

            "Molecular_weight_g_mol": molecular_weight,

            "Threshold_air_found": (
                sensory.threshold_air_found
            ),

            "Odor_threshold_air_value_original": (
                sensory.threshold_air_value
            ),

            "Odor_threshold_air_unit_original": (
                sensory.threshold_air_unit_original or ""
            ),

            "Odor_threshold_air_min_original": (
                sensory.threshold_air_min_original
            ),

            "Odor_threshold_air_max_original": (
                sensory.threshold_air_max_original
            ),

            "Odor_threshold_air_ug_m3": threshold_ug_m3,

            "Threshold_conversion_status": (
                conversion_status
            ),

            "Threshold_type": (
                sensory.threshold_type or ""
            ),

            "Threshold_conditions": (
                sensory.threshold_conditions or ""
            ),

            "Odor_descriptors": odor_descriptors,
            "Flavor_descriptors": flavor_descriptors,
            "Sensory_family": sensory_families,

            "Threshold_source_name": (
                sensory.threshold_source_name or ""
            ),

            "Threshold_source_url": (
                sensory.threshold_source_url or ""
            ),

            "Threshold_reference": (
                sensory.threshold_reference or ""
            ),

            "Descriptor_source_names": (
                descriptor_names
            ),

            "Descriptor_source_urls": (
                descriptor_urls
            ),

            "Confidence": sensory.confidence,

            "Review_required": (
                sensory.review_required
            ),

            "Notes": sensory.notes or "",

            "PubChem_CID": pubchem_data.get(
                "PubChem_CID"
            ),

            "PubChem_URL": (
                pubchem_data.get("PubChem_URL") or ""
            ),

            "PubChem_error": (
                pubchem_data.get("PubChem_error") or ""
            ),

            "Retrieved_on": str(date.today())
        }
    )

    # ---------------------------------------------------------------
    # Tabella separata delle fonti delle soglie
    # ---------------------------------------------------------------

    if (
        sensory.threshold_source_name
        or sensory.threshold_source_url
    ):

        threshold_source_rows.append(
            {
                "CAS": cas_number,
                "IUPAC_name_input": iupac_name,
                "Threshold_value": (
                    sensory.threshold_air_value
                ),
                "Threshold_unit": (
                    sensory.threshold_air_unit_original
                ),
                "Threshold_value_ug_m3": threshold_ug_m3,
                "Threshold_min_original": (
                    sensory.threshold_air_min_original
                ),
                "Threshold_max_original": (
                    sensory.threshold_air_max_original
                ),
                "Threshold_type": sensory.threshold_type,
                "Threshold_medium": (
                    sensory.threshold_medium
                ),
                "Source_name": (
                    sensory.threshold_source_name
                ),
                "Source_URL": (
                    sensory.threshold_source_url
                ),
                "Reference": (
                    sensory.threshold_reference
                ),
                "Conditions": (
                    sensory.threshold_conditions
                ),
                "Retrieved_on": str(date.today())
            }
        )

    # ---------------------------------------------------------------
    # Tabella separata delle fonti dei descrittori
    # ---------------------------------------------------------------

    for source_name, source_url in zip_longest(
        sensory.descriptor_source_names,
        sensory.descriptor_source_urls,
        fillvalue=""
    ):

        if source_name or source_url:

            descriptor_source_rows.append(
                {
                    "CAS": cas_number,
                    "IUPAC_name_input": iupac_name,
                    "Source_name": source_name,
                    "Source_URL": source_url,
                    "Retrieved_on": str(date.today())
                }
            )

    # ---------------------------------------------------------------
    # Informazioni visualizzate durante l'esecuzione
    # ---------------------------------------------------------------

    print(
        "Soglia in aria:",
        sensory.threshold_air_value,
        sensory.threshold_air_unit_original
    )

    print(
        "Soglia convertita:",
        threshold_ug_m3,
        "µg/m³"
    )

    print(
        "Descrittori olfattivi:",
        odor_descriptors
    )

    if conversion_status != "Conversione eseguita":
        print(
            "ATTENZIONE:",
            conversion_status
        )

    time.sleep(PAUSE_SECONDS)


# =====================================================================
# 14. CREAZIONE DELLE TABELLE
# =====================================================================

new_records_df = pd.DataFrame(records)

if existing_cache_df is not None and not existing_cache_df.empty:

    if not new_records_df.empty:
        cache_df = pd.concat(
            [existing_cache_df, new_records_df],
            ignore_index=True,
            sort=False
        )
    else:
        cache_df = existing_cache_df.copy()

    cache_df = cache_df.drop_duplicates(
        subset=[CAS_COLUMN],
        keep="last"
    ).reset_index(drop=True)

else:
    cache_df = new_records_df

new_compounds_added = len(new_records_df) > 0

threshold_source_columns = [
    "CAS",
    "IUPAC_name_input",
    "Threshold_value",
    "Threshold_unit",
    "Threshold_value_ug_m3",
    "Threshold_min_original",
    "Threshold_max_original",
    "Threshold_type",
    "Threshold_medium",
    "Source_name",
    "Source_URL",
    "Reference",
    "Conditions",
    "Retrieved_on"
]

descriptor_source_columns = [
    "CAS",
    "IUPAC_name_input",
    "Source_name",
    "Source_URL",
    "Retrieved_on"
]

threshold_sources_df = pd.DataFrame(
    threshold_source_rows,
    columns=threshold_source_columns
)

descriptor_sources_df = pd.DataFrame(
    descriptor_source_rows,
    columns=descriptor_source_columns
)


# Record che richiedono controllo manuale.
review_mask = (
    cache_df["Review_required"].fillna(True)
    |
    ~cache_df["Threshold_air_found"].fillna(False)
    |
    cache_df["Odor_threshold_air_ug_m3"].isna()
    |
    (cache_df["Identity_match"] != "confirmed")
)

review_df = cache_df.loc[
    review_mask
].copy()


# Riepilogo.
summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "Composti caricati",
            "CAS formalmente validi",
            "Soglie in aria trovate",
            "Soglie convertite in µg/m³",
            "Record da revisionare",
            "Modello OpenAI utilizzato",
            "Data di elaborazione"
        ],
        "Valore": [
            len(cache_df),
            int(
                cache_df["CAS_valid"]
                .fillna(False)
                .sum()
            ),
            int(
                cache_df["Threshold_air_found"]
                .fillna(False)
                .sum()
            ),
            int(
                cache_df["Odor_threshold_air_ug_m3"]
                .notna()
                .sum()
            ),
            len(review_df),
            MODEL,
            str(date.today())
        ]
    }
)


# =====================================================================
# 15. SALVATAGGIO CON XLSXWRITER
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    cache_df.to_excel(
        writer,
        sheet_name="Sensory_Cache",
        index=False
    )

    threshold_sources_df.to_excel(
        writer,
        sheet_name="Threshold_Sources",
        index=False
    )

    descriptor_sources_df.to_excel(
        writer,
        sheet_name="Descriptor_Sources",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    analytes_df.to_excel(
        writer,
        sheet_name="Input_Compounds",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    # ---------------------------------------------------------------
    # Formattazione del workbook
    # ---------------------------------------------------------------

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    numeric_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Sensory_Cache": cache_df,
        "Threshold_Sources": threshold_sources_df,
        "Descriptor_Sources": descriptor_sources_df,
        "Manual_Review": review_df,
        "Input_Compounds": analytes_df,
        "Processing_Summary": summary_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)

        if len(dataframe.columns) > 0:

            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        # Intestazioni.
        for col_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                col_index,
                column_name,
                header_format
            )

            # Larghezza stimata.
            if dataframe.empty:
                max_length = len(str(column_name))
            else:
                values_length = (
                    dataframe[column_name]
                    .fillna("")
                    .astype(str)
                    .map(len)
                    .max()
                )

                max_length = max(
                    len(str(column_name)),
                    int(values_length)
                )

            width = min(
                max(max_length + 2, 12),
                45
            )

            worksheet.set_column(
                col_index,
                col_index,
                width,
                wrap_format
            )

        worksheet.set_default_row(30)

    # Formati specifici per il foglio principale.
    if "Sensory_Cache" in writer.sheets:

        sensory_ws = writer.sheets[
            "Sensory_Cache"
        ]

        for column_name in [
            "Odor_threshold_air_value_original",
            "Odor_threshold_air_min_original",
            "Odor_threshold_air_max_original",
            "Odor_threshold_air_ug_m3"
        ]:

            if column_name in cache_df.columns:

                col_index = cache_df.columns.get_loc(
                    column_name
                )

                sensory_ws.set_column(
                    col_index,
                    col_index,
                    18,
                    scientific_format
                )

        for column_name in [
            "Threshold_source_url",
            "Descriptor_source_urls",
            "PubChem_URL"
        ]:

            if column_name in cache_df.columns:

                col_index = cache_df.columns.get_loc(
                    column_name
                )

                sensory_ws.set_column(
                    col_index,
                    col_index,
                    40,
                    url_format
                )


# =====================================================================
# 16. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("ELABORAZIONE COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE)
print("Composti elaborati:", len(cache_df))

print(
    "Soglie in aria trovate:",
    int(
        cache_df["Threshold_air_found"]
        .fillna(False)
        .sum()
    )
)

print(
    "Soglie convertite in µg/m³:",
    int(
        cache_df["Odor_threshold_air_ug_m3"]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare manualmente:",
    len(review_df)
)

print()
print("Anteprima dei risultati:")

display_columns = [
    "IUPAC_name_input",
    "CAS",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    "Odor_threshold_air_ug_m3",
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required"
]

display(
    cache_df[
        [
            column
            for column in display_columns
            if column in cache_df.columns
        ]
    ]
)

if new_compounds_added:
    print()
    print(
        f"Sono stati aggiunti {len(new_records_df)} nuovi composti "
        "rispetto alla Sensory_Cache di partenza."
    )
    print("Download del file aggiornato...")
    files.download(OUTPUT_FILE)
else:
    print()
    print(
        "Nessun nuovo composto aggiunto rispetto alla Sensory_Cache "
        "già disponibile: il download non viene ripetuto."
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.3 MB/s eta 0:00:00
Chiave API caricata correttamente.

CARICAMENTO DEL FILE
Trovato un file già presente in /content con il foglio 'Compound_Mapping': GCMS_Areas.xlsx
Verrà utilizzato questo file, senza richiederne un nuovo caricamento.

File utilizzato: /content/GCMS_Areas.xlsx
Foglio utilizzato: Compound_Mapping

Composti da elaborare: 4

RICERCA DI UNA SENSORY_CACHE GIÀ ESISTENTE
Nessuna Sensory_Cache trovata automaticamente in /content.
Se disponi già di una Sensory_Cache (n).xlsx da un'esecuzione precedente, caricala ora per limitare la ricerca ai soli composti nuovi. Se non la carichi (premi 'Annulla'/'Cancel upload'), verranno elaborati tutti i composti.


Saving Sensory_Cache (17).xlsx to Sensory_Cache (17).xlsx
Sensory_Cache caricata: Sensory_Cache (17).xlsx

Composti totali nel file di input: 4
Già presenti nella Sensory_Cache: 4
Da elaborare in questa esecuzione: 0

ELABORAZIONE COMPLETATA
File creato: Sensory_Cache.xlsx
Composti elaborati: 4
Soglie in aria trovate: 4
Soglie convertite in µg/m³: 4
Record da controllare manualmente: 4

Anteprima dei risultati:


,IUPAC_name_input,CAS,Odor_threshold_air_value_original,Odor_threshold_air_unit_original,Odor_threshold_air_ug_m3,Threshold_type,Odor_descriptors,Flavor_descriptors,Sensory_family,Confidence,Review_required
0,2-methoxyphenol,90-05-1,0.084,ng/L_air,0.084000,unspecified,phenolic; smoky; spicy; medicinal; vanilla-lik...,woody; phenolic; bacon; savory; smoky; medicinal,Affumicato | Fenolico/Medicinale | Speziato | ...,medium,True
1,"3,7-dimethylocta-1,6-dien-3-ol",78-70-6,3.200,ng/L (air),3.200000,unspecified,citrus; floral; soapy; fresh; lemon-like; swee...,citrus; orange; lemon; floral; waxy; aldehydic...,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True
2,"2,3,5-trimethylpyrazine",14667-55-1,50.000,ng/L air,50.000000,unspecified,roasty; nutty; earthy; cocoa; musty; powdery; ...,raw nut skin; vegetable; cocoa; toasted; earth...,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium,True
3,phenol,108-95-2,0.006,ppm,23.094479,Detection,phenolic; plastic; rubbery; sweet; tarry; acri...,NaN,Fenolico/Medicinale | Chimico/Solvente | Dolce...,medium,True



Nessun nuovo composto aggiunto rispetto alla Sensory_Cache già disponibile: il download non viene ripetuto.


In [3]:
# @title
# =====================================================================
# SECONDA CELLA — COSTRUZIONE DELLA GCMS MASTER TABLE
#
# Questa cella deve essere eseguita nella stessa sessione Colab
# utilizzata per la prima cella.
#
# NON richiede di caricare nuovamente i file.
#
# INPUT già presenti in /content:
#   - file GC-MS contenente i fogli:
#       GCMS_Areas
#       Compound_Mapping
#
#   - Sensory_Cache.xlsx, creato dalla prima cella
#
# OUTPUT:
#   GCMS_Master_Table.xlsx
#
# Non vengono ancora calcolati IPA e LIPA.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL SOLO MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

AREAS_SHEET = "GCMS_Areas"
MAPPING_SHEET = "Compound_Mapping"
SENSORY_SHEET = "Sensory_Cache"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
SENSORY_CACHE_FILE = WORKING_DIRECTORY / "Sensory_Cache.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

MAPPING_NAME_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"
ROLE_COLUMN = "Compound_role"

THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def clean_text(value):
    """
    Restituisce una stringa pulita oppure NaN.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "" or text.lower() == "nan":
        return np.nan

    return text


def normalize_cas(value):
    """
    Uniforma trattini e spazi nel numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    return re.sub(r"\s+", "", value)


def normalize_role(value):
    """
    Uniforma le descrizioni del ruolo del composto.
    """
    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che una tabella contenga tutte le colonne richieste.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def find_gcms_source_file(directory):
    """
    Cerca nella cartella /content un file Excel contenente
    contemporaneamente i fogli GCMS_Areas e Compound_Mapping.

    Vengono esclusi:
    - Sensory_Cache.xlsx
    - GCMS_Master_Table.xlsx
    - file temporanei Excel
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        if filepath.name in {
            SENSORY_CACHE_FILE.name,
            OUTPUT_FILE.name
        }:
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            required_sheets = {
                AREAS_SHEET,
                MAPPING_SHEET
            }

            if required_sheets.issubset(
                set(excel_file.sheet_names)
            ):
                candidates.append(filepath)

        except Exception:
            # Il file non è leggibile come workbook Excel valido.
            continue

    if len(candidates) == 0:
        raise FileNotFoundError(
            "Non è stato trovato in /content alcun file Excel "
            f"contenente entrambi i fogli '{AREAS_SHEET}' e "
            f"'{MAPPING_SHEET}'.\n\n"
            "La seconda cella deve essere eseguita nella stessa "
            "sessione Colab della prima cella."
        )

    if len(candidates) == 1:
        return candidates[0]

    # Se esistono più copie, usa quella modificata più recentemente.
    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        "Sono stati trovati più file GC-MS compatibili."
    )
    print(
        "Verrà utilizzato il file modificato più recentemente:"
    )
    print(candidates[0].name)

    print()
    print("Altri file compatibili rilevati:")

    for filepath in candidates[1:]:
        print(" -", filepath.name)

    return candidates[0]


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(value_length)
    ) + 2

    return min(max(width, 12), maximum)


# =====================================================================
# 4. RICERCA AUTOMATICA DEI FILE GIÀ PRESENTI
# =====================================================================

print("=" * 72)
print("RICERCA DEI FILE NELLA SESSIONE COLAB")
print("=" * 72)

gcms_source_file = find_gcms_source_file(
    WORKING_DIRECTORY
)

if not SENSORY_CACHE_FILE.exists():
    raise FileNotFoundError(
        f"Il file '{SENSORY_CACHE_FILE.name}' non è presente "
        "nella cartella /content.\n\n"
        "Eseguire prima la cella 1 nella stessa sessione Colab "
        "e verificare che abbia creato Sensory_Cache.xlsx."
    )

print("File sorgente GC-MS:", gcms_source_file.name)
print("Cache sensoriale:", SENSORY_CACHE_FILE.name)


# =====================================================================
# 5. CONTROLLO DEI FOGLI
# =====================================================================

gcms_excel = pd.ExcelFile(
    gcms_source_file
)

sensory_excel = pd.ExcelFile(
    SENSORY_CACHE_FILE
)

if AREAS_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{AREAS_SHEET}'."
    )

if MAPPING_SHEET not in gcms_excel.sheet_names:
    raise ValueError(
        f"Nel file '{gcms_source_file.name}' manca il foglio "
        f"'{MAPPING_SHEET}'."
    )

if SENSORY_SHEET not in sensory_excel.sheet_names:
    raise ValueError(
        f"Nel file '{SENSORY_CACHE_FILE.name}' manca il foglio "
        f"'{SENSORY_SHEET}'."
    )

print()
print("Fogli del file GC-MS:", gcms_excel.sheet_names)
print("Fogli della cache:", sensory_excel.sheet_names)


# =====================================================================
# 6. LETTURA DEI DATI
# =====================================================================

areas_df = pd.read_excel(
    gcms_source_file,
    sheet_name=AREAS_SHEET
)

mapping_df = pd.read_excel(
    gcms_source_file,
    sheet_name=MAPPING_SHEET,
    dtype={CAS_COLUMN: str}
)

sensory_df = pd.read_excel(
    SENSORY_CACHE_FILE,
    sheet_name=SENSORY_SHEET,
    dtype={CAS_COLUMN: str}
)

print()
print("Dimensioni matrice delle aree:", areas_df.shape)
print("Dimensioni mapping:", mapping_df.shape)
print("Dimensioni cache sensoriale:", sensory_df.shape)


# =====================================================================
# 7. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

check_required_columns(
    areas_df,
    [
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    AREAS_SHEET
)

check_required_columns(
    mapping_df,
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN,
        ROLE_COLUMN
    ],
    MAPPING_SHEET
)

check_required_columns(
    sensory_df,
    [
        CAS_COLUMN,
        THRESHOLD_COLUMN
    ],
    SENSORY_SHEET
)


# =====================================================================
# 8. PULIZIA DEL MAPPING
# =====================================================================

mapping_df[MAPPING_NAME_COLUMN] = (
    mapping_df[MAPPING_NAME_COLUMN]
    .apply(clean_text)
)

mapping_df[IUPAC_COLUMN] = (
    mapping_df[IUPAC_COLUMN]
    .apply(clean_text)
)

mapping_df[CAS_COLUMN] = (
    mapping_df[CAS_COLUMN]
    .apply(normalize_cas)
)

mapping_df["Compound_role_normalized"] = (
    mapping_df[ROLE_COLUMN]
    .apply(normalize_role)
)


# =====================================================================
# 9. IDENTIFICAZIONE DELLO STANDARD INTERNO
# =====================================================================

internal_standard_labels = {
    "internal standard",
    "internalstandard",
    "standard interno",
    "internal std",
    "is"
}

internal_standard_rows = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        internal_standard_labels
    )
].copy()

if internal_standard_rows.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non è stato identificato "
        "alcuno standard interno."
    )

if len(internal_standard_rows) > 1:
    raise ValueError(
        "Nel foglio Compound_Mapping sono presenti più standard "
        "interni. Questa versione gestisce un solo standard interno."
    )

internal_standard_column = internal_standard_rows.iloc[0][
    MAPPING_NAME_COLUMN
]

if internal_standard_column not in areas_df.columns:
    raise ValueError(
        f"La colonna dello standard interno "
        f"'{internal_standard_column}' non è presente nel foglio "
        f"'{AREAS_SHEET}'."
    )

print()
print("Standard interno identificato:", internal_standard_column)


# =====================================================================
# 10. IDENTIFICAZIONE DEGLI ANALITI
# =====================================================================

analyte_labels = {
    "analyte",
    "analita",
    "compound",
    "voc"
}

analyte_mapping = mapping_df.loc[
    mapping_df["Compound_role_normalized"].isin(
        analyte_labels
    )
].copy()

if analyte_mapping.empty:
    raise ValueError(
        "Nel foglio Compound_Mapping non sono stati identificati "
        "analiti."
    )

analyte_mapping = analyte_mapping.drop_duplicates(
    subset=[MAPPING_NAME_COLUMN],
    keep="first"
)

analyte_columns = analyte_mapping[
    MAPPING_NAME_COLUMN
].tolist()

missing_area_columns = [
    column
    for column in analyte_columns
    if column not in areas_df.columns
]

if missing_area_columns:
    raise ValueError(
        "Le seguenti colonne indicate nel mapping non sono presenti "
        "nel foglio GCMS_Areas: "
        + ", ".join(missing_area_columns)
    )

print("Analiti identificati:", len(analyte_columns))

for compound in analyte_columns:
    print(" -", compound)
    # =====================================================================
# 11. CONTROLLO DELLE COLONNE NON MAPPATE
# =====================================================================

metadata_columns = {
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
}

mapped_area_columns = set(
    analyte_columns + [internal_standard_column]
)

unmapped_columns = [
    column
    for column in areas_df.columns
    if column not in metadata_columns
    and column not in mapped_area_columns
]

if unmapped_columns:
    print()
    print(
        "ATTENZIONE: le seguenti colonne non sono presenti nel "
        "Compound_Mapping e non verranno elaborate:"
    )

    for column in unmapped_columns:
        print(" -", column)


# =====================================================================
# 12. CONVERSIONE DELLE AREE IN VALORI NUMERICI
# =====================================================================

numeric_area_columns = (
    analyte_columns
    + [internal_standard_column]
)

for column in numeric_area_columns:

    original_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    areas_df[column] = pd.to_numeric(
        areas_df[column],
        errors="coerce"
    )

    converted_non_empty = (
        areas_df[column]
        .notna()
        .sum()
    )

    if converted_non_empty < original_non_empty:
        print(
            f"Attenzione: nella colonna '{column}' alcuni valori "
            "non numerici sono stati convertiti in dato mancante."
        )


# =====================================================================
# 13. PULIZIA DEGLI IDENTIFICATIVI DEI CAMPIONI
# =====================================================================

areas_df[SAMPLE_COLUMN] = (
    areas_df[SAMPLE_COLUMN]
    .apply(clean_text)
)

areas_df[REPLICATE_COLUMN] = pd.to_numeric(
    areas_df[REPLICATE_COLUMN],
    errors="coerce"
)

if areas_df[SAMPLE_COLUMN].isna().any():
    number_missing = int(
        areas_df[SAMPLE_COLUMN]
        .isna()
        .sum()
    )

    raise ValueError(
        f"Sono presenti {number_missing} righe prive di Sample_ID."
    )

if areas_df[REPLICATE_COLUMN].isna().any():
    print(
        "ATTENZIONE: alcune repliche sono mancanti o non numeriche."
    )


# =====================================================================
# 14. TRASFORMAZIONE DAL FORMATO LARGO AL FORMATO LUNGO
# =====================================================================

long_df = areas_df.melt(
    id_vars=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        internal_standard_column
    ],
    value_vars=analyte_columns,
    var_name=MAPPING_NAME_COLUMN,
    value_name="Area"
)

long_df = long_df.rename(
    columns={
        internal_standard_column: "IS_area"
    }
)

print()
print(
    "Righe create nella tabella lunga:",
    len(long_df)
)


# =====================================================================
# 15. ABBINAMENTO CON IL COMPOUND MAPPING
# =====================================================================

mapping_for_merge = analyte_mapping[
    [
        MAPPING_NAME_COLUMN,
        IUPAC_COLUMN,
        CAS_COLUMN
    ]
].copy()

master_df = long_df.merge(
    mapping_for_merge,
    on=MAPPING_NAME_COLUMN,
    how="left",
    validate="many_to_one"
)

if master_df[CAS_COLUMN].isna().any():

    missing_names = (
        master_df.loc[
            master_df[CAS_COLUMN].isna(),
            MAPPING_NAME_COLUMN
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        "Non è stato possibile associare il CAS alle colonne: "
        + ", ".join(missing_names)
    )


# =====================================================================
# 16. CALCOLO DELLE AREE NORMALIZZATE
# =====================================================================

valid_area = (
    master_df["Area"].notna()
    &
    (master_df["Area"] >= 0)
)

valid_internal_standard = (
    master_df["IS_area"].notna()
    &
    (master_df["IS_area"] > 0)
)

master_df["Normalized_area"] = np.where(
    valid_area & valid_internal_standard,
    master_df["Area"] / master_df["IS_area"],
    np.nan
)

master_df["Log10_normalized_area"] = np.where(
    master_df["Normalized_area"] > 0,
    np.log10(master_df["Normalized_area"]),
    np.nan
)


# =====================================================================
# 17. PULIZIA DELLA CACHE SENSORIALE
# =====================================================================

sensory_df[CAS_COLUMN] = (
    sensory_df[CAS_COLUMN]
    .apply(normalize_cas)
)

sensory_df[THRESHOLD_COLUMN] = pd.to_numeric(
    sensory_df[THRESHOLD_COLUMN],
    errors="coerce"
)

duplicated_cas = (
    sensory_df.loc[
        sensory_df[CAS_COLUMN].duplicated(keep=False),
        CAS_COLUMN
    ]
    .dropna()
    .unique()
    .tolist()
)

if duplicated_cas:
    print()
    print(
        "ATTENZIONE: nella cache sensoriale sono presenti "
        "CAS duplicati."
    )
    print(
        "Per ciascun CAS verrà utilizzata la prima riga."
    )

    for cas in duplicated_cas:
        print(" -", cas)

    sensory_df = sensory_df.drop_duplicates(
        subset=[CAS_COLUMN],
        keep="first"
    )


# =====================================================================
# 18. SELEZIONE DEI CAMPI SENSORIALI
# =====================================================================

preferred_sensory_columns = [
    CAS_COLUMN,
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

sensory_columns_to_merge = [
    column
    for column in preferred_sensory_columns
    if column in sensory_df.columns
]

sensory_for_merge = sensory_df[
    sensory_columns_to_merge
].copy()


# =====================================================================
# 19. ABBINAMENTO TRAMITE CAS
# =====================================================================

master_df = master_df.merge(
    sensory_for_merge,
    on=CAS_COLUMN,
    how="left",
    validate="many_to_one"
)


# =====================================================================
# 20. COLONNE DI CONTROLLO QUALITÀ
# =====================================================================

sensory_record_condition = (
    master_df[THRESHOLD_COLUMN].notna()
)

if "Odor_descriptors" in master_df.columns:
    sensory_record_condition = (
        sensory_record_condition
        |
        master_df["Odor_descriptors"].notna()
    )

master_df["Sensory_record_found"] = np.where(
    sensory_record_condition,
    "Sì",
    "No"
)

master_df["Normalized_area_calculable"] = np.where(
    master_df["Normalized_area"].notna(),
    "Sì",
    "No"
)

master_df["Processing_note"] = ""

master_df.loc[
    master_df["Area"].isna(),
    "Processing_note"
] = "Area analita mancante o non numerica"

master_df.loc[
    master_df["IS_area"].isna(),
    "Processing_note"
] = "Area standard interno mancante o non numerica"

master_df.loc[
    master_df["IS_area"].notna()
    & (master_df["IS_area"] <= 0),
    "Processing_note"
] = "Area standard interno uguale o inferiore a zero"

master_df.loc[
    master_df["Area"] == 0,
    "Processing_note"
] = "Composto non rilevato: area uguale a zero"

master_df.loc[
    master_df[THRESHOLD_COLUMN].isna()
    & (master_df["Processing_note"] == ""),
    "Processing_note"
] = "Soglia olfattiva in aria non disponibile"

master_df.loc[
    master_df[THRESHOLD_COLUMN].notna()
    & (master_df[THRESHOLD_COLUMN] <= 0),
    "Processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 21. ORDINAMENTO
# =====================================================================

master_df = master_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        MAPPING_NAME_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ]
).reset_index(drop=True)


# =====================================================================
# 22. ORDINE DELLE COLONNE
# =====================================================================

main_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area"
]

sensory_output_columns = [
    "Common_name",
    "Molecular_formula",
    "Molecular_weight_g_mol",
    "Threshold_air_found",
    "Odor_threshold_air_value_original",
    "Odor_threshold_air_unit_original",
    THRESHOLD_COLUMN,
    "Threshold_conversion_status",
    "Threshold_type",
    "Threshold_conditions",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Threshold_source_name",
    "Threshold_source_url",
    "Threshold_reference",
    "Descriptor_source_names",
    "Descriptor_source_urls",
    "Confidence",
    "Review_required",
    "Notes",
    "Retrieved_on"
]

quality_columns = [
    "Sensory_record_found",
    "Normalized_area_calculable",
    "Processing_note"
]

ordered_columns = (
    main_columns
    + [
        column
        for column in sensory_output_columns
        if column in master_df.columns
    ]
    + quality_columns
)

master_df = master_df[
    [
        column
        for column in ordered_columns
        if column in master_df.columns
    ]
]
# =====================================================================
# 23. TABELLA DI RIEPILOGO
# =====================================================================

summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "File GC-MS utilizzato",
            "Cache sensoriale utilizzata",
            "Numero di righe/iniezioni nel file GC-MS",
            "Numero di campioni distinti",
            "Numero di analiti",
            "Numero di righe nella Master Table",
            "Aree normalizzate calcolate",
            "Record con soglia in aria disponibile",
            "Record senza soglia in aria",
            "Standard interno utilizzato"
        ],
        "Valore": [
            gcms_source_file.name,
            SENSORY_CACHE_FILE.name,
            len(areas_df),
            areas_df[SAMPLE_COLUMN].nunique(),
            len(analyte_columns),
            len(master_df),
            int(
                master_df["Normalized_area"]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .notna()
                .sum()
            ),
            int(
                master_df[THRESHOLD_COLUMN]
                .isna()
                .sum()
            ),
            internal_standard_column
        ]
    }
)


# =====================================================================
# 24. TABELLA DI REVISIONE
# =====================================================================

review_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    THRESHOLD_COLUMN,
    "Confidence",
    "Review_required",
    "Processing_note"
]

review_columns = [
    column
    for column in review_columns
    if column in master_df.columns
]

review_mask = (
    master_df["Processing_note"] != ""
)

if "Review_required" in master_df.columns:
    review_mask = (
        review_mask
        |
        master_df["Review_required"]
        .fillna(False)
        .astype(bool)
    )

review_df = master_df.loc[
    review_mask,
    review_columns
].copy()


# =====================================================================
# 25. MATRICE DELLE AREE NORMALIZZATE
# =====================================================================

normalized_matrix = master_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=MAPPING_NAME_COLUMN,
    values="Normalized_area",
    aggfunc="first"
)


# =====================================================================
# 26. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    master_df.to_excel(
        writer,
        sheet_name="Master_Table",
        index=False
    )

    normalized_matrix.to_excel(
        writer,
        sheet_name="Normalized_Areas"
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    mapping_df.to_excel(
        writer,
        sheet_name="Mapping_Used",
        index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Master_Table": master_df,
        "Processing_Summary": summary_df,
        "Manual_Review": review_df,
        "Mapping_Used": mapping_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:
            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            column_width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                column_width,
                wrap_format
            )

    master_ws = writer.sheets["Master_Table"]

    for column_name in [
        "Area",
        "IS_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        "Normalized_area",
        "Log10_normalized_area"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if THRESHOLD_COLUMN in master_df.columns:

        column_index = master_df.columns.get_loc(
            THRESHOLD_COLUMN
        )

        master_ws.set_column(
            column_index,
            column_index,
            19,
            scientific_format
        )

    for column_name in [
        "Threshold_source_url",
        "Descriptor_source_urls"
    ]:
        if column_name in master_df.columns:

            column_index = master_df.columns.get_loc(
                column_name
            )

            master_ws.set_column(
                column_index,
                column_index,
                40,
                url_format
            )

    normalized_ws = writer.sheets[
        "Normalized_Areas"
    ]

    normalized_ws.freeze_panes(1, 2)


# =====================================================================
# 27. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("ELABORAZIONE COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)
print("File GC-MS utilizzato:", gcms_source_file.name)
print("Cache utilizzata:", SENSORY_CACHE_FILE.name)
print("Righe della Master Table:", len(master_df))
print("Campioni distinti:", areas_df[SAMPLE_COLUMN].nunique())
print("Analiti elaborati:", len(analyte_columns))

print(
    "Aree normalizzate calcolate:",
    int(
        master_df["Normalized_area"]
        .notna()
        .sum()
    )
)

print(
    "Record con soglia in aria:",
    int(
        master_df[THRESHOLD_COLUMN]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare:",
    len(review_df)
)

print()
print("Anteprima della Master Table:")

preview_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    MAPPING_NAME_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    "Normalized_area",
    "Log10_normalized_area",
    THRESHOLD_COLUMN,
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Processing_note"
]

preview_columns = [
    column
    for column in preview_columns
    if column in master_df.columns
]

display(
    master_df[
        preview_columns
    ].head(20)
)

files.download(
    str(OUTPUT_FILE)
)

RICERCA DEI FILE NELLA SESSIONE COLAB
File sorgente GC-MS: GCMS_Areas.xlsx
Cache sensoriale: Sensory_Cache.xlsx

Fogli del file GC-MS: ['GCMS_Areas', 'Compound_Mapping', 'metodo_analitico']
Fogli della cache: ['Sensory_Cache', 'Threshold_Sources', 'Descriptor_Sources', 'Manual_Review', 'Input_Compounds', 'Processing_Summary']

Dimensioni matrice delle aree: (6, 7)
Dimensioni mapping: (5, 4)
Dimensioni cache sensoriale: (4, 31)

Standard interno identificato: Internal_standard
Analiti identificati: 4
 - 2-methoxyphenol
 - linalool
 - 2,3,5-trimethylpyrazine
 - phenol

Righe create nella tabella lunga: 24

ELABORAZIONE COMPLETATA
File creato: GCMS_Master_Table.xlsx
File GC-MS utilizzato: GCMS_Areas.xlsx
Cache utilizzata: Sensory_Cache.xlsx
Righe della Master Table: 24
Campioni distinti: 2
Analiti elaborati: 4
Aree normalizzate calcolate: 24
Record con soglia in aria: 24
Record da controllare: 24

Anteprima della Master Table:


,Sample_ID,Replicate,GCMS_column_name,CAS,Area,IS_area,Normalized_area,Log10_normalized_area,Odor_threshold_air_ug_m3,Sensory_family,Confidence,Review_required,Processing_note
0,Cacao_01,1,"2,3,5-trimethylpyrazine",14667-55-1,845230,510000,1.657314,0.219405,50.000000,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium,True,
1,Cacao_01,1,2-methoxyphenol,90-05-1,125300,510000,0.245686,-0.609619,0.084000,Affumicato | Fenolico/Medicinale | Speziato | ...,medium,True,
2,Cacao_01,1,linalool,78-70-6,64900,510000,0.127255,-0.895325,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
3,Cacao_01,1,phenol,108-95-2,94900,510000,0.186078,-0.730304,23.094479,Fenolico/Medicinale | Chimico/Solvente | Dolce...,medium,True,
4,Cacao_01,2,"2,3,5-trimethylpyrazine",14667-55-1,861500,506000,1.702569,0.231105,50.000000,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium,True,
5,Cacao_01,2,2-methoxyphenol,90-05-1,128100,506000,0.253162,-0.596601,0.084000,Affumicato | Fenolico/Medicinale | Speziato | ...,medium,True,
6,Cacao_01,2,linalool,78-70-6,63200,506000,0.124901,-0.903433,3.200000,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium,True,
7,Cacao_01,2,phenol,108-95-2,93200,506000,0.184190,-0.734735,23.094479,Fenolico/Medicinale | Chimico/Solvente | Dolce...,medium,True,
8,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,837600,514000,1.629572,0.212074,50.000000,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium,True,
9,Cacao_01,3,2-methoxyphenol,90-05-1,121900,514000,0.237160,-0.624959,0.084000,Affumicato | Fenolico/Medicinale | Speziato | ...,medium,True,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# @title
# =====================================================================
# TERZA CELLA — CALCOLO DI IPA, LIPA E RANKING
#
# INPUT:
#   - variabile master_df prodotta dalla cella 2
#     oppure
#   - /content/GCMS_Master_Table.xlsx, foglio Master_Table
#
# OUTPUT:
#   /content/GCMS_IPA_Results.xlsx
#
# Nessun nuovo caricamento di file.
# Nessuna chiamata API.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DEL MOTORE EXCEL, SE NECESSARIO
# =====================================================================

!pip -q install xlsxwriter


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter

from google.colab import files


# =====================================================================
# 2. PARAMETRI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

MASTER_FILE = WORKING_DIRECTORY / "GCMS_Master_Table.xlsx"
MASTER_SHEET = "Master_Table"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_IPA_Results.xlsx"

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"

COMPOUND_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

NORMALIZED_AREA_COLUMN = "Normalized_area"
THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"

# Soglia convenzionale di riferimento.
# Deve avere la stessa unità delle soglie della cache.
REFERENCE_THRESHOLD_UG_M3 = 1.0


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che siano presenti tutte le colonne obbligatorie.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    maximum_value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(maximum_value_length)
    ) + 2

    return min(max(width, 12), maximum)


def classify_lipa(value):
    """
    Classificazione operativa preliminare.

    Non rappresenta una classificazione sensoriale assoluta:
    serve soltanto a facilitare la lettura e il filtraggio.
    """
    if pd.isna(value):
        return "Non calcolabile"

    if value >= 3:
        return "Estremamente elevata"

    if value >= 2:
        return "Molto elevata"

    if value >= 1:
        return "Elevata"

    if value >= 0:
        return "Moderata"

    if value >= -1:
        return "Bassa"

    return "Molto bassa"


def coefficient_of_variation(mean_value, sd_value):
    """
    Calcola CV% evitando divisioni per zero.
    """
    if (
        pd.isna(mean_value)
        or pd.isna(sd_value)
        or mean_value == 0
    ):
        return np.nan

    return 100.0 * sd_value / mean_value


# =====================================================================
# 4. RECUPERO DELLA MASTER TABLE
# =====================================================================

print("=" * 72)
print("RECUPERO DELLA MASTER TABLE")
print("=" * 72)

# Se master_df esiste già nella memoria della sessione,
# ne viene utilizzata una copia.
if (
    "master_df" in globals()
    and isinstance(master_df, pd.DataFrame)
    and not master_df.empty
):

    ipa_df = master_df.copy()

    print(
        "È stata utilizzata la variabile master_df "
        "presente nella memoria della sessione."
    )

else:

    if not MASTER_FILE.exists():
        raise FileNotFoundError(
            f"Non è disponibile la variabile master_df e non è stato "
            f"trovato il file '{MASTER_FILE.name}' in /content.\n\n"
            "Eseguire prima la cella 2 nella stessa sessione Colab."
        )

    master_excel = pd.ExcelFile(
        MASTER_FILE
    )

    if MASTER_SHEET not in master_excel.sheet_names:
        raise ValueError(
            f"Nel file '{MASTER_FILE.name}' manca il foglio "
            f"'{MASTER_SHEET}'."
        )

    ipa_df = pd.read_excel(
        MASTER_FILE,
        sheet_name=MASTER_SHEET,
        dtype={CAS_COLUMN: str}
    )

    print(
        f"È stato letto il file '{MASTER_FILE.name}', "
        f"foglio '{MASTER_SHEET}'."
    )

print("Righe disponibili:", len(ipa_df))


# =====================================================================
# 5. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

required_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

check_required_columns(
    ipa_df,
    required_columns,
    MASTER_SHEET
)


# =====================================================================
# 6. CONVERSIONE DELLE COLONNE NUMERICHE
# =====================================================================

numeric_columns = [
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN
]

for column in numeric_columns:
    ipa_df[column] = pd.to_numeric(
        ipa_df[column],
        errors="coerce"
    )

ipa_df[REPLICATE_COLUMN] = pd.to_numeric(
    ipa_df[REPLICATE_COLUMN],
    errors="coerce"
)


# =====================================================================
# 7. CONTROLLO DELLA SOGLIA DI RIFERIMENTO
# =====================================================================

if REFERENCE_THRESHOLD_UG_M3 <= 0:
    raise ValueError(
        "REFERENCE_THRESHOLD_UG_M3 deve essere maggiore di zero."
    )


# =====================================================================
# 8. VALIDITÀ DEI DATI PER IL CALCOLO
# =====================================================================

valid_normalized_area = (
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    &
    (ipa_df[NORMALIZED_AREA_COLUMN] > 0)
)

valid_threshold = (
    ipa_df[THRESHOLD_COLUMN].notna()
    &
    (ipa_df[THRESHOLD_COLUMN] > 0)
)

ipa_df["IPA_calculable"] = np.where(
    valid_normalized_area & valid_threshold,
    "Sì",
    "No"
)


# =====================================================================
# 9. CALCOLO DI IPA
# =====================================================================

ipa_df["IPA"] = np.where(
    valid_normalized_area & valid_threshold,

    ipa_df[NORMALIZED_AREA_COLUMN]
    * REFERENCE_THRESHOLD_UG_M3
    / ipa_df[THRESHOLD_COLUMN],

    np.nan
)


# =====================================================================
# 10. CALCOLO DI LIPA
# =====================================================================

ipa_df["LIPA"] = np.where(
    ipa_df["IPA"] > 0,
    np.log10(ipa_df["IPA"]),
    np.nan
)


# =====================================================================
# 11. SCOMPOSIZIONE DEL LIPA
#
# LIPA = contributo analitico + contributo della soglia
# =====================================================================

ipa_df["Log10_analytical_response"] = np.where(
    ipa_df[NORMALIZED_AREA_COLUMN] > 0,
    np.log10(
        ipa_df[NORMALIZED_AREA_COLUMN]
    ),
    np.nan
)

ipa_df["Log10_odor_potency"] = np.where(
    ipa_df[THRESHOLD_COLUMN] > 0,

    np.log10(
        REFERENCE_THRESHOLD_UG_M3
        / ipa_df[THRESHOLD_COLUMN]
    ),

    np.nan
)


# =====================================================================
# 12. CLASSE OPERATIVA
# =====================================================================

ipa_df["IPA_priority_class"] = (
    ipa_df["LIPA"].apply(classify_lipa)
)


# =====================================================================
# 13. NOTE SUL CALCOLO
# =====================================================================

ipa_df["IPA_processing_note"] = ""

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].isna(),
    "IPA_processing_note"
] = "Area normalizzata non disponibile"

ipa_df.loc[
    ipa_df[NORMALIZED_AREA_COLUMN].notna()
    & (ipa_df[NORMALIZED_AREA_COLUMN] <= 0),
    "IPA_processing_note"
] = "Area normalizzata uguale o inferiore a zero"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].isna(),
    "IPA_processing_note"
] = "Soglia olfattiva in aria non disponibile"

ipa_df.loc[
    ipa_df[THRESHOLD_COLUMN].notna()
    & (ipa_df[THRESHOLD_COLUMN] <= 0),
    "IPA_processing_note"
] = "Soglia olfattiva uguale o inferiore a zero"


# =====================================================================
# 14. RANKING ENTRO CIASCUNA REPLICA
# =====================================================================

ranking_group_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN
]

ipa_df["IPA_rank_replica"] = (
    ipa_df
    .groupby(
        ranking_group_columns,
        dropna=False
    )["LIPA"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

ipa_df["IPA_rank_replica"] = (
    ipa_df["IPA_rank_replica"]
    .astype("Int64")
)


# =====================================================================
# 15. ORDINAMENTO DELLA TABELLA A LIVELLO DI REPLICA
# =====================================================================

ipa_df = ipa_df.sort_values(
    by=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN,
        "IPA_rank_replica",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 16. RIEPILOGO PER CAMPIONE
#
# Le aree normalizzate vengono mediate tra le repliche.
# L'IPA del campione viene calcolato sulla media delle aree normalizzate.
# =====================================================================

sample_group_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN
]

aggregation_dictionary = {
    NORMALIZED_AREA_COLUMN: [
        "count",
        "mean",
        "std",
        "min",
        "max"
    ],
    THRESHOLD_COLUMN: "first"
}

# Conserva anche i principali dati sensoriali, se presenti.
optional_first_columns = [
    "Common_name",
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

for column in optional_first_columns:
    if column in ipa_df.columns:
        aggregation_dictionary[column] = "first"

sample_summary = (
    ipa_df
    .groupby(
        sample_group_columns,
        dropna=False
    )
    .agg(aggregation_dictionary)
    .reset_index()
)


# =====================================================================
# 17. APPIATTIMENTO DELLE INTESTAZIONI DEL RIEPILOGO
# =====================================================================

flattened_columns = []

for column in sample_summary.columns:

    if isinstance(column, tuple):

        first_part = str(column[0]).strip()
        second_part = str(column[1]).strip()

        if second_part:
            flattened_columns.append(
                f"{first_part}_{second_part}"
            )
        else:
            flattened_columns.append(first_part)

    else:
        flattened_columns.append(str(column))

sample_summary.columns = flattened_columns


# Rinomina le colonne principali.
rename_summary_columns = {
    f"{NORMALIZED_AREA_COLUMN}_count": "Number_of_replicates",
    f"{NORMALIZED_AREA_COLUMN}_mean": "Mean_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_std": "SD_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_min": "Min_normalized_area",
    f"{NORMALIZED_AREA_COLUMN}_max": "Max_normalized_area",
    f"{THRESHOLD_COLUMN}_first": THRESHOLD_COLUMN
}

for column in optional_first_columns:
    rename_summary_columns[
        f"{column}_first"
    ] = column

sample_summary = sample_summary.rename(
    columns=rename_summary_columns
)


# =====================================================================
# 18. COEFFICIENTE DI VARIAZIONE DELLE REPLICHE
# =====================================================================

sample_summary["CV_normalized_area_percent"] = (
    sample_summary.apply(
        lambda row: coefficient_of_variation(
            row["Mean_normalized_area"],
            row["SD_normalized_area"]
        ),
        axis=1
    )
)


# =====================================================================
# 19. IPA E LIPA SULLA MEDIA DELLE REPLICHE
# =====================================================================

valid_sample_mean = (
    sample_summary["Mean_normalized_area"].notna()
    &
    (sample_summary["Mean_normalized_area"] > 0)
)

valid_sample_threshold = (
    sample_summary[THRESHOLD_COLUMN].notna()
    &
    (sample_summary[THRESHOLD_COLUMN] > 0)
)

sample_summary["IPA_mean"] = np.where(
    valid_sample_mean & valid_sample_threshold,

    sample_summary["Mean_normalized_area"]
    * REFERENCE_THRESHOLD_UG_M3
    / sample_summary[THRESHOLD_COLUMN],

    np.nan
)

sample_summary["LIPA_mean"] = np.where(
    sample_summary["IPA_mean"] > 0,
    np.log10(
        sample_summary["IPA_mean"]
    ),
    np.nan
)

sample_summary["IPA_priority_class"] = (
    sample_summary["LIPA_mean"]
    .apply(classify_lipa)
)


# =====================================================================
# 20. RANKING ENTRO CIASCUN CAMPIONE
# =====================================================================

sample_summary["IPA_rank_sample"] = (
    sample_summary
    .groupby(
        SAMPLE_COLUMN,
        dropna=False
    )["LIPA_mean"]
    .rank(
        method="dense",
        ascending=False,
        na_option="bottom"
    )
)

sample_summary["IPA_rank_sample"] = (
    sample_summary["IPA_rank_sample"]
    .astype("Int64")
)


sample_summary = sample_summary.sort_values(
    by=[
        SAMPLE_COLUMN,
        "IPA_rank_sample",
        COMPOUND_COLUMN
    ],
    ascending=[
        True,
        True,
        True
    ],
    na_position="last"
).reset_index(drop=True)


# =====================================================================
# 21. MATRICE IPA PER REPLICA
# =====================================================================

ipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="IPA",
    aggfunc="first"
)


# =====================================================================
# 22. MATRICE LIPA PER REPLICA
# =====================================================================

lipa_matrix_replica = ipa_df.pivot_table(
    index=[
        SAMPLE_COLUMN,
        REPLICATE_COLUMN
    ],
    columns=COMPOUND_COLUMN,
    values="LIPA",
    aggfunc="first"
)


# =====================================================================
# 23. MATRICE IPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

ipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="IPA_mean",
    aggfunc="first"
)


# =====================================================================
# 24. MATRICE LIPA SULLA MEDIA DEL CAMPIONE
# =====================================================================

lipa_matrix_sample = sample_summary.pivot_table(
    index=SAMPLE_COLUMN,
    columns=COMPOUND_COLUMN,
    values="LIPA_mean",
    aggfunc="first"
)


# =====================================================================
# 25. TOP COMPOUNDS PER CAMPIONE
# =====================================================================

TOP_N = 20

top_compounds = sample_summary.loc[
    sample_summary["IPA_rank_sample"].notna()
    &
    (
        sample_summary["IPA_rank_sample"]
        <= TOP_N
    )
].copy()


# =====================================================================
# 26. RECORD NON CALCOLABILI
# =====================================================================

not_calculable = ipa_df.loc[
    ipa_df["IPA"].isna()
].copy()

not_calculable_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA_processing_note"
]

not_calculable_columns = [
    column
    for column in not_calculable_columns
    if column in not_calculable.columns
]

not_calculable = not_calculable[
    not_calculable_columns
]


# =====================================================================
# 27. RIEPILOGO DELL'ELABORAZIONE
# =====================================================================

processing_summary = pd.DataFrame(
    {
        "Indicatore": [
            "Soglia di riferimento T_r (µg/m³)",
            "Numero di righe analita-replica",
            "Numero di campioni distinti",
            "Numero di analiti distinti",
            "Valori IPA calcolati",
            "Valori IPA non calcolabili",
            "Numero di righe nel riepilogo campione",
            "Top N utilizzato"
        ],
        "Valore": [
            REFERENCE_THRESHOLD_UG_M3,
            len(ipa_df),
            ipa_df[SAMPLE_COLUMN].nunique(),
            ipa_df[CAS_COLUMN].nunique(),
            int(
                ipa_df["IPA"]
                .notna()
                .sum()
            ),
            int(
                ipa_df["IPA"]
                .isna()
                .sum()
            ),
            len(sample_summary),
            TOP_N
        ]
    }
)


# =====================================================================
# 28. ORDINE DELLE COLONNE DEL RISULTATO A LIVELLO DI REPLICA
# =====================================================================

main_replica_columns = [
    SAMPLE_COLUMN,
    REPLICATE_COLUMN,
    "IPA_rank_replica",
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    "Area",
    "IS_area",
    NORMALIZED_AREA_COLUMN,
    THRESHOLD_COLUMN,
    "IPA",
    "LIPA",
    "Log10_analytical_response",
    "Log10_odor_potency",
    "IPA_priority_class"
]

sensory_replica_columns = [
    "Threshold_type",
    "Odor_descriptors",
    "Flavor_descriptors",
    "Sensory_family",
    "Confidence",
    "Review_required",
    "Threshold_source_name",
    "Threshold_source_url"
]

quality_replica_columns = [
    "IPA_calculable",
    "IPA_processing_note"
]

ordered_replica_columns = (
    main_replica_columns
    + [
        column
        for column in sensory_replica_columns
        if column in ipa_df.columns
    ]
    + quality_replica_columns
)

# Conserva alla fine eventuali colonne originali non incluse sopra.
remaining_columns = [
    column
    for column in ipa_df.columns
    if column not in ordered_replica_columns
]

ipa_df = ipa_df[
    ordered_replica_columns
    + remaining_columns
]


# =====================================================================
# 29. SALVATAGGIO DEL FILE EXCEL
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    ipa_df.to_excel(
        writer,
        sheet_name="IPA_by_Replicate",
        index=False
    )

    sample_summary.to_excel(
        writer,
        sheet_name="IPA_by_Sample",
        index=False
    )

    top_compounds.to_excel(
        writer,
        sheet_name="Top_Compounds",
        index=False
    )

    ipa_matrix_replica.to_excel(
        writer,
        sheet_name="IPA_Matrix_Replicate"
    )

    lipa_matrix_replica.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Replicate"
    )

    ipa_matrix_sample.to_excel(
        writer,
        sheet_name="IPA_Matrix_Sample"
    )

    lipa_matrix_sample.to_excel(
        writer,
        sheet_name="LIPA_Matrix_Sample"
    )

    not_calculable.to_excel(
        writer,
        sheet_name="IPA_Not_Calculable",
        index=False
    )

    processing_summary.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    # -----------------------------------------------------------------
    # Formati Excel
    # -----------------------------------------------------------------

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    integer_format = workbook.add_format(
        {
            "num_format": "0",
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    rank_format = workbook.add_format(
        {
            "num_format": "0",
            "align": "center",
            "valign": "top"
        }
    )

    url_format = workbook.add_format(
        {
            "font_color": "blue",
            "underline": True,
            "text_wrap": True,
            "valign": "top"
        }
    )

    # Colori usati soltanto per la formattazione condizionale.
    high_format = workbook.add_format(
        {
            "bg_color": "#C6EFCE",
            "font_color": "#006100"
        }
    )

    medium_format = workbook.add_format(
        {
            "bg_color": "#FFEB9C",
            "font_color": "#9C6500"
        }
    )

    low_format = workbook.add_format(
        {
            "bg_color": "#FFC7CE",
            "font_color": "#9C0006"
        }
    )


    dataframe_by_sheet = {
        "IPA_by_Replicate": ipa_df,
        "IPA_by_Sample": sample_summary,
        "Top_Compounds": top_compounds,
        "IPA_Not_Calculable": not_calculable,
        "Processing_Summary": processing_summary
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(24)

        if len(dataframe.columns) > 0:

            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                width,
                wrap_format
            )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Replicate
    # -----------------------------------------------------------------

    replica_ws = writer.sheets[
        "IPA_by_Replicate"
    ]

    for column_name in [
        "Area",
        "IS_area"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                14,
                integer_format
            )

    for column_name in [
        NORMALIZED_AREA_COLUMN,
        THRESHOLD_COLUMN,
        "IPA"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA",
        "Log10_analytical_response",
        "Log10_odor_potency"
    ]:

        if column_name in ipa_df.columns:

            column_index = ipa_df.columns.get_loc(
                column_name
            )

            replica_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_replica" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "IPA_rank_replica"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "Threshold_source_url" in ipa_df.columns:

        column_index = ipa_df.columns.get_loc(
            "Threshold_source_url"
        )

        replica_ws.set_column(
            column_index,
            column_index,
            40,
            url_format
        )

    if "LIPA" in ipa_df.columns and len(ipa_df) > 0:

        lipa_col = ipa_df.columns.get_loc("LIPA")

        replica_ws.conditional_format(
            1,
            lipa_col,
            len(ipa_df),
            lipa_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione IPA_by_Sample
    # -----------------------------------------------------------------

    sample_ws = writer.sheets[
        "IPA_by_Sample"
    ]

    for column_name in [
        "Mean_normalized_area",
        "SD_normalized_area",
        "Min_normalized_area",
        "Max_normalized_area",
        THRESHOLD_COLUMN,
        "IPA_mean"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                scientific_format
            )

    for column_name in [
        "LIPA_mean",
        "CV_normalized_area_percent"
    ]:

        if column_name in sample_summary.columns:

            column_index = (
                sample_summary.columns.get_loc(
                    column_name
                )
            )

            sample_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "IPA_rank_sample" in sample_summary.columns:

        column_index = (
            sample_summary.columns.get_loc(
                "IPA_rank_sample"
            )
        )

        sample_ws.set_column(
            column_index,
            column_index,
            12,
            rank_format
        )

    if "LIPA_mean" in sample_summary.columns and len(sample_summary) > 0:

        lipa_mean_col = (
            sample_summary.columns.get_loc(
                "LIPA_mean"
            )
        )

        sample_ws.conditional_format(
            1,
            lipa_mean_col,
            len(sample_summary),
            lipa_mean_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


    # -----------------------------------------------------------------
    # Formattazione delle matrici
    # -----------------------------------------------------------------

    matrix_sheet_names = [
        "IPA_Matrix_Replicate",
        "LIPA_Matrix_Replicate",
        "IPA_Matrix_Sample",
        "LIPA_Matrix_Sample"
    ]

    for sheet_name in matrix_sheet_names:

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 2)

        worksheet.set_column(
            0,
            1,
            18
        )

        worksheet.set_column(
            2,
            200,
            16,
            scientific_format
            if sheet_name.startswith("IPA_")
            else decimal_format
        )


# =====================================================================
# 30. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("CALCOLO IPA COMPLETATO")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)

print(
    "Righe analita-replica:",
    len(ipa_df)
)

print(
    "Valori IPA calcolati:",
    int(
        ipa_df["IPA"]
        .notna()
        .sum()
    )
)

print(
    "Valori IPA non calcolabili:",
    int(
        ipa_df["IPA"]
        .isna()
        .sum()
    )
)

print(
    "Campioni distinti:",
    ipa_df[SAMPLE_COLUMN].nunique()
)

print(
    "Analiti distinti:",
    ipa_df[CAS_COLUMN].nunique()
)

print()
print("Anteprima del ranking medio per campione:")

preview_columns = [
    SAMPLE_COLUMN,
    "IPA_rank_sample",
    COMPOUND_COLUMN,
    CAS_COLUMN,
    "Mean_normalized_area",
    THRESHOLD_COLUMN,
    "IPA_mean",
    "LIPA_mean",
    "IPA_priority_class",
    "Sensory_family",
    "Confidence"
]

preview_columns = [
    column
    for column in preview_columns
    if column in sample_summary.columns
]

display(
    sample_summary[
        preview_columns
    ].head(30)
)

files.download(
    str(OUTPUT_FILE)
)

RECUPERO DELLA MASTER TABLE
È stata utilizzata la variabile master_df presente nella memoria della sessione.
Righe disponibili: 24

CALCOLO IPA COMPLETATO
File creato: GCMS_IPA_Results.xlsx
Righe analita-replica: 24
Valori IPA calcolati: 24
Valori IPA non calcolabili: 0
Campioni distinti: 2
Analiti distinti: 4

Anteprima del ranking medio per campione:


,Sample_ID,IPA_rank_sample,GCMS_column_name,CAS,Mean_normalized_area,Odor_threshold_air_ug_m3,IPA_mean,LIPA_mean,IPA_priority_class,Sensory_family,Confidence
0,Cacao_01,1,2-methoxyphenol,90-05-1,0.245336,0.084000,2.920666,0.465482,Moderata,Affumicato | Fenolico/Medicinale | Speziato | ...,medium
1,Cacao_01,2,linalool,78-70-6,0.126724,3.200000,0.039601,-1.402292,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
2,Cacao_01,3,"2,3,5-trimethylpyrazine",14667-55-1,1.663152,50.000000,0.033263,-1.478038,Molto bassa,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium
3,Cacao_01,4,phenol,108-95-2,0.185550,23.094479,0.008034,-2.095048,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce...,medium
4,Cacao_02,1,2-methoxyphenol,90-05-1,0.199480,0.084000,2.374760,0.375620,Moderata,Affumicato | Fenolico/Medicinale | Speziato | ...,medium
5,Cacao_02,2,linalool,78-70-6,0.145738,3.200000,0.045543,-1.341576,Molto bassa,Floreale | Fruttato | Dolce | Verde/Erbaceo | ...,medium
6,Cacao_02,3,"2,3,5-trimethylpyrazine",14667-55-1,1.845060,50.000000,0.036901,-1.432959,Molto bassa,Cacao/Cioccolato | Tostato | Caffè | Frutta se...,medium
7,Cacao_02,4,phenol,108-95-2,0.253445,23.094479,0.010974,-1.959625,Molto bassa,Fenolico/Medicinale | Chimico/Solvente | Dolce...,medium


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# @title
# =====================================================================
# QUARTA CELLA — STIMA BAYESIANA DELLA CONCENTRAZIONE (VERSIONE 4A)
# Versione corretta per una singola cella Google Colab
#
# Questa cella deve essere eseguita nella stessa sessione Colab
# utilizzata per le celle 1, 2 e 3.
#
# INPUT già presenti in /content (oppure variabile in memoria):
#   - variabile ipa_df prodotta dalla cella 3
#     oppure
#   - GCMS_IPA_Results.xlsx, foglio IPA_by_Replicate
#
#   - il file GC-MS di origine (lo stesso caricato nella cella 2),
#     contenente i fogli:
#       Compound_Mapping   (per identificare lo standard interno)
#       metodo_analitico   (per la concentrazione dello standard
#                           interno nel campione, e — solo in
#                           modalità "headspace" — per contenuto
#                           lipidico, contenuto d'acqua e
#                           temperatura di equilibrio)
#
# QUESTA CELLA RICHIEDE UNA CHIAVE API OPENAI (come la cella 1):
# determina automaticamente il livello di similarità di ogni analita
# rispetto allo standard interno, e — solo in modalità "headspace" —
# la costante di Henry e il logP dello standard interno, tramite
# PubChem + ricerca web su fonti scientifiche.
#
# OUTPUT:
#   GCMS_Concentration_Estimates.xlsx
#   Similarity_Cache.xlsx
#
# PARAMETRO CHE DEVI FORNIRE TU NEL FOGLIO metodo_analitico (SEMPRE):
#   La concentrazione nota dello standard interno nel campione
#   originale (non è reperibile né da PubChem né altrove nel file):
#
#       concentrazione standard interno nel campione [µg/kg]    <valore>
#
# SCELTA INTERATTIVA ALL'AVVIO — "Composition referred to":
#
#   1) Headspace in itself
#      La cella stima direttamente la concentrazione degli ANALITI
#      nell'headspace della vial (µg/m3), comparabile senza altri
#      passaggi con Odor_threshold_air_ug_m3. La cella 5 non serve
#      più in questo caso.
#
#      Lo standard interno resta comunque aggiunto nella matrice
#      (non nell'headspace): per sapere quanto di esso passa in
#      fase gas, la cella applica IL MODELLO DI PARTIZIONE A DUE
#      FASI (acqua/lipide, stesso principio della cella 5) — ma
#      SOLO allo standard interno, la cui identità è nota con
#      certezza (a differenza degli analiti, per cui non si tenta
#      più questo passaggio).
#
#   2) The solid/liquid sample that generated the headspace
#      Comportamento originario: la cella stima la concentrazione
#      degli analiti nella matrice del campione (µg/kg). La
#      conversione verso l'aria (se serve) resta demandata alla
#      cella 5.
#
# METODO — MODALITÀ "sample" (Versione 4A, invariata):
#
#   log10(C_i,campione) = log10(C_IS,campione) + log10(R_i) - delta_i
#
# METODO — MODALITÀ "headspace" (nuova):
#
#   Passo 1 — SOLO per lo standard interno:
#     log10(C_IS,headspace) = log10(C_IS,campione)
#                              + log10(K_IS,matrice->aria)
#     con K_IS,matrice->aria calcolato come nella cella 5:
#       K = K_aw / ( w_acqua/rho_acqua + K_ow * w_lipide/rho_lipide )
#
#   Passo 2 — per ogni analita, come nella modalità "sample" ma
#   riferendosi all'headspace dello standard invece che al campione:
#     log10(C_i,headspace) = log10(C_IS,headspace)
#                             + log10(R_i) - delta_i
#
#   L'incertezza totale somma tre contributi indipendenti (radice
#   della somma dei quadrati): incertezza di R_i, incertezza del
#   prior di similarità (delta_i), incertezza della conversione
#   campione->headspace dello standard interno.
#
#   dove:
#     R_i = Area_i / Area_IS       (già calcolato nella cella 3,
#                                    Log10_analytical_response)
#     delta_i = differenza di risposta/affinità di fibra SPME tra
#               analita e standard interno, con prior Normale la
#               cui incertezza dipende dal livello di similarità
#               (determinato automaticamente da questa cella).
#
#   NOTA IMPORTANTE:
#   Questa è la Versione 4A, esplicitamente minimale. Non sono
#   ancora inseriti composti "ancora" a concentrazione nota
#   (Versione 4B) né proprietà fisico-chimiche predittive per OGNI
#   analita (Versione 4C). Il risultato è un ordine di grandezza
#   con intervallo credibile ampio, non una quantificazione
#   validata.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DELLE LIBRERIE
# =====================================================================

!pip -q install --upgrade openai pydantic scipy xlsxwriter openpyxl


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
import json
import time
import math
import getpass
import unicodedata
from datetime import date
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
import xlsxwriter
from scipy.stats import norm

from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import files, userdata


# =====================================================================
# 2. PARAMETRI MODIFICABILI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

IPA_RESULTS_FILE = WORKING_DIRECTORY / "GCMS_IPA_Results.xlsx"
IPA_REPLICATE_SHEET = "IPA_by_Replicate"

OUTPUT_FILE = WORKING_DIRECTORY / "GCMS_Concentration_Estimates.xlsx"
SIMILARITY_CACHE_FILE = WORKING_DIRECTORY / "Similarity_Cache.xlsx"

COMPOUND_MAPPING_SHEET = "Compound_Mapping"
METHOD_SHEET_NAME = "metodo_analitico"
ROLE_COLUMN = "Compound_role"

# Modello OpenAI, coerente con la cella 1.
MODEL = "gpt-5.6"

# Pausa fra una chiamata API e la successiva.
PAUSE_SECONDS = 1.0

# Numero massimo di tentativi per ogni composto.
MAX_RETRIES = 2

SAMPLE_COLUMN = "Sample_ID"
REPLICATE_COLUMN = "Replicate"
COMPOUND_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

# Colonna già calcolata nella cella 3: log10(Area / Area_IS) per replica.
LOG_RESPONSE_COLUMN = "Log10_analytical_response"

THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"

# Unità di concentrazione per ciascuna modalità.
SAMPLE_CONCENTRATION_UNIT = "µg/kg"
HEADSPACE_CONCENTRATION_UNIT = "µg/m3"

# ---------------------------------------------------------------------
# Tabella dei prior sul fattore di risposta/affinità di fibra
# relativo (delta), in funzione del livello di similarità tra
# analita e standard interno.
#
# Questi valori sono ipotesi metodologiche, non parametri validati
# sperimentalmente. Vanno rivisti caso per caso.
# ---------------------------------------------------------------------

SIMILARITY_PRIOR_TABLE = {
    "stesso composto isotopico": {"mu": 0.0, "sigma": 0.2},
    "analogo molto vicino": {"mu": 0.0, "sigma": 0.5},
    "stessa classe chimica": {"mu": 0.0, "sigma": 1.0},
    "classe diversa": {"mu": 0.0, "sigma": 2.0},
    "comportamento hs spme molto diverso": {"mu": 0.0, "sigma": 2.5},
}

DEFAULT_REPLICATE_LOG_SD = 0.15
MIN_REPLICATES_FOR_EMPIRICAL_SD = 2
CREDIBLE_INTERVAL_PROBABILITY = 0.90
DECADE_SIGNIFICANT_DIGITS = 3

# ---------------------------------------------------------------------
# Parametri per la conversione campione->headspace dello standard
# interno (usati SOLO in modalità "headspace"), stesso modello a due
# fasi già usato nella cella 5.
# ---------------------------------------------------------------------

WATER_DENSITY_KG_M3 = 1000.0
DEFAULT_LIPID_DENSITY_KG_M3 = 900.0
GAS_CONSTANT_PA_M3_MOL_K = 8.314

RIGOROUS_RESIDUAL_LOG_SD = 0.3
WATER_ONLY_RESIDUAL_LOG_SD = 0.6


# =====================================================================
# 3. FUNZIONI DI SUPPORTO GENERICHE
# =====================================================================

def normalize_cas(value):
    """
    Uniforma trattini e spazi nel numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    return re.sub(r"\s+", "", value)


def validate_cas(cas_number):
    """
    Controlla formato del CAS e cifra di controllo.
    """
    if pd.isna(cas_number):
        return False

    cas_number = str(cas_number).strip()

    if not re.fullmatch(r"\d{2,7}-\d{2}-\d", cas_number):
        return False

    digits = cas_number.replace("-", "")
    body = digits[:-1]
    expected_check_digit = int(digits[-1])

    calculated_sum = sum(
        position * int(digit)
        for position, digit in enumerate(reversed(body), start=1)
    )

    return (calculated_sum % 10) == expected_check_digit


def normalize_role(value):
    """
    Normalizza il ruolo del composto (analita / standard interno).
    """
    if pd.isna(value):
        return ""

    return (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )


def normalize_similarity_level(value):
    """
    Normalizza un'etichetta di livello di similarità, per renderla
    confrontabile con le chiavi di SIMILARITY_PRIOR_TABLE.
    """
    if value is None or pd.isna(value):
        return None

    text = (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    return re.sub(r"\s+", " ", text).strip() or None


def get_similarity_prior(similarity_level_normalized):
    """
    Restituisce (mu, sigma) del prior su delta per un dato livello
    di similarità già normalizzato.

    Restituisce np.nan (non None) quando il livello non è
    disponibile o non è riconosciuto, così le colonne risultanti
    restano di tipo numerico anche quando TUTTI i valori sono
    mancanti.
    """
    if similarity_level_normalized is None:
        return np.nan, np.nan

    entry = SIMILARITY_PRIOR_TABLE.get(similarity_level_normalized)

    if entry is None:
        return np.nan, np.nan

    return entry["mu"], entry["sigma"]


def format_significant(value, digits=DECADE_SIGNIFICANT_DIGITS):
    """
    Formatta un numero con un numero fisso di cifre significative.
    """
    if value is None or pd.isna(value) or value <= 0:
        return ""

    return f"{value:.{digits}g}"


def classify_decade(median_value):
    """
    Classifica il valore mediano stimato nella decade di appartenenza.
    """
    if median_value is None or pd.isna(median_value) or median_value <= 0:
        return "Non calcolabile"

    exponent = math.floor(math.log10(median_value))

    lower_bound = 10 ** exponent
    upper_bound = 10 ** (exponent + 1)

    return (
        f"{format_significant(lower_bound)} – "
        f"{format_significant(upper_bound)}"
    )


def classify_model_quality(similarity_level_normalized, replicate_count):
    """
    Valutazione qualitativa dell'affidabilità della stima. Nella
    Versione 4A (nessuna ancora sperimentale) la qualità massima
    raggiungibile resta comunque limitata.
    """

    if similarity_level_normalized is None:
        return "Non valutabile: livello di similarità mancante"

    if (
        replicate_count is None
        or pd.isna(replicate_count)
        or replicate_count < MIN_REPLICATES_FOR_EMPIRICAL_SD
    ):
        return (
            "Bassa: repliche insufficienti per stimare "
            "empiricamente la variabilità analitica"
        )

    if similarity_level_normalized in {
        "stesso composto isotopico", "analogo molto vicino"
    }:
        return "Moderata-buona (solo su prior, nessuna ancora sperimentale)"

    if similarity_level_normalized == "stessa classe chimica":
        return "Moderata (solo su prior, nessuna ancora sperimentale)"

    return (
        "Bassa: classe diversa o comportamento HS-SPME "
        "molto diverso dallo standard interno"
    )


def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che una tabella contenga tutte le colonne richieste.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    maximum_value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(header_length, int(maximum_value_length)) + 2

    return min(max(width, 12), maximum)


def find_file_with_required_sheets(
    directory,
    required_sheets,
    exclude_names=frozenset()
):
    """
    Cerca nella cartella indicata un file Excel contenente tutti i
    fogli richiesti. Se ne trova più di uno, utilizza quello
    modificato più recentemente.
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        if filepath.name in exclude_names:
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            if set(required_sheets).issubset(
                set(excel_file.sheet_names)
            ):
                candidates.append(filepath)

        except Exception:
            continue

    if len(candidates) == 0:
        return None

    if len(candidates) == 1:
        return candidates[0]

    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        "Sono stati trovati più file compatibili con i fogli "
        f"richiesti {sorted(required_sheets)}."
    )
    print("Verrà utilizzato il file modificato più recentemente:")
    print(candidates[0].name)

    print()
    print("Altri file compatibili rilevati:")

    for filepath in candidates[1:]:
        print(" -", filepath.name)

    return candidates[0]


def strip_accents(text):
    """
    Rimuove gli accenti da una stringa, per rendere più robusto il
    riconoscimento delle etichette del foglio metodo_analitico.
    """
    normalized = unicodedata.normalize("NFKD", text)

    return "".join(
        character
        for character in normalized
        if not unicodedata.combining(character)
    )


def normalize_label_key(value):
    """
    Normalizza un'etichetta per il riconoscimento per parole chiave.
    """
    if value is None or pd.isna(value):
        return ""

    text = strip_accents(str(value)).strip().lower()

    return re.sub(r"\s+", " ", text)


def parse_numeric_value(value):
    """
    Converte in float un valore che potrebbe contenere unità o
    testo accessorio. Restituisce NaN se non interpretabile.
    """
    if value is None or pd.isna(value):
        return np.nan

    match = re.search(r"[-+]?\d+(?:[.,]\d+)?", str(value))

    if not match:
        return np.nan

    return float(match.group(0).replace(",", "."))


def parse_internal_standard_concentration(dataframe):
    """
    Cerca nel foglio metodo_analitico (formato a due colonne:
    etichetta, valore) la riga contenente la concentrazione nota
    dello standard interno nel campione.

    Riconosce etichette contenenti "concentrazione" insieme a
    "standard interno" oppure "is".
    """

    for _, row in dataframe.iterrows():

        if len(row) < 2:
            continue

        label_key = normalize_label_key(row.iloc[0])

        if not label_key:
            continue

        has_concentration_word = "concentrazion" in label_key

        has_is_word = (
            "standard interno" in label_key
            or "standardinterno" in label_key
            or re.search(r"\bis\b", label_key) is not None
        )

        if has_concentration_word and has_is_word:
            return parse_numeric_value(row.iloc[1])

    return np.nan


def parse_matrix_composition(dataframe):
    """
    Cerca nel foglio metodo_analitico i parametri necessari alla
    conversione campione->headspace: contenuto lipidico, contenuto
    d'acqua, temperatura di equilibrio.

    Usata SOLO in modalità "headspace". Solleva un errore se manca
    uno di questi tre parametri.
    """

    parsed = {
        "lipid_percent": None,
        "water_percent": None,
        "equilibrium_temperature_c": None,
    }

    for _, row in dataframe.iterrows():

        if len(row) < 2:
            continue

        label_key = normalize_label_key(row.iloc[0])
        value = row.iloc[1]

        if not label_key:
            continue

        if "lipid" in label_key:
            parsed["lipid_percent"] = parse_numeric_value(value)

        elif "acqua" in label_key:
            parsed["water_percent"] = parse_numeric_value(value)

        elif "temperat" in label_key and "equilibri" in label_key:
            parsed["equilibrium_temperature_c"] = parse_numeric_value(
                value
            )

    missing = [
        name
        for name, key in [
            ("contenuto lipidico", "lipid_percent"),
            ("contenuto d'acqua", "water_percent"),
            ("temperatura di equilibrio", "equilibrium_temperature_c"),
        ]
        if parsed[key] is None or pd.isna(parsed[key])
    ]

    if missing:
        raise ValueError(
            f"Modalità 'headspace': nel foglio '{METHOD_SHEET_NAME}' "
            "non sono stati riconosciuti i seguenti parametri "
            "indispensabili per convertire lo standard interno in "
            "concentrazione di headspace: " + ", ".join(missing)
        )

    return parsed


def convert_henry_to_dimensionless(value, unit_text, temperature_k):
    """
    Converte una costante di Henry in forma adimensionale K_aw.
    Unità riconosciute: Pa*m3/mol, atm*m3/mol, oppure già
    adimensionale.
    """

    if value is None or pd.isna(value):
        return np.nan

    if unit_text is None or pd.isna(unit_text):
        return np.nan

    unit_clean = str(unit_text).strip().lower().replace(" ", "")

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if "dimension" in unit_clean or unit_clean in {"kaw", "-"}:
        return value

    if "atm" in unit_clean:
        h_pa_m3_mol = value * 101325.0

    elif "pa" in unit_clean:
        h_pa_m3_mol = value

    else:
        return np.nan

    return h_pa_m3_mol / (GAS_CONSTANT_PA_M3_MOL_K * temperature_k)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 35.6 MB/s eta 0:00:00


In [6]:
# @title
# =====================================================================
# 4. SCELTA DELLA MODALITÀ (usa quella del Modulo 0, se presente)
# =====================================================================

if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE in {"headspace", "sample"}:

    print("=" * 72)
    print("Modalità già impostata nel Modulo 0:", COMPOSITION_MODE)
    print("=" * 72)

else:

    print("=" * 72)
    print("Composition referred to:")
    print("  1) Headspace in itself")
    print("  2) The solid/liquid sample that generated the headspace")
    print("=" * 72)

    composition_choice = input("Enter 1 or 2: ").strip()

    while composition_choice not in {"1", "2"}:
        composition_choice = input(
            "Valore non valido. Inserire 1 oppure 2: "
        ).strip()

    COMPOSITION_MODE = "headspace" if composition_choice == "1" else "sample"

print()
if COMPOSITION_MODE == "headspace":
    print(
        "Modalità selezionata: HEADSPACE — le concentrazioni stimate "
        "per gli analiti saranno riferite all'headspace della vial "
        f"({HEADSPACE_CONCENTRATION_UNIT}), direttamente confrontabili "
        f"con '{THRESHOLD_COLUMN}'. La cella 5 non è necessaria."
    )
else:
    print(
        "Modalità selezionata: CAMPIONE — le concentrazioni stimate "
        "per gli analiti saranno riferite alla matrice del campione "
        f"({SAMPLE_CONCENTRATION_UNIT}), come nella versione originaria. "
        "Per il confronto con l'aria, eseguire poi la cella 5."
    )

# =====================================================================
# 5. LETTURA DELLA CHIAVE API OPENAI
# =====================================================================

try:
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    print(
        "Il Secret OPENAI_API_KEY non è stato trovato o non è accessibile."
    )
    api_key = getpass.getpass(
        "Inserire la chiave API OpenAI. "
        "La chiave non sarà visualizzata: "
    )

if not api_key or not api_key.strip():
    raise ValueError("La chiave API OpenAI non è disponibile.")

client = OpenAI(api_key=api_key.strip())

print()
print("Chiave API caricata correttamente.")


# =====================================================================
# 6. RECUPERO IDENTITÀ SU PUBCHEM
# =====================================================================

def get_pubchem_identity(cas_number):
    """
    Recupera da PubChem CID, nome IUPAC, formula molecolare, massa
    molecolare e SMILES canonico.
    """

    encoded_cas = requests.utils.quote(str(cas_number), safe="")

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded_cas}/property/"
        "IUPACName,MolecularFormula,MolecularWeight,"
        "CanonicalSMILES/JSON"
    )

    try:

        response = requests.get(url, timeout=30)

        if response.status_code == 404:
            return {}

        response.raise_for_status()

        payload = response.json()

        properties = (
            payload.get("PropertyTable", {}).get("Properties", [])
        )

        if not properties:
            return {}

        record = properties[0]
        cid = record.get("CID")

        return {
            "PubChem_CID": cid,
            "PubChem_IUPAC_name": record.get("IUPACName"),
            "PubChem_formula": record.get("MolecularFormula"),
            "PubChem_MW": record.get("MolecularWeight"),
            "PubChem_SMILES": record.get("CanonicalSMILES"),
            "PubChem_URL": (
                f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}"
                if cid is not None else None
            ),
            "PubChem_error": None
        }

    except Exception as error:

        return {
            "PubChem_CID": None,
            "PubChem_IUPAC_name": None,
            "PubChem_formula": None,
            "PubChem_MW": None,
            "PubChem_SMILES": None,
            "PubChem_URL": None,
            "PubChem_error": str(error)
        }


# =====================================================================
# 7. STRUTTURE DELLE RISPOSTE DEL MODELLO
# =====================================================================

SIMILARITY_LEVELS_TEXT = ", ".join(SIMILARITY_PRIOR_TABLE.keys())


class SimilarityRecord(BaseModel):

    cas_input: str
    iupac_input: str

    identity_match: str = Field(
        description=(
            "Valori consentiti: confirmed, probable, "
            "conflicting, not_found"
        )
    )

    common_name: Optional[str] = None
    chemical_class: Optional[str] = Field(
        default=None,
        description=(
            "Classe chimica principale dell'analita, es. pirazina, "
            "aldeide, terpene, fenolo, estere, chetone"
        )
    )

    similarity_level: str = Field(
        description=(
            "Livello di similarità della risposta analitica "
            "HS-SPME-GC-MS dell'analita rispetto allo standard "
            f"interno. Valori consentiti esclusivamente: "
            f"{SIMILARITY_LEVELS_TEXT}"
        )
    )

    rationale: Optional[str] = Field(
        default=None,
        description=(
            "Motivazione sintetica: somiglianza strutturale, "
            "gruppo funzionale, polarità, volatilità attesa, "
            "affinità per fibra SPME rispetto allo standard interno"
        )
    )

    source_name: Optional[str] = None
    source_url: Optional[str] = None

    confidence: str = Field(
        description="Valori consentiti: high, medium, low"
    )

    review_required: bool = False
    notes: Optional[str] = None


class InternalStandardPhysChemRecord(BaseModel):

    cas_input: str
    iupac_input: str

    identity_match: str = Field(
        description=(
            "Valori consentiti: confirmed, probable, "
            "conflicting, not_found"
        )
    )

    henry_constant_value: Optional[float] = Field(
        default=None,
        description="Valore della costante di Henry, se reperita"
    )
    henry_constant_unit: Optional[str] = Field(
        default=None,
        description="Unità: 'Pa*m3/mol', 'atm*m3/mol', oppure 'dimensionless'"
    )

    logp_value: Optional[float] = Field(
        default=None,
        description="Coefficiente di ripartizione ottanolo/acqua, log10(Kow)"
    )

    source_name: Optional[str] = None
    source_url: Optional[str] = None

    confidence: str = Field(
        description="Valori consentiti: high, medium, low"
    )

    review_required: bool = False
    notes: Optional[str] = None


# =====================================================================
# 8. FUNZIONI DI RICERCA ONLINE
# =====================================================================

WEB_SEARCH_ALLOWED_DOMAINS = [
    "pubchem.ncbi.nlm.nih.gov",
    "webbook.nist.gov",
    "thegoodscentscompany.com",
    "sciencedirect.com",
    "acs.org",
    "springer.com",
    "wiley.com",
    "tandfonline.com",
    "en.wikipedia.org"
]


def search_similarity_online(
    analyte_cas,
    analyte_iupac,
    analyte_pubchem,
    internal_standard_context
):
    """
    Determina il livello di similarità strutturale/analitica tra un
    analita e lo standard interno, tramite PubChem e ricerca web su
    fonti scientifiche.
    """

    analyte_context = json.dumps(
        analyte_pubchem, ensure_ascii=False, indent=2
    )

    is_context = json.dumps(
        internal_standard_context, ensure_ascii=False, indent=2
    )

    system_prompt = f"""
Sei un chimico esperto di HS-SPME-GC-MS e di standardizzazione
interna in analisi degli aromi.

Devi valutare quanto la risposta analitica relativa (rapporto
area/standard interno) di un composto analita sia probabilmente
simile a quella dello standard interno usato nel metodo, e
classificare questa similarità in una scala discreta.

CATEGORIE AMMESSE (usa ESATTAMENTE una di queste stringhe nel
campo similarity_level):
- "stesso composto isotopico": l'analita è la versione isotopicamente
  marcata dello stesso identico composto dello standard interno.
- "analogo molto vicino": stesso gruppo funzionale principale,
  struttura molecolare molto simile, peso molecolare vicino,
  polarità e volatilità comparabili.
- "stessa classe chimica": stessa classe generale (es. entrambi
  pirazine, entrambi aldeidi) ma struttura diversa.
- "classe diversa": classi chimiche diverse (es. estere vs fenolo),
  polarità o volatilità sensibilmente diverse.
- "comportamento hs spme molto diverso": differenze marcate di
  polarità/volatilità/affinità per la fibra SPME tali da rendere
  il confronto con lo standard interno poco affidabile.

REGOLE OBBLIGATORIE:
- Verifica che il CAS e il nome forniti corrispondano alla stessa
  sostanza (usa il campo identity_match).
- Basa la classificazione su struttura chimica, gruppo funzionale,
  polarità, volatilità attesa e comportamento noto in HS-SPME,
  non solo sulla famiglia sensoriale.
- Non inventare fonti o URL.
- Imposta review_required = true quando l'identificazione è incerta
  o le informazioni strutturali sono insufficienti.
"""

    user_prompt = f"""
STANDARD INTERNO DEL METODO
{is_context}

ANALITA DA CLASSIFICARE

Nome IUPAC fornito:
{analyte_iupac}

CAS fornito:
{analyte_cas}

Informazioni preliminari da PubChem sull'analita:
{analyte_context}

Determina il livello di similarità dell'analita rispetto allo
standard interno sopra indicato, secondo le categorie definite.
"""

    response = client.responses.parse(
        model=MODEL,
        tools=[
            {
                "type": "web_search",
                "filters": {"allowed_domains": WEB_SEARCH_ALLOWED_DOMAINS}
            }
        ],
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=SimilarityRecord
    )

    if response.output_parsed is None:
        raise ValueError(
            "La risposta API non contiene un record strutturato."
        )

    return response.output_parsed


def search_internal_standard_physicochemical(
    is_cas, is_iupac, is_pubchem
):
    """
    Recupera costante di Henry e logP dello standard interno tramite
    PubChem e ricerca web su fonti scientifiche (usata SOLO in
    modalità "headspace").
    """

    is_context = json.dumps(is_pubchem, ensure_ascii=False, indent=2)

    system_prompt = """
Sei un chimico esperto di proprietà fisico-chimiche di composti
volatili (costante di Henry, logP).

Devi recuperare, per il composto indicato, la costante di Henry
(aria/acqua) e il logP (log10 del coefficiente di ripartizione
ottanolo/acqua), da fonti scientifiche affidabili (letteratura
peer-reviewed, NIST WebBook, database EPA/EPI Suite).

REGOLE OBBLIGATORIE:
- Verifica che il CAS e il nome forniti corrispondano alla stessa
  sostanza (usa il campo identity_match).
- Se trovi più valori in letteratura, usa quello più citato o più
  recente e indicalo nelle note.
- Specifica sempre l'unità della costante di Henry esattamente come
  'Pa*m3/mol', 'atm*m3/mol', oppure 'dimensionless'.
- Se non trovi un valore affidabile per uno dei due parametri,
  lascialo vuoto (null) invece di inventarlo.
- Non inventare fonti o URL.
- Imposta review_required = true se la confidenza è bassa o i dati
  sono discordanti tra fonti diverse.
"""

    user_prompt = f"""
COMPOSTO (standard interno del metodo)

Nome IUPAC fornito:
{is_iupac}

CAS fornito:
{is_cas}

Informazioni preliminari da PubChem:
{is_context}

Recupera costante di Henry e logP per questo composto.
"""

    response = client.responses.parse(
        model=MODEL,
        tools=[
            {
                "type": "web_search",
                "filters": {"allowed_domains": WEB_SEARCH_ALLOWED_DOMAINS}
            }
        ],
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=InternalStandardPhysChemRecord
    )

    if response.output_parsed is None:
        raise ValueError(
            "La risposta API non contiene un record strutturato."
        )

    return response.output_parsed

Modalità già impostata nel Modulo 0: sample

Modalità selezionata: CAMPIONE — le concentrazioni stimate per gli analiti saranno riferite alla matrice del campione (µg/kg), come nella versione originaria. Per il confronto con l'aria, eseguire poi la cella 5.

Chiave API caricata correttamente.


In [7]:
# @title
# =====================================================================
# 9. RECUPERO DEI DATI A LIVELLO DI REPLICA (CELLA 3)
# =====================================================================

print()
print("=" * 72)
print("RECUPERO DEI DATI IPA A LIVELLO DI REPLICA")
print("=" * 72)

if (
    "ipa_df" in globals()
    and isinstance(ipa_df, pd.DataFrame)
    and not ipa_df.empty
):

    replicate_df = ipa_df.copy()

    print(
        "È stata utilizzata la variabile ipa_df "
        "presente nella memoria della sessione."
    )

else:

    if not IPA_RESULTS_FILE.exists():
        raise FileNotFoundError(
            f"Non è disponibile la variabile ipa_df e non è stato "
            f"trovato il file '{IPA_RESULTS_FILE.name}' in /content.\n\n"
            "Eseguire prima la cella 3 nella stessa sessione Colab."
        )

    ipa_excel = pd.ExcelFile(IPA_RESULTS_FILE)

    if IPA_REPLICATE_SHEET not in ipa_excel.sheet_names:
        raise ValueError(
            f"Nel file '{IPA_RESULTS_FILE.name}' manca il foglio "
            f"'{IPA_REPLICATE_SHEET}'."
        )

    replicate_df = pd.read_excel(
        IPA_RESULTS_FILE,
        sheet_name=IPA_REPLICATE_SHEET,
        dtype={CAS_COLUMN: str}
    )

    print(
        f"È stato letto il file '{IPA_RESULTS_FILE.name}', "
        f"foglio '{IPA_REPLICATE_SHEET}'."
    )

print("Righe disponibili:", len(replicate_df))


# =====================================================================
# 10. CONTROLLO DELLE COLONNE OBBLIGATORIE
# =====================================================================

required_replicate_columns = [
    SAMPLE_COLUMN, REPLICATE_COLUMN, COMPOUND_COLUMN,
    IUPAC_COLUMN, CAS_COLUMN, LOG_RESPONSE_COLUMN
]

check_required_columns(
    replicate_df, required_replicate_columns, IPA_REPLICATE_SHEET
)

replicate_df[CAS_COLUMN] = replicate_df[CAS_COLUMN].apply(normalize_cas)

replicate_df[LOG_RESPONSE_COLUMN] = pd.to_numeric(
    replicate_df[LOG_RESPONSE_COLUMN], errors="coerce"
)


# =====================================================================
# 11. AGGREGAZIONE DI log10(R_i) PER CAMPIONE E COMPOSTO
# =====================================================================

optional_passthrough_columns = [
    "Common_name", "Sensory_family", "Confidence", THRESHOLD_COLUMN
]

passthrough_present = [
    column for column in optional_passthrough_columns
    if column in replicate_df.columns
]

aggregation_dictionary = {LOG_RESPONSE_COLUMN: ["count", "mean", "std"]}

for column in passthrough_present:
    aggregation_dictionary[column] = "first"

response_summary = (
    replicate_df
    .groupby(
        [SAMPLE_COLUMN, COMPOUND_COLUMN, IUPAC_COLUMN, CAS_COLUMN],
        dropna=False
    )
    .agg(aggregation_dictionary)
    .reset_index()
)

flattened_columns = []

for column in response_summary.columns:

    if isinstance(column, tuple):
        first_part = str(column[0]).strip()
        second_part = str(column[1]).strip()

        flattened_columns.append(
            f"{first_part}_{second_part}" if second_part else first_part
        )

    else:
        flattened_columns.append(str(column))

response_summary.columns = flattened_columns

response_summary = response_summary.rename(
    columns={
        f"{LOG_RESPONSE_COLUMN}_count": "Valid_replicate_count",
        f"{LOG_RESPONSE_COLUMN}_mean": "Mean_log10_response",
        f"{LOG_RESPONSE_COLUMN}_std": "SD_log10_response",
    }
)

for column in passthrough_present:
    response_summary = response_summary.rename(
        columns={f"{column}_first": column}
    )

print()
print("Combinazioni campione-composto disponibili:", len(response_summary))


# =====================================================================
# 12. RECUPERO AUTOMATICO DEL FILE GC-MS DI ORIGINE
# =====================================================================

print()
print("=" * 72)
print("RECUPERO DEI PARAMETRI DAL FILE GC-MS DI ORIGINE")
print("=" * 72)

source_file = find_file_with_required_sheets(
    WORKING_DIRECTORY,
    required_sheets={COMPOUND_MAPPING_SHEET, METHOD_SHEET_NAME},
    exclude_names={
        IPA_RESULTS_FILE.name,
        OUTPUT_FILE.name,
        SIMILARITY_CACHE_FILE.name
    }
)

if source_file is None:

    print(
        "Nessun file con i fogli "
        f"'{COMPOUND_MAPPING_SHEET}' e '{METHOD_SHEET_NAME}' è stato "
        "trovato in /content."
    )
    print("Caricare il file GC-MS di origine (lo stesso della cella 2).")

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError("È necessario caricare un solo file Excel.")

    source_file = WORKING_DIRECTORY / next(iter(uploaded))

print("File GC-MS di origine utilizzato:", source_file.name)

mapping_df = pd.read_excel(
    source_file,
    sheet_name=COMPOUND_MAPPING_SHEET,
    dtype={CAS_COLUMN: str}
)

check_required_columns(
    mapping_df, [IUPAC_COLUMN, CAS_COLUMN], COMPOUND_MAPPING_SHEET
)

mapping_df[CAS_COLUMN] = mapping_df[CAS_COLUMN].apply(normalize_cas)


# =====================================================================
# 13. IDENTIFICAZIONE DELLO STANDARD INTERNO
# =====================================================================

if ROLE_COLUMN not in mapping_df.columns:
    raise ValueError(
        f"Nel foglio '{COMPOUND_MAPPING_SHEET}' manca la colonna "
        f"'{ROLE_COLUMN}', necessaria per identificare lo standard "
        "interno."
    )

internal_standard_labels = {
    "internal standard", "internalstandard", "standard interno",
    "internal std", "is"
}

normalized_roles = mapping_df[ROLE_COLUMN].apply(normalize_role)

internal_standard_rows = mapping_df.loc[
    normalized_roles.isin(internal_standard_labels)
]

if len(internal_standard_rows) == 0:
    raise ValueError(
        f"Nessuna riga di '{COMPOUND_MAPPING_SHEET}' ha "
        f"'{ROLE_COLUMN}' impostato su un valore che indichi lo "
        "standard interno (es. 'Internal standard')."
    )

if len(internal_standard_rows) > 1:
    raise ValueError(
        f"Più di una riga di '{COMPOUND_MAPPING_SHEET}' indica uno "
        "standard interno. Il metodo attuale ne prevede uno solo."
    )

internal_standard_row = internal_standard_rows.iloc[0]

internal_standard_cas = internal_standard_row[CAS_COLUMN]
internal_standard_iupac = internal_standard_row[IUPAC_COLUMN]

print()
print("Standard interno identificato:")
print(" - IUPAC:", internal_standard_iupac)
print(" - CAS:", internal_standard_cas)

internal_standard_pubchem = get_pubchem_identity(internal_standard_cas)

internal_standard_context = {
    "IUPAC_name": internal_standard_iupac,
    "CAS": internal_standard_cas,
    **internal_standard_pubchem
}


# =====================================================================
# 14. LETTURA DELLA CONCENTRAZIONE DELLO STANDARD INTERNO NEL CAMPIONE
# =====================================================================

method_raw_df = pd.read_excel(
    source_file, sheet_name=METHOD_SHEET_NAME, header=None
)

internal_standard_concentration_sample = (
    parse_internal_standard_concentration(method_raw_df)
)

if pd.isna(internal_standard_concentration_sample):
    raise ValueError(
        f"Nel foglio '{METHOD_SHEET_NAME}' non è stata trovata la "
        "concentrazione dello standard interno nel campione.\n\n"
        "Aggiungere una riga con un'etichetta che contenga "
        "'concentrazione' e 'standard interno', per esempio:\n"
        "  concentrazione standard interno nel campione [µg/kg]    <valore>"
    )

print()
print(
    "Concentrazione dello standard interno nel campione "
    f"({SAMPLE_CONCENTRATION_UNIT}):",
    internal_standard_concentration_sample
)


RECUPERO DEI DATI IPA A LIVELLO DI REPLICA
È stata utilizzata la variabile ipa_df presente nella memoria della sessione.
Righe disponibili: 24

Combinazioni campione-composto disponibili: 8

RECUPERO DEI PARAMETRI DAL FILE GC-MS DI ORIGINE
File GC-MS di origine utilizzato: GCMS_Areas.xlsx

Standard interno identificato:
 - IUPAC: eugenol
 - CAS: 97-53-0

Concentrazione dello standard interno nel campione (µg/kg): 25.0


In [8]:
# @title
# =====================================================================
# 15. CONVERSIONE DELLO STANDARD INTERNO IN HEADSPACE
#     (SOLO IN MODALITÀ "headspace")
# =====================================================================

internal_standard_concentration_headspace = np.nan
internal_standard_headspace_log10_sd = np.nan
internal_standard_conversion_method = "non applicabile (modalità campione)"

if COMPOSITION_MODE == "headspace":

    print()
    print("=" * 72)
    print("CONVERSIONE DELLO STANDARD INTERNO IN CONCENTRAZIONE DI HEADSPACE")
    print("=" * 72)

    matrix_composition = parse_matrix_composition(method_raw_df)

    equilibrium_temperature_k = (
        matrix_composition["equilibrium_temperature_c"] + 273.15
    )
    water_fraction = matrix_composition["water_percent"] / 100.0
    lipid_fraction = matrix_composition["lipid_percent"] / 100.0

    print("Contenuto d'acqua:", water_fraction * 100, "%")
    print("Contenuto lipidico:", lipid_fraction * 100, "%")
    print(
        "Temperatura di equilibrio:",
        matrix_composition["equilibrium_temperature_c"], "°C"
    )

    print()
    print(
        "Recupero costante di Henry e logP dello standard interno "
        "tramite PubChem e ricerca web..."
    )

    is_physchem = None
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            if attempt > 1:
                print(f"Nuovo tentativo API ({attempt}/{MAX_RETRIES})...")

            is_physchem = search_internal_standard_physicochemical(
                is_cas=internal_standard_cas,
                is_iupac=internal_standard_iupac,
                is_pubchem=internal_standard_pubchem
            )

            break

        except Exception as error:

            last_error = error
            print(f"Errore nella chiamata API: {error}")
            time.sleep(PAUSE_SECONDS)

    henry_dimensionless = np.nan
    logp_value = np.nan

    if is_physchem is not None:

        print(
            "Costante di Henry recuperata:",
            is_physchem.henry_constant_value,
            is_physchem.henry_constant_unit
        )
        print("LogP recuperato:", is_physchem.logp_value)
        print("Confidenza:", is_physchem.confidence)

        if is_physchem.source_url:
            print("Fonte:", is_physchem.source_url)

        henry_dimensionless = convert_henry_to_dimensionless(
            is_physchem.henry_constant_value,
            is_physchem.henry_constant_unit,
            equilibrium_temperature_k
        )

        logp_value = (
            is_physchem.logp_value
            if is_physchem.logp_value is not None
            else np.nan
        )

    else:
        print(
            "Impossibile recuperare le proprietà fisico-chimiche "
            f"dello standard interno dopo i tentativi previsti "
            f"(ultimo errore: {last_error})."
        )

    specific_volume_water = water_fraction / WATER_DENSITY_KG_M3
    specific_volume_lipid = lipid_fraction / DEFAULT_LIPID_DENSITY_KG_M3

    henry_ok = pd.notna(henry_dimensionless) and henry_dimensionless > 0
    logp_ok = pd.notna(logp_value)

    if henry_ok and logp_ok:

        k_ow = 10 ** logp_value
        denominator = (
            specific_volume_water + k_ow * specific_volume_lipid
        )

        log10_k_matrix_air = (
            math.log10(henry_dimensionless) - math.log10(denominator)
        )
        internal_standard_headspace_log10_sd = RIGOROUS_RESIDUAL_LOG_SD
        internal_standard_conversion_method = (
            "Rigoroso (costante di Henry + logP dello standard interno)"
        )

    elif henry_ok and not logp_ok:

        log10_k_matrix_air = (
            math.log10(henry_dimensionless)
            - math.log10(specific_volume_water)
        )
        internal_standard_headspace_log10_sd = WATER_ONLY_RESIDUAL_LOG_SD
        internal_standard_conversion_method = (
            "Approssimato (solo Henry, correzione lipidica non "
            "applicata: logP dello standard non reperito — possibile "
            "sovrastima se lo standard è lipofilo)"
        )

    else:

        log10_k_matrix_air = np.nan
        internal_standard_headspace_log10_sd = np.nan
        internal_standard_conversion_method = (
            "Non calcolabile: costante di Henry e/o logP dello "
            "standard interno non reperiti"
        )

    if pd.notna(log10_k_matrix_air):

        internal_standard_concentration_headspace = 10 ** (
            math.log10(internal_standard_concentration_sample)
            + log10_k_matrix_air
        )

    print()
    print("Metodo di conversione:", internal_standard_conversion_method)
    print(
        "Concentrazione stimata dello standard interno in headspace "
        f"({HEADSPACE_CONCENTRATION_UNIT}):",
        internal_standard_concentration_headspace
    )

In [9]:
# @title
# =====================================================================
# 16. ELENCO DEGLI ANALITI DA CLASSIFICARE
# =====================================================================

unique_analytes = (
    response_summary
    .loc[
        response_summary[CAS_COLUMN] != internal_standard_cas,
        [CAS_COLUMN, IUPAC_COLUMN]
    ]
    .drop_duplicates(subset=[CAS_COLUMN])
    .dropna(subset=[CAS_COLUMN, IUPAC_COLUMN])
    .reset_index(drop=True)
)

print()
print("Analiti da classificare:", len(unique_analytes))


# =====================================================================
# 17. RIUSO DI UNA CACHE DI SIMILARITÀ GIÀ DISPONIBILE
# =====================================================================

cached_similarity_df = None

if (
    "similarity_df" in globals()
    and isinstance(similarity_df, pd.DataFrame)
    and not similarity_df.empty
    and set(unique_analytes[CAS_COLUMN]).issubset(
        set(similarity_df[CAS_COLUMN])
    )
):

    cached_similarity_df = similarity_df.copy()

    print(
        "È stata riutilizzata la variabile similarity_df già presente "
        "in sessione (nessuna nuova chiamata API)."
    )

elif SIMILARITY_CACHE_FILE.exists():

    try:
        candidate_cache_df = pd.read_excel(
            SIMILARITY_CACHE_FILE,
            sheet_name="Similarity_Cache",
            dtype={CAS_COLUMN: str}
        )

        candidate_cache_df[CAS_COLUMN] = (
            candidate_cache_df[CAS_COLUMN].apply(normalize_cas)
        )

        if set(unique_analytes[CAS_COLUMN]).issubset(
            set(candidate_cache_df[CAS_COLUMN])
        ):

            cached_similarity_df = candidate_cache_df

            print(
                f"È stato riutilizzato il file esistente "
                f"'{SIMILARITY_CACHE_FILE.name}' "
                "(nessuna nuova chiamata API)."
            )

    except Exception:
        cached_similarity_df = None


# =====================================================================
# 18. CLASSIFICAZIONE AUTOMATICA DELLA SIMILARITÀ (SE NECESSARIA)
# =====================================================================

if cached_similarity_df is not None:

    similarity_records = cached_similarity_df.to_dict("records")

else:

    print()
    print("=" * 72)
    print("CLASSIFICAZIONE AUTOMATICA DELLA SIMILARITÀ ANALITA-STANDARD")
    print("=" * 72)

    similarity_records = []

    for index, row in unique_analytes.iterrows():

        analyte_cas = row[CAS_COLUMN]
        analyte_iupac = row[IUPAC_COLUMN]

        print()
        print("=" * 72)
        print(
            f"[{index + 1}/{len(unique_analytes)}] "
            f"{analyte_iupac} — {analyte_cas}"
        )

        cas_valid = validate_cas(analyte_cas)

        if not cas_valid:

            print(
                "CAS formalmente non valido: "
                "la ricerca online non verrà eseguita."
            )

            similarity_records.append(
                {
                    IUPAC_COLUMN: analyte_iupac,
                    CAS_COLUMN: analyte_cas,
                    "Identity_match": "conflicting",
                    "Common_name": "",
                    "Chemical_class": "",
                    "Similarity_level": "",
                    "Rationale": "",
                    "Source_name": "",
                    "Source_url": "",
                    "Confidence": "low",
                    "Review_required": True,
                    "Notes": "Numero CAS formalmente non valido",
                    "PubChem_CID": np.nan,
                    "PubChem_URL": "",
                    "PubChem_error": "",
                    "Retrieved_on": str(date.today())
                }
            )

            continue

        analyte_pubchem = get_pubchem_identity(analyte_cas)

        similarity_result = None
        last_error = None

        for attempt in range(1, MAX_RETRIES + 1):

            try:

                if attempt > 1:
                    print(f"Nuovo tentativo API ({attempt}/{MAX_RETRIES})...")

                similarity_result = search_similarity_online(
                    analyte_cas=analyte_cas,
                    analyte_iupac=analyte_iupac,
                    analyte_pubchem=analyte_pubchem,
                    internal_standard_context=internal_standard_context
                )

                break

            except Exception as error:

                last_error = error
                print(f"Errore nella chiamata API: {error}")
                time.sleep(PAUSE_SECONDS)

        if similarity_result is None:

            print(
                "Impossibile ottenere una classificazione per questo "
                "composto dopo i tentativi previsti."
            )

            similarity_records.append(
                {
                    IUPAC_COLUMN: analyte_iupac,
                    CAS_COLUMN: analyte_cas,
                    "Identity_match": "not_found",
                    "Common_name": "",
                    "Chemical_class": "",
                    "Similarity_level": "",
                    "Rationale": "",
                    "Source_name": "",
                    "Source_url": "",
                    "Confidence": "low",
                    "Review_required": True,
                    "Notes": f"Errore API: {last_error}",
                    "PubChem_CID": analyte_pubchem.get("PubChem_CID"),
                    "PubChem_URL": analyte_pubchem.get("PubChem_URL") or "",
                    "PubChem_error": analyte_pubchem.get("PubChem_error") or "",
                    "Retrieved_on": str(date.today())
                }
            )

            time.sleep(PAUSE_SECONDS)
            continue

        print("Similarity_level:", similarity_result.similarity_level)
        print("Confidence:", similarity_result.confidence)

        similarity_records.append(
            {
                IUPAC_COLUMN: analyte_iupac,
                CAS_COLUMN: analyte_cas,
                "Identity_match": similarity_result.identity_match,
                "Common_name": similarity_result.common_name or "",
                "Chemical_class": similarity_result.chemical_class or "",
                "Similarity_level": similarity_result.similarity_level,
                "Rationale": similarity_result.rationale or "",
                "Source_name": similarity_result.source_name or "",
                "Source_url": similarity_result.source_url or "",
                "Confidence": similarity_result.confidence,
                "Review_required": similarity_result.review_required,
                "Notes": similarity_result.notes or "",
                "PubChem_CID": analyte_pubchem.get("PubChem_CID"),
                "PubChem_URL": analyte_pubchem.get("PubChem_URL") or "",
                "PubChem_error": analyte_pubchem.get("PubChem_error") or "",
                "Retrieved_on": str(date.today())
            }
        )

        time.sleep(PAUSE_SECONDS)


similarity_df = pd.DataFrame(similarity_records)

similarity_df[CAS_COLUMN] = similarity_df[CAS_COLUMN].apply(normalize_cas)

similarity_df["Similarity_level_normalized"] = (
    similarity_df["Similarity_level"].apply(normalize_similarity_level)
)

unrecognized_levels = (
    similarity_df.loc[
        similarity_df["Similarity_level_normalized"].notna()
        & ~similarity_df["Similarity_level_normalized"].isin(
            SIMILARITY_PRIOR_TABLE.keys()
        ),
        "Similarity_level"
    ]
    .unique()
    .tolist()
)

if unrecognized_levels:
    print()
    print(
        "ATTENZIONE: il modello ha restituito i seguenti valori di "
        "similarità non riconosciuti (trattati come mancanti):"
    )
    for level in unrecognized_levels:
        print(" -", level)


Analiti da classificare: 4

CLASSIFICAZIONE AUTOMATICA DELLA SIMILARITÀ ANALITA-STANDARD

[1/4] 2,3,5-trimethylpyrazine — 14667-55-1
Similarity_level: comportamento hs spme molto diverso
Confidence: high

[2/4] 2-methoxyphenol — 90-05-1
Similarity_level: stessa classe chimica
Confidence: high

[3/4] 3,7-dimethylocta-1,6-dien-3-ol — 78-70-6
Similarity_level: classe diversa
Confidence: high

[4/4] phenol — 108-95-2
Similarity_level: comportamento hs spme molto diverso
Confidence: high


In [10]:
# @title
# =====================================================================
# 19. SALVATAGGIO DELLA CACHE DI SIMILARITÀ
# =====================================================================

with pd.ExcelWriter(SIMILARITY_CACHE_FILE, engine="xlsxwriter") as writer:

    similarity_df.to_excel(
        writer, sheet_name="Similarity_Cache", index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True, "bg_color": "#D9EAF7", "border": 1,
            "text_wrap": True, "valign": "top"
        }
    )

    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})

    worksheet = writer.sheets["Similarity_Cache"]
    worksheet.freeze_panes(1, 0)

    if len(similarity_df.columns) > 0:
        worksheet.autofilter(
            0, 0, max(len(similarity_df), 1), len(similarity_df.columns) - 1
        )

    for column_index, column_name in enumerate(similarity_df.columns):

        worksheet.write(0, column_index, column_name, header_format)

        width = calculate_column_width(similarity_df, column_name)

        worksheet.set_column(column_index, column_index, width, wrap_format)

print()
print("Cache di similarità salvata in:", SIMILARITY_CACHE_FILE.name)


# =====================================================================
# 20. ABBINAMENTO CON I DATI DI RISPOSTA ANALITICA
# =====================================================================

estimate_df = response_summary.merge(
    similarity_df[
        [CAS_COLUMN, "Similarity_level_normalized", "Confidence",
         "Review_required", "Rationale", "Chemical_class"]
    ].rename(columns={"Confidence": "Similarity_confidence"}),
    on=CAS_COLUMN,
    how="left"
)

# Lo standard interno stesso non viene stimato (per definizione R=1).
estimate_df = estimate_df.loc[
    estimate_df[CAS_COLUMN] != internal_standard_cas
].reset_index(drop=True)

if COMPOSITION_MODE == "headspace":
    is_reference_log10_concentration = (
        math.log10(internal_standard_concentration_headspace)
        if pd.notna(internal_standard_concentration_headspace)
        and internal_standard_concentration_headspace > 0
        else np.nan
    )
    is_reference_log10_sd = internal_standard_headspace_log10_sd
    reference_concentration_unit = HEADSPACE_CONCENTRATION_UNIT
else:
    is_reference_log10_concentration = (
        math.log10(internal_standard_concentration_sample)
        if internal_standard_concentration_sample > 0
        else np.nan
    )
    is_reference_log10_sd = 0.0
    reference_concentration_unit = SAMPLE_CONCENTRATION_UNIT

estimate_df["Concentration_unit"] = reference_concentration_unit


# =====================================================================
# 21. PRIOR SU DELTA IN FUNZIONE DEL LIVELLO DI SIMILARITÀ
# =====================================================================

priors = estimate_df["Similarity_level_normalized"].apply(
    get_similarity_prior
)

estimate_df["Delta_prior_mu"] = pd.to_numeric(
    pd.Series([prior[0] for prior in priors], index=estimate_df.index),
    errors="coerce"
)
estimate_df["Delta_prior_sigma"] = pd.to_numeric(
    pd.Series([prior[1] for prior in priors], index=estimate_df.index),
    errors="coerce"
)


# =====================================================================
# 22. SIGMA DI log10(R_i): EMPIRICA O DI DEFAULT
# =====================================================================

has_enough_replicates = (
    estimate_df["Valid_replicate_count"] >= MIN_REPLICATES_FOR_EMPIRICAL_SD
)

estimate_df["Response_log_sd_used"] = np.where(
    has_enough_replicates & estimate_df["SD_log10_response"].notna(),
    estimate_df["SD_log10_response"],
    DEFAULT_REPLICATE_LOG_SD
)

estimate_df["Response_log_sd_source"] = np.where(
    has_enough_replicates & estimate_df["SD_log10_response"].notna(),
    "Empirica (deviazione standard tra repliche)",
    "Valore di default (repliche insufficienti)"
)


# =====================================================================
# 23. COMBINAZIONE NORMALE-NORMALE: MEDIA E SIGMA TOTALI
# =====================================================================

data_available = (
    estimate_df["Mean_log10_response"].notna()
    & estimate_df["Delta_prior_mu"].notna()
    & estimate_df["Delta_prior_sigma"].notna()
    & pd.notna(is_reference_log10_concentration)
    & pd.notna(is_reference_log10_sd)
)

estimate_df["Concentration_calculable"] = np.where(
    data_available, "Sì", "No"
)

estimate_df["Estimated_log10_concentration"] = np.where(
    data_available,
    is_reference_log10_concentration
    + estimate_df["Mean_log10_response"]
    - estimate_df["Delta_prior_mu"],
    np.nan
)

estimate_df["Estimated_log10_sd"] = np.where(
    data_available,
    np.sqrt(
        estimate_df["Response_log_sd_used"] ** 2
        + estimate_df["Delta_prior_sigma"] ** 2
        + is_reference_log10_sd ** 2
    ),
    np.nan
)


# =====================================================================
# 24. MEDIANA E INTERVALLO CREDIBILE
# =====================================================================

z_score = norm.ppf(0.5 + CREDIBLE_INTERVAL_PROBABILITY / 2.0)

estimate_df["Estimated_concentration_median"] = np.where(
    estimate_df["Estimated_log10_concentration"].notna(),
    10 ** estimate_df["Estimated_log10_concentration"],
    np.nan
)

estimate_df["Estimated_concentration_low"] = np.where(
    estimate_df["Estimated_log10_concentration"].notna(),
    10 ** (
        estimate_df["Estimated_log10_concentration"]
        - z_score * estimate_df["Estimated_log10_sd"]
    ),
    np.nan
)

estimate_df["Estimated_concentration_high"] = np.where(
    estimate_df["Estimated_log10_concentration"].notna(),
    10 ** (
        estimate_df["Estimated_log10_concentration"]
        + z_score * estimate_df["Estimated_log10_sd"]
    ),
    np.nan
)

estimate_df["Credible_interval_width_log"] = (
    2.0 * z_score * estimate_df["Estimated_log10_sd"]
)

estimate_df["Concentration_decade"] = (
    estimate_df["Estimated_concentration_median"].apply(classify_decade)
)


Cache di similarità salvata in: Similarity_Cache.xlsx


In [11]:
# @title
# =====================================================================
# 25. CONFRONTO CON LA SOGLIA OLFATTIVA (SOLO IN MODALITÀ "headspace")
# =====================================================================

if COMPOSITION_MODE == "headspace" and THRESHOLD_COLUMN in estimate_df.columns:

    has_threshold = (
        estimate_df[THRESHOLD_COLUMN].notna()
        & (estimate_df[THRESHOLD_COLUMN] > 0)
        & estimate_df["Estimated_log10_concentration"].notna()
    )

    log10_threshold = np.log10(
        estimate_df[THRESHOLD_COLUMN].where(
            estimate_df[THRESHOLD_COLUMN] > 0
        )
    )

    estimate_df["Log10_ratio_to_odor_threshold"] = np.where(
        has_threshold,
        estimate_df["Estimated_log10_concentration"] - log10_threshold,
        np.nan
    )

    z_threshold = (
        estimate_df["Estimated_log10_concentration"] - log10_threshold
    ) / estimate_df["Estimated_log10_sd"]

    estimate_df["Probability_above_odor_threshold"] = np.where(
        has_threshold,
        norm.cdf(z_threshold),
        np.nan
    )


# =====================================================================
# 26. QUALITÀ DELLA STIMA E DISTANZA DALLE ANCORE
# =====================================================================

estimate_df["Model_quality"] = estimate_df.apply(
    lambda row: classify_model_quality(
        row["Similarity_level_normalized"], row["Valid_replicate_count"]
    ),
    axis=1
)

# Nella Versione 4A non sono ancora presenti composti "ancora":
# questa colonna è predisposta per la Versione 4B.
estimate_df["Anchor_distance"] = (
    "Non applicabile: nessuna ancora inserita (Versione 4A)"
)


# =====================================================================
# 27. NOTE DI ELABORAZIONE
# =====================================================================

estimate_df["Processing_note"] = ""

if COMPOSITION_MODE == "headspace" and pd.isna(
    internal_standard_concentration_headspace
):
    estimate_df["Processing_note"] = (
        "Concentrazione dello standard interno in headspace non "
        "calcolabile: vedi metodo di conversione nel riepilogo"
    )

estimate_df.loc[
    (estimate_df["Processing_note"] == "")
    & estimate_df["Similarity_level_normalized"].isna(),
    "Processing_note"
] = "Similarità non classificata automaticamente (vedi Similarity_Cache)"

estimate_df.loc[
    (estimate_df["Processing_note"] == "")
    & (estimate_df["Review_required"] == True),
    "Processing_note"
] = "Classificazione della similarità segnalata per revisione dal modello"

estimate_df.loc[
    (estimate_df["Processing_note"] == "")
    & estimate_df["Mean_log10_response"].isna(),
    "Processing_note"
] = "Nessuna area normalizzata disponibile per questo composto"

estimate_df.loc[
    (estimate_df["Processing_note"] == "")
    & (
        estimate_df["Valid_replicate_count"]
        < MIN_REPLICATES_FOR_EMPIRICAL_SD
    ),
    "Processing_note"
] = "Deviazione standard tra repliche non stimabile: usato valore di default"


# =====================================================================
# 28. ORDINE DELLE COLONNE
# =====================================================================

output_columns = [
    SAMPLE_COLUMN, COMPOUND_COLUMN, IUPAC_COLUMN, CAS_COLUMN,
    "Valid_replicate_count", "Mean_log10_response", "SD_log10_response",
    "Response_log_sd_used", "Response_log_sd_source",
    "Chemical_class", "Similarity_level_normalized", "Rationale",
    "Similarity_confidence", "Review_required",
    "Delta_prior_mu", "Delta_prior_sigma",
    "Concentration_unit", "Concentration_calculable",
    "Estimated_log10_concentration", "Estimated_log10_sd",
    "Estimated_concentration_median", "Estimated_concentration_low",
    "Estimated_concentration_high", "Credible_interval_width_log",
    "Concentration_decade",
    THRESHOLD_COLUMN,
    "Log10_ratio_to_odor_threshold", "Probability_above_odor_threshold",
    "Model_quality", "Anchor_distance", "Processing_note"
] + [
    column for column in passthrough_present if column != THRESHOLD_COLUMN
]

output_columns = [
    column for column in output_columns if column in estimate_df.columns
]

estimate_df = estimate_df[output_columns]

estimate_df = estimate_df.sort_values(
    by=[SAMPLE_COLUMN, COMPOUND_COLUMN], ascending=[True, True]
).reset_index(drop=True)


# =====================================================================
# 29. TABELLA DI REVISIONE MANUALE
# =====================================================================

review_mask = (
    (estimate_df["Concentration_calculable"] == "No")
    | (estimate_df["Processing_note"] != "")
    | estimate_df["Model_quality"].str.startswith("Bassa", na=False)
    | estimate_df["Model_quality"].str.startswith(
        "Non valutabile", na=False
    )
)

review_df = estimate_df.loc[review_mask].copy()


# =====================================================================
# 30. TABELLA DEI PRIOR UTILIZZATI (TRASPARENZA METODOLOGICA)
# =====================================================================

priors_used_df = pd.DataFrame(
    [
        {
            "Similarity_level": level,
            "Delta_prior_mu": values["mu"],
            "Delta_prior_sigma": values["sigma"]
        }
        for level, values in SIMILARITY_PRIOR_TABLE.items()
    ]
)

priors_used_df["Default_replicate_log_sd"] = DEFAULT_REPLICATE_LOG_SD
priors_used_df["Credible_interval_probability"] = (
    CREDIBLE_INTERVAL_PROBABILITY
)


# =====================================================================
# 31. TABELLA DI RIEPILOGO
# =====================================================================

summary_rows = [
    ("File GC-MS di origine utilizzato", source_file.name),
    ("Modalità selezionata", COMPOSITION_MODE),
    ("Standard interno (IUPAC)", internal_standard_iupac),
    ("Standard interno (CAS)", internal_standard_cas),
    (
        "Concentrazione standard interno nel campione",
        f"{internal_standard_concentration_sample} {SAMPLE_CONCENTRATION_UNIT}"
    ),
]

if COMPOSITION_MODE == "headspace":
    summary_rows.extend(
        [
            (
                "Metodo di conversione standard interno->headspace",
                internal_standard_conversion_method
            ),
            (
                "Concentrazione standard interno in headspace",
                f"{internal_standard_concentration_headspace} "
                f"{HEADSPACE_CONCENTRATION_UNIT}"
                if pd.notna(internal_standard_concentration_headspace)
                else "Non calcolabile"
            ),
        ]
    )

summary_rows.extend(
    [
        (
            "Combinazioni campione-composto elaborate",
            len(estimate_df)
        ),
        (
            "Stime di concentrazione calcolabili",
            int((estimate_df["Concentration_calculable"] == "Sì").sum())
        ),
        (
            "Classificazioni di similarità con revisione richiesta",
            int(similarity_df["Review_required"].fillna(False).sum())
        ),
        ("Record da revisionare (totale)", len(review_df)),
        (
            "Livello dell'intervallo credibile",
            f"{int(CREDIBLE_INTERVAL_PROBABILITY * 100)}%"
        ),
        (
            "Versione del modello",
            "4A — prior per livello di similarità determinato "
            "automaticamente, nessuna ancora"
        ),
    ]
)

summary_df = pd.DataFrame(summary_rows, columns=["Indicatore", "Valore"])

In [12]:
# @title
# =====================================================================
# 32. SALVATAGGIO CON XLSXWRITER
# =====================================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="xlsxwriter") as writer:

    estimate_df.to_excel(
        writer, sheet_name="Concentration_Estimates", index=False
    )
    priors_used_df.to_excel(writer, sheet_name="Priors_Used", index=False)
    review_df.to_excel(writer, sheet_name="Manual_Review", index=False)
    summary_df.to_excel(
        writer, sheet_name="Processing_Summary", index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True, "bg_color": "#D9EAF7", "border": 1,
            "text_wrap": True, "valign": "top"
        }
    )
    wrap_format = workbook.add_format({"text_wrap": True, "valign": "top"})
    decimal_format = workbook.add_format(
        {"num_format": "0.000", "valign": "top"}
    )
    scientific_format = workbook.add_format(
        {"num_format": "0.000E+00", "valign": "top"}
    )
    percent_format = workbook.add_format(
        {"num_format": "0.0%", "valign": "top"}
    )

    dataframe_by_sheet = {
        "Concentration_Estimates": estimate_df,
        "Priors_Used": priors_used_df,
        "Manual_Review": review_df,
        "Processing_Summary": summary_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]
        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(28)

        if len(dataframe.columns) > 0:
            worksheet.autofilter(
                0, 0, max(len(dataframe), 1), len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(dataframe.columns):

            worksheet.write(0, column_index, column_name, header_format)

            width = calculate_column_width(dataframe, column_name)

            worksheet.set_column(
                column_index, column_index, width, wrap_format
            )

    estimates_ws = writer.sheets["Concentration_Estimates"]

    for column_name in [
        "Estimated_concentration_median", "Estimated_concentration_low",
        "Estimated_concentration_high", THRESHOLD_COLUMN
    ]:
        if column_name in estimate_df.columns:
            column_index = estimate_df.columns.get_loc(column_name)
            estimates_ws.set_column(
                column_index, column_index, 20, scientific_format
            )

    for column_name in [
        "Mean_log10_response", "SD_log10_response",
        "Response_log_sd_used", "Delta_prior_mu", "Delta_prior_sigma",
        "Estimated_log10_concentration", "Estimated_log10_sd",
        "Credible_interval_width_log", "Log10_ratio_to_odor_threshold"
    ]:
        if column_name in estimate_df.columns:
            column_index = estimate_df.columns.get_loc(column_name)
            estimates_ws.set_column(
                column_index, column_index, 18, decimal_format
            )

    if "Probability_above_odor_threshold" in estimate_df.columns:
        column_index = estimate_df.columns.get_loc(
            "Probability_above_odor_threshold"
        )
        estimates_ws.set_column(
            column_index, column_index, 16, percent_format
        )

    if (
        "Estimated_log10_concentration" in estimate_df.columns
        and len(estimate_df) > 0
    ):
        log_col = estimate_df.columns.get_loc(
            "Estimated_log10_concentration"
        )
        estimates_ws.conditional_format(
            1, log_col, len(estimate_df), log_col,
            {
                "type": "3_color_scale",
                "min_color": "#F8696B",
                "mid_color": "#FFEB84",
                "max_color": "#63BE7B"
            }
        )


# =====================================================================
# 33. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("STIMA DELLA CONCENTRAZIONE COMPLETATA (VERSIONE 4A)")
print(f"Modalità: {COMPOSITION_MODE}")
print("=" * 72)

print("File creati:", OUTPUT_FILE.name, "e", SIMILARITY_CACHE_FILE.name)
print("Righe elaborate:", len(estimate_df))
print(
    "Stime calcolabili:",
    int((estimate_df["Concentration_calculable"] == "Sì").sum())
)
print("Record da controllare manualmente:", len(review_df))

print()
if COMPOSITION_MODE == "headspace":
    print(
        "PROMEMORIA: le concentrazioni stimate sono riferite "
        "all'headspace della vial, non alla matrice del campione. "
        "La conversione dello standard interno in headspace dipende "
        "dalla sua costante di Henry/logP e dalla composizione della "
        "matrice — controllare il metodo di conversione riportato in "
        "Processing_Summary. La cella 5 non è necessaria in questa "
        "modalità."
    )
else:
    print(
        "PROMEMORIA: questa è una stima esplorativa basata su prior "
        "metodologici, non una quantificazione validata. Le "
        "concentrazioni sono riferite alla matrice del campione: per "
        "il confronto con la soglia olfattiva in aria, eseguire la "
        "cella 5."
    )

print()
print("Anteprima dei risultati:")

preview_columns = [
    SAMPLE_COLUMN, COMPOUND_COLUMN, CAS_COLUMN,
    "Similarity_level_normalized", "Confidence",
    "Estimated_concentration_median", "Estimated_concentration_low",
    "Estimated_concentration_high", "Concentration_unit",
    "Concentration_decade", "Probability_above_odor_threshold",
    "Model_quality", "Processing_note"
]

preview_columns = [
    column for column in preview_columns if column in estimate_df.columns
]

display(estimate_df[preview_columns].head(30))

files.download(str(OUTPUT_FILE))
files.download(str(SIMILARITY_CACHE_FILE))


STIMA DELLA CONCENTRAZIONE COMPLETATA (VERSIONE 4A)
Modalità: sample
File creati: GCMS_Concentration_Estimates.xlsx e Similarity_Cache.xlsx
Righe elaborate: 8
Stime calcolabili: 8
Record da controllare manualmente: 6

PROMEMORIA: questa è una stima esplorativa basata su prior metodologici, non una quantificazione validata. Le concentrazioni sono riferite alla matrice del campione: per il confronto con la soglia olfattiva in aria, eseguire la cella 5.

Anteprima dei risultati:


,Sample_ID,GCMS_column_name,CAS,Similarity_level_normalized,Confidence,Estimated_concentration_median,Estimated_concentration_low,Estimated_concentration_high,Concentration_unit,Concentration_decade,Model_quality,Processing_note
0,Cacao_01,"2,3,5-trimethylpyrazine",14667-55-1,comportamento hs spme molto diverso,medium,41.572010,0.003211,538226.900994,µg/kg,10 – 100,Bassa: classe diversa o comportamento HS-SPME ...,
1,Cacao_01,2-methoxyphenol,90-05-1,stessa classe chimica,medium,6.131217,0.138844,270.748488,µg/kg,1 – 10,"Moderata (solo su prior, nessuna ancora sperim...",
2,Cacao_01,linalool,78-70-6,classe diversa,medium,3.167923,0.001626,6172.977402,µg/kg,1 – 10,Bassa: classe diversa o comportamento HS-SPME ...,
3,Cacao_01,phenol,108-95-2,comportamento hs spme molto diverso,medium,4.638682,0.000358,60052.519633,µg/kg,1 – 10,Bassa: classe diversa o comportamento HS-SPME ...,
4,Cacao_02,"2,3,5-trimethylpyrazine",14667-55-1,comportamento hs spme molto diverso,medium,46.120735,0.003562,597108.930963,µg/kg,10 – 100,Bassa: classe diversa o comportamento HS-SPME ...,
5,Cacao_02,2-methoxyphenol,90-05-1,stessa classe chimica,medium,4.986411,0.112948,220.138644,µg/kg,1 – 10,"Moderata (solo su prior, nessuna ancora sperim...",
6,Cacao_02,linalool,78-70-6,classe diversa,medium,3.642717,0.001869,7098.723774,µg/kg,1 – 10,Bassa: classe diversa o comportamento HS-SPME ...,
7,Cacao_02,phenol,108-95-2,comportamento hs spme molto diverso,medium,6.333886,0.000489,82010.502042,µg/kg,1 – 10,Bassa: classe diversa o comportamento HS-SPME ...,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
# @title
# =====================================================================
# QUINTA CELLA — CONVERSIONE CAMPIONE→ARIA E CONFRONTO CON LA SOGLIA
# Versione corretta per una singola cella Google Colab
#
# Questa cella deve essere eseguita nella stessa sessione Colab
# utilizzata per le celle precedenti (in particolare la cella 4),
# oppure dopo aver caricato manualmente i file che esse producono.
#
# NON richiede nuove chiamate API.
#
# INPUT già presenti in /content (oppure variabile in memoria):
#   - variabile estimate_df prodotta dalla cella 4
#     oppure
#   - GCMS_Concentration_Estimates.xlsx, foglio Concentration_Estimates
#
#   - il file GC-MS di origine (lo stesso usato dalla cella 2),
#     contenente il foglio metodo_analitico
#
#   - un file Excel caricato dall'utente contenente il foglio
#       Physicochemical_Parameters
#           CAS
#           Henry_constant_value      (facoltativa)
#           Henry_constant_unit       (facoltativa; richiesta se
#                                       Henry_constant_value è presente)
#           LogP                      (facoltativa)
#           Volatility_class          (facoltativa; usata come
#                                       fallback quando la costante
#                                       di Henry non è disponibile)
#
# OUTPUT:
#   GCMS_Air_Concentration_Estimates.xlsx
#
# METODO:
#
#   La concentrazione nel campione (Cella 4) viene convertita nella
#   concentrazione equivalente in aria all'equilibrio, trattando il
#   campione come due fasi condensate (acqua + lipide) in equilibrio
#   con la fase gassosa:
#
#       K_matrice->aria = K_aw / ( w_acqua/rho_acqua
#                                  + K_ow * w_lipide/rho_lipide )
#
#   dove K_aw è la costante di Henry adimensionale (derivata dalla
#   costante di Henry dimensionale e dalla temperatura di equilibrio)
#   e K_ow = 10^logP è il coefficiente di ripartizione ottanolo/acqua,
#   usato come surrogato della fase lipidica.
#
#   Questa è una proprietà intensiva della composizione della
#   matrice: non dipende dalla massa del campione né dal volume
#   dell'headspace. Vale sotto l'ipotesi (ragionevole per analiti in
#   tracce) che il trasferimento verso l'aria non impoverisca in modo
#   apprezzabile la fase condensata.
#
#   Se logP non è disponibile, si usa un'approssimazione che ignora
#   la fase lipidica, con incertezza aumentata. ATTENZIONE: questa
#   approssimazione tende a SOVRAstimare la concentrazione in aria
#   dei composti lipofili, perché non riconosce che verrebbero in
#   parte trattenuti nella fase grassa del campione.
#
#   Se la costante di Henry non è disponibile, si utilizza un prior
#   statistico su log10(K_matrice->aria) basato su una classe di
#   volatilità dichiarata. QUESTI VALORI SONO PROVVISORI E NON
#   VALIDATI: vanno rivisti con dati sperimentali o di letteratura
#   specifici della matrice.
#
#   L'incertezza si combina in scala log10 come somma di normali
#   indipendenti (stessa logica della cella 4):
#
#       sigma_log10(C_aria) = sqrt( sigma_log10(C_campione)^2
#                                    + sigma_log10(K)^2 )
#
# VINCOLO IMPORTANTE:
#   La concentrazione nel campione (colonna Concentration_unit della
#   cella 4) deve essere espressa in µg/kg. Unità diverse vengono
#   segnalate e le relative righe non vengono elaborate.
# =====================================================================


# =====================================================================
# 0. INSTALLAZIONE DELLE LIBRERIE
# =====================================================================

!pip -q install --upgrade scipy pandas xlsxwriter openpyxl


# =====================================================================
# 1. IMPORTAZIONI
# =====================================================================

import re
import math
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import xlsxwriter
from scipy.stats import norm

from google.colab import files


# =====================================================================
# 2. PARAMETRI MODIFICABILI
# =====================================================================

WORKING_DIRECTORY = Path("/content")

CONCENTRATION_FILE = (
    WORKING_DIRECTORY / "GCMS_Concentration_Estimates.xlsx"
)
CONCENTRATION_SHEET = "Concentration_Estimates"

OUTPUT_FILE = (
    WORKING_DIRECTORY / "GCMS_Air_Concentration_Estimates.xlsx"
)

METHOD_SHEET_NAME = "metodo_analitico"
PHYSICOCHEMICAL_SHEET_NAME = "Physicochemical_Parameters"

SAMPLE_COLUMN = "Sample_ID"
COMPOUND_COLUMN = "GCMS_column_name"
IUPAC_COLUMN = "IUPAC_name"
CAS_COLUMN = "CAS"

MATRIX_LOG_CONCENTRATION_COLUMN = "Estimated_log10_concentration"
MATRIX_LOG_SD_COLUMN = "Estimated_log10_sd"
MATRIX_CONCENTRATION_UNIT_COLUMN = "Concentration_unit"
THRESHOLD_COLUMN = "Odor_threshold_air_ug_m3"

# Unità richiesta per la concentrazione nel campione, coerente con
# la formula di conversione utilizzata in questa cella.
REQUIRED_MATRIX_CONCENTRATION_UNIT = "µg/kg"

# Colonne attese nel foglio Physicochemical_Parameters.
HENRY_VALUE_COLUMN = "Henry_constant_value"
HENRY_UNIT_COLUMN = "Henry_constant_unit"
LOGP_COLUMN = "LogP"
VOLATILITY_CLASS_COLUMN = "Volatility_class"

# ---------------------------------------------------------------------
# Densità di riferimento delle sotto-fasi condensate.
# La densità della fase lipidica è un valore tipico (burro di cacao e
# simili grassi vegetali); se disponibile un valore più preciso per
# la matrice specifica, aggiornarlo qui.
# ---------------------------------------------------------------------

WATER_DENSITY_KG_M3 = 1000.0
DEFAULT_LIPID_DENSITY_KG_M3 = 900.0

# Costante universale dei gas, in Pa*m3/(mol*K).
GAS_CONSTANT_PA_M3_MOL_K = 8.314

# ---------------------------------------------------------------------
# Incertezza residua (log10) aggiunta al coefficiente di ripartizione
# stimato, a seconda del metodo utilizzato per calcolarlo.
# ---------------------------------------------------------------------

# Henry e logP entrambi noti: correzione completa a due fasi.
RIGOROUS_RESIDUAL_LOG_SD = 0.3

# Solo Henry noto, logP mancante: fase lipidica ignorata.
WATER_ONLY_RESIDUAL_LOG_SD = 0.6

# ---------------------------------------------------------------------
# Prior statistico su log10(K_matrice->aria), usato quando la
# costante di Henry non è disponibile.
#
# VALORI PROVVISORI, NON VALIDATI. Sono ordini di grandezza indicativi
# per composti volatili di aroma, da rivedere con dati sperimentali o
# di letteratura specifici della matrice prima di dare peso
# quantitativo ai risultati che ne derivano.
# ---------------------------------------------------------------------

VOLATILITY_CLASS_PRIOR_TABLE = {
    "molto volatile": {"mu": -1.5, "sigma": 1.0},
    "volatile": {"mu": -2.5, "sigma": 1.0},
    "moderatamente volatile": {"mu": -3.5, "sigma": 1.2},
    "poco volatile": {"mu": -4.5, "sigma": 1.5},
    "praticamente non volatile": {"mu": -5.5, "sigma": 2.0},
}

# Livello di probabilità per l'intervallo credibile riportato.
CREDIBLE_INTERVAL_PROBABILITY = 0.90


# =====================================================================
# 3. FUNZIONI DI SUPPORTO
# =====================================================================

def normalize_cas(value):
    """
    Uniforma trattini e spazi nel numero CAS.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = (
        value
        .replace("–", "-")
        .replace("—", "-")
        .replace("−", "-")
    )

    return re.sub(r"\s+", "", value)


def normalize_category_text(value):
    """
    Normalizza un'etichetta testuale (livello di volatilità, unità
    di misura) per renderla confrontabile in modo robusto.
    """
    if value is None or pd.isna(value):
        return None

    text = (
        str(value)
        .strip()
        .lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text if text else None


def strip_accents(text):
    """
    Rimuove gli accenti da una stringa, per rendere più robusto il
    riconoscimento delle etichette del foglio metodo_analitico
    (che sono testo libero inserito manualmente).
    """
    normalized = unicodedata.normalize("NFKD", text)

    return "".join(
        character
        for character in normalized
        if not unicodedata.combining(character)
    )


def normalize_label_key(value):
    """
    Normalizza un'etichetta per il riconoscimento per parole chiave.
    """
    if value is None or pd.isna(value):
        return ""

    text = strip_accents(str(value)).strip().lower()

    return re.sub(r"\s+", " ", text)


def parse_numeric_value(value):
    """
    Converte in float un valore che potrebbe contenere unità o
    testo accessorio (es. "1.1 g/ml"). Restituisce NaN se non
    interpretabile.
    """
    if value is None or pd.isna(value):
        return np.nan

    match = re.search(
        r"[-+]?\d+(?:[.,]\d+)?",
        str(value)
    )

    if not match:
        return np.nan

    return float(match.group(0).replace(",", "."))


def find_file_with_required_sheets(
    directory,
    required_sheets,
    exclude_names=frozenset()
):
    """
    Cerca nella cartella indicata un file Excel contenente tutti i
    fogli richiesti. Se ne trova più di uno, utilizza quello
    modificato più recentemente.
    """

    candidates = []

    for filepath in directory.glob("*.xlsx"):

        if filepath.name.startswith("~$"):
            continue

        if filepath.name in exclude_names:
            continue

        try:
            excel_file = pd.ExcelFile(filepath)

            if set(required_sheets).issubset(
                set(excel_file.sheet_names)
            ):
                candidates.append(filepath)

        except Exception:
            continue

    if len(candidates) == 0:
        return None

    if len(candidates) == 1:
        return candidates[0]

    candidates = sorted(
        candidates,
        key=lambda path: path.stat().st_mtime,
        reverse=True
    )

    print(
        "Sono stati trovati più file compatibili con i fogli "
        f"richiesti {sorted(required_sheets)}."
    )
    print(
        "Verrà utilizzato il file modificato più recentemente:"
    )
    print(candidates[0].name)

    print()
    print("Altri file compatibili rilevati:")

    for filepath in candidates[1:]:
        print(" -", filepath.name)

    return candidates[0]


def parse_method_sheet(dataframe):
    """
    Estrae i parametri del metodo analitico da un foglio a due
    colonne (etichetta, valore), riconoscendo le etichette per
    parole chiave per tollerare piccole variazioni di formulazione.

    Restituisce un dizionario con i parametri riconosciuti.
    Solleva un errore se mancano i parametri indispensabili al
    calcolo (contenuto lipidico, contenuto d'acqua, temperatura
    di equilibrio).
    """

    parsed = {
        "sample_mass_g": None,
        "dilution_text": None,
        "lipid_percent": None,
        "water_percent": None,
        "headspace_volume_ml": None,
        "spme_fiber": None,
        "equilibrium_temperature_c": None,
        "sample_density_g_ml": None,
        "split_ratio": None,
        "detector_type": None,
    }

    for _, row in dataframe.iterrows():

        if len(row) < 2:
            continue

        label_key = normalize_label_key(row.iloc[0])
        value = row.iloc[1]

        if not label_key:
            continue

        if "massa" in label_key and "campion" in label_key:
            parsed["sample_mass_g"] = parse_numeric_value(value)

        elif "diluizion" in label_key:
            parsed["dilution_text"] = value

        elif "lipid" in label_key:
            parsed["lipid_percent"] = parse_numeric_value(value)

        elif "acqua" in label_key:
            parsed["water_percent"] = parse_numeric_value(value)

        elif "head" in label_key and "space" in label_key:
            parsed["headspace_volume_ml"] = parse_numeric_value(
                value
            )

        elif "spme" in label_key:
            parsed["spme_fiber"] = value

        elif "temperat" in label_key and "equilibri" in label_key:
            parsed["equilibrium_temperature_c"] = parse_numeric_value(
                value
            )

        elif "densit" in label_key:
            parsed["sample_density_g_ml"] = parse_numeric_value(
                value
            )

        elif "splitt" in label_key or "split" in label_key:
            parsed["split_ratio"] = value

        elif "detector" in label_key or "rivelat" in label_key:
            parsed["detector_type"] = value

    missing_required = [
        name
        for name, key in [
            ("contenuto lipidico", "lipid_percent"),
            ("contenuto d'acqua", "water_percent"),
            ("temperatura di equilibrio", "equilibrium_temperature_c"),
        ]
        if parsed[key] is None or pd.isna(parsed[key])
    ]

    if missing_required:
        raise ValueError(
            f"Nel foglio '{METHOD_SHEET_NAME}' non sono stati "
            "riconosciuti i seguenti parametri indispensabili: "
            + ", ".join(missing_required)
            + ".\nControllare le etichette usate nel foglio."
        )

    return parsed


def convert_henry_to_dimensionless(value, unit_text, temperature_k):
    """
    Converte una costante di Henry in forma adimensionale K_aw
    (rapporto di concentrazioni gas/acqua).

    Unità riconosciute:
    - Pa*m3/mol
    - atm*m3/mol
    - già adimensionale (K_aw, "dimensionless", "adimensionale")

    Restituisce NaN se il valore o l'unità non sono utilizzabili.
    """

    if value is None or pd.isna(value):
        return np.nan

    if unit_text is None or pd.isna(unit_text):
        return np.nan

    unit_clean = str(unit_text).strip().lower().replace(" ", "")

    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan

    if "dimension" in unit_clean or unit_clean in {"kaw", "-"}:
        return value

    if "atm" in unit_clean:
        h_pa_m3_mol = value * 101325.0

    elif "pa" in unit_clean:
        h_pa_m3_mol = value

    else:
        return np.nan

    return h_pa_m3_mol / (
        GAS_CONSTANT_PA_M3_MOL_K * temperature_k
    )


def classify_transfer_method(
    henry_dimensionless,
    logp_value,
    volatility_class_normalized
):
    """
    Determina quale approccio è utilizzabile per stimare il
    coefficiente di ripartizione matrice->aria per un composto.
    """

    henry_available = (
        henry_dimensionless is not None
        and pd.notna(henry_dimensionless)
    )

    logp_available = (
        logp_value is not None
        and pd.notna(logp_value)
    )

    if henry_available and logp_available:
        return "rigoroso"

    if henry_available and not logp_available:
        return "acqua_solo"

    if (
        volatility_class_normalized is not None
        and volatility_class_normalized in VOLATILITY_CLASS_PRIOR_TABLE
    ):
        return "statistico"

    return "non_calcolabile"


def calculate_column_width(dataframe, column_name, maximum=45):
    """
    Calcola una larghezza leggibile per una colonna Excel.
    """
    header_length = len(str(column_name))

    if dataframe.empty:
        return min(max(header_length + 2, 12), maximum)

    maximum_value_length = (
        dataframe[column_name]
        .fillna("")
        .astype(str)
        .map(len)
        .max()
    )

    width = max(
        header_length,
        int(maximum_value_length)
    ) + 2

    return min(max(width, 12), maximum)


def check_required_columns(dataframe, required_columns, table_name):
    """
    Verifica che una tabella contenga tutte le colonne richieste.
    """
    missing = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing:
        raise ValueError(
            f"Nella tabella '{table_name}' mancano le colonne: "
            + ", ".join(missing)
        )
if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE == "headspace":
    print(
        "\033[1;32m"
        + "=" * 72 + "\n"
        + "MODALITÀ 'HEADSPACE' RILEVATA (scelta dell'utente)\n"
        + "Quindi le concentrazioni nel campione solido o liquido non "
        "sono state stimate\n"
        + "=" * 72
        + "\033[0m"
    )
    raise SystemExit()
# =====================================================================
# 4. RECUPERO DELLE STIME DI CONCENTRAZIONE NEL CAMPIONE (CELLA 4)
# =====================================================================

print("=" * 72)
print("RECUPERO DELLE STIME DI CONCENTRAZIONE NEL CAMPIONE")
print("=" * 72)

if (
    "estimate_df" in globals()
    and isinstance(estimate_df, pd.DataFrame)
    and not estimate_df.empty
):

    matrix_df = estimate_df.copy()

    print(
        "È stata utilizzata la variabile estimate_df "
        "presente nella memoria della sessione."
    )

else:

    if not CONCENTRATION_FILE.exists():
        raise FileNotFoundError(
            f"Non è disponibile la variabile estimate_df e non è "
            f"stato trovato il file '{CONCENTRATION_FILE.name}' in "
            "/content.\n\nEseguire prima la cella 4 nella stessa "
            "sessione Colab."
        )

    concentration_excel = pd.ExcelFile(
        CONCENTRATION_FILE
    )

    if CONCENTRATION_SHEET not in concentration_excel.sheet_names:
        raise ValueError(
            f"Nel file '{CONCENTRATION_FILE.name}' manca il foglio "
            f"'{CONCENTRATION_SHEET}'."
        )

    matrix_df = pd.read_excel(
        CONCENTRATION_FILE,
        sheet_name=CONCENTRATION_SHEET,
        dtype={CAS_COLUMN: str}
    )

    print(
        f"È stato letto il file '{CONCENTRATION_FILE.name}', "
        f"foglio '{CONCENTRATION_SHEET}'."
    )

print("Righe disponibili:", len(matrix_df))

# =====================================================================
# 3B. VERIFICA MODALITÀ — LA CELLA 5 NON SERVE IN MODALITÀ "headspace"
# =====================================================================

if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE == "headspace":

    print("=" * 72)
    print("MODALITÀ 'HEADSPACE' RILEVATA (impostata nella cella 4)")
    print("=" * 72)
    print(
        "Le concentrazioni stimate nella cella 4 sono già espresse in "
        "aria (µg/m3) e già confrontabili con la soglia olfattiva. "
        "La cella 5 serve solo per la modalità 'campione'."
    )

    raise SystemExit(
        "Interruzione volontaria: non è un errore. "
        "In modalità 'headspace' non è necessario eseguire le celle "
        "5b, 5c, 5d, 5e — puoi ignorarle e passare oltre."
    )

# =====================================================================
# 5. CONTROLLO DELLE COLONNE E DELL'UNITÀ DI MISURA
# =====================================================================

required_matrix_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    MATRIX_LOG_CONCENTRATION_COLUMN,
    MATRIX_LOG_SD_COLUMN,
    MATRIX_CONCENTRATION_UNIT_COLUMN
]

check_required_columns(
    matrix_df,
    required_matrix_columns,
    CONCENTRATION_SHEET
)

matrix_df[CAS_COLUMN] = (
    matrix_df[CAS_COLUMN]
    .apply(normalize_cas)
)

wrong_unit_mask = (
    matrix_df[MATRIX_CONCENTRATION_UNIT_COLUMN].notna()
    & (
        matrix_df[MATRIX_CONCENTRATION_UNIT_COLUMN].astype(str).str.strip()
        != REQUIRED_MATRIX_CONCENTRATION_UNIT
    )
)

if wrong_unit_mask.any():

    wrong_units = (
        matrix_df.loc[wrong_unit_mask, MATRIX_CONCENTRATION_UNIT_COLUMN]
        .unique()
        .tolist()
    )

    print()
    print(
        "ATTENZIONE: le seguenti righe hanno un'unità di "
        f"concentrazione diversa da '{REQUIRED_MATRIX_CONCENTRATION_UNIT}' "
        "e non verranno convertite in aria:"
    )
    for unit in wrong_units:
        print(" -", unit)

if THRESHOLD_COLUMN not in matrix_df.columns:
    print()
    print(
        f"ATTENZIONE: la colonna '{THRESHOLD_COLUMN}' non è presente. "
        "Il confronto con la soglia olfattiva non sarà calcolato."
    )
    matrix_df[THRESHOLD_COLUMN] = np.nan


# =====================================================================
# 6. RECUPERO AUTOMATICO DEL FOGLIO METODO_ANALITICO
# =====================================================================

print()
print("=" * 72)
print("PARAMETRI DEL METODO ANALITICO")
print("=" * 72)

method_source_file = find_file_with_required_sheets(
    WORKING_DIRECTORY,
    required_sheets={METHOD_SHEET_NAME},
    exclude_names={CONCENTRATION_FILE.name, OUTPUT_FILE.name}
)

if method_source_file is None:

    print(
        f"Nessun file con il foglio '{METHOD_SHEET_NAME}' è stato "
        "trovato in /content."
    )
    print(
        "Caricare il file GC-MS di origine (lo stesso della cella 2), "
        f"contenente il foglio '{METHOD_SHEET_NAME}'."
    )

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            "È necessario caricare un solo file Excel."
        )

    method_source_file = WORKING_DIRECTORY / next(iter(uploaded))

print("File del metodo analitico utilizzato:", method_source_file.name)

method_raw_df = pd.read_excel(
    method_source_file,
    sheet_name=METHOD_SHEET_NAME,
    header=None
)

method_parameters = parse_method_sheet(method_raw_df)

equilibrium_temperature_k = (
    method_parameters["equilibrium_temperature_c"] + 273.15
)

water_fraction = method_parameters["water_percent"] / 100.0
lipid_fraction = method_parameters["lipid_percent"] / 100.0

print()
print("Parametri riconosciuti:")
print(" - Contenuto d'acqua:", water_fraction * 100, "%")
print(" - Contenuto lipidico:", lipid_fraction * 100, "%")
print(
    " - Temperatura di equilibrio:",
    method_parameters["equilibrium_temperature_c"], "°C"
)
print(
    " - Massa campione (informativo):",
    method_parameters["sample_mass_g"], "g"
)
print(
    " - Volume headspace (informativo):",
    method_parameters["headspace_volume_ml"], "mL"
)
print(
    " - Densità campione (informativo):",
    method_parameters["sample_density_g_ml"], "g/mL"
)
print(
    " - Diluizione (informativo):",
    method_parameters["dilution_text"]
)
print(
    " - Fibra SPME (informativo):",
    method_parameters["spme_fiber"]
)
print(
    " - Splittaggio (informativo, non usato nel calcolo):",
    method_parameters["split_ratio"]
)
print(
    " - Detector (informativo, non usato nel calcolo):",
    method_parameters["detector_type"]
)

if water_fraction + lipid_fraction > 1.0:
    print()
    print(
        "ATTENZIONE: la somma di contenuto d'acqua e contenuto "
        "lipidico supera il 100%. Verificare i valori nel foglio "
        f"'{METHOD_SHEET_NAME}'."
    )


# =====================================================================
# 7. PARAMETRI FISICO-CHIMICI PER COMPOSTO (CON RICERCA AUTOMATICA)
# =====================================================================

import json
import time
import getpass
from datetime import date
from typing import Optional

from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import userdata

PHYSICOCHEMICAL_MODEL = "gpt-5.6"
PHYSICOCHEMICAL_PAUSE_SECONDS = 1.0
PHYSICOCHEMICAL_MAX_RETRIES = 2

PHYSICOCHEMICAL_CACHE_FILE = (
    WORKING_DIRECTORY / "Physicochemical_Cache.xlsx"
)

PHYSICOCHEMICAL_WEB_SEARCH_DOMAINS = [
    "pubchem.ncbi.nlm.nih.gov",
    "webbook.nist.gov",
    "thegoodscentscompany.com",
    "sciencedirect.com",
    "acs.org",
    "springer.com",
    "wiley.com",
    "tandfonline.com",
    "en.wikipedia.org"
]

print()
print("=" * 72)
print("PARAMETRI FISICO-CHIMICI PER COMPOSTO")
print("=" * 72)

# ---------------------------------------------------------------------
# 7a. Chiave API OpenAI
# ---------------------------------------------------------------------

try:
    physico_api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    physico_api_key = None

if not physico_api_key:
    print("Il Secret OPENAI_API_KEY non è stato trovato o non è accessibile.")
    try:
        physico_api_key = getpass.getpass(
            "Inserire la chiave API OpenAI. "
            "La chiave non sarà visualizzata: "
        )
    except Exception as error:
        print(f"getpass non disponibile in questo ambiente ({error}).")
        physico_api_key = None

    if not isinstance(physico_api_key, str) or not physico_api_key.strip():
        physico_api_key = input("Chiave API OpenAI: ")

if not isinstance(physico_api_key, str) or not physico_api_key.strip():
    raise ValueError("La chiave API OpenAI non è disponibile.")

physico_client = OpenAI(api_key=physico_api_key.strip())

print("Chiave API caricata correttamente.")


# ---------------------------------------------------------------------
# 7b. Identità PubChem (stessa funzione già usata nella cella 4)
# ---------------------------------------------------------------------

def get_pubchem_identity_for_physico(cas_number):

    encoded_cas = requests.utils.quote(str(cas_number), safe="")

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        f"compound/name/{encoded_cas}/property/"
        "IUPACName,MolecularFormula,MolecularWeight,"
        "CanonicalSMILES/JSON"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code == 404:
            return {}

        response.raise_for_status()
        payload = response.json()
        properties = payload.get("PropertyTable", {}).get("Properties", [])

        if not properties:
            return {}

        record = properties[0]
        cid = record.get("CID")

        return {
            "PubChem_CID": cid,
            "PubChem_IUPAC_name": record.get("IUPACName"),
            "PubChem_formula": record.get("MolecularFormula"),
            "PubChem_MW": record.get("MolecularWeight"),
            "PubChem_SMILES": record.get("CanonicalSMILES"),
            "PubChem_URL": (
                f"https://pubchem.ncbi.nlm.nih.gov/compound/{cid}"
                if cid is not None else None
            ),
        }

    except Exception as error:
        return {"PubChem_error": str(error)}


# ---------------------------------------------------------------------
# 7c. Struttura della risposta del modello e funzione di ricerca
# ---------------------------------------------------------------------

class PhysicochemicalRecord(BaseModel):

    cas_input: str
    iupac_input: str

    identity_match: str = Field(
        description="Valori consentiti: confirmed, probable, conflicting, not_found"
    )

    henry_constant_value: Optional[float] = Field(
        default=None,
        description="Valore della costante di Henry, se reperita"
    )
    henry_constant_unit: Optional[str] = Field(
        default=None,
        description="Unità: 'Pa*m3/mol', 'atm*m3/mol', oppure 'dimensionless'"
    )
    logp_value: Optional[float] = Field(
        default=None,
        description="log10(Kow), coefficiente di ripartizione ottanolo/acqua"
    )
    volatility_class: Optional[str] = Field(
        default=None,
        description=(
            "Da usare SOLO come fallback se la costante di Henry non è "
            "reperibile con confidenza sufficiente. Uno tra: "
            "'molto volatile', 'volatile', 'moderatamente volatile', "
            "'poco volatile', 'praticamente non volatile'"
        )
    )

    source_name: Optional[str] = None
    source_url: Optional[str] = None

    confidence: str = Field(description="Valori consentiti: high, medium, low")
    review_required: bool = False
    notes: Optional[str] = None


def search_physicochemical_online(analyte_cas, analyte_iupac, analyte_pubchem):

    analyte_context = json.dumps(analyte_pubchem, ensure_ascii=False, indent=2)

    system_prompt = """
Sei un chimico esperto di proprietà fisico-chimiche di composti
volatili (costante di Henry, logP), applicate a modelli di
ripartizione matrice/aria in analisi HS-SPME-GC-MS.

Recupera, per il composto indicato, la costante di Henry
(aria/acqua) e il logP, da fonti scientifiche affidabili
(letteratura peer-reviewed, NIST WebBook, database EPA/EPI Suite).

REGOLE OBBLIGATORIE:
- Verifica che il CAS e il nome forniti corrispondano alla stessa
  sostanza (usa identity_match).
- Specifica sempre l'unità della costante di Henry esattamente come
  'Pa*m3/mol', 'atm*m3/mol', oppure 'dimensionless'.
- Se non trovi un valore affidabile di Henry, lascialo vuoto e
  fornisci invece una stima di volatility_class basata su struttura
  chimica, punto di ebollizione noto e comportamento atteso, con
  bassa confidenza.
- Non inventare fonti o URL.
- Imposta review_required = true se la confidenza è bassa o i dati
  sono discordanti tra fonti diverse.
"""

    user_prompt = f"""
Nome IUPAC fornito:
{analyte_iupac}

CAS fornito:
{analyte_cas}

Informazioni preliminari da PubChem:
{analyte_context}

Recupera costante di Henry e logP (o, in mancanza, una classe di
volatilità) per questo composto.
"""

    response = physico_client.responses.parse(
        model=PHYSICOCHEMICAL_MODEL,
        tools=[
            {
                "type": "web_search",
                "filters": {"allowed_domains": PHYSICOCHEMICAL_WEB_SEARCH_DOMAINS}
            }
        ],
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        text_format=PhysicochemicalRecord
    )

    if response.output_parsed is None:
        raise ValueError("La risposta API non contiene un record strutturato.")

    return response.output_parsed


# ---------------------------------------------------------------------
# 7d. Elenco degli analiti presenti nella tabella di concentrazione
# ---------------------------------------------------------------------

analytes_needing_physico = (
    matrix_df[[CAS_COLUMN, IUPAC_COLUMN]]
    .dropna(subset=[CAS_COLUMN, IUPAC_COLUMN])
    .drop_duplicates(subset=[CAS_COLUMN])
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------
# 7e. Ricerca di un file fisico-chimico già esistente (facoltativo)
# ---------------------------------------------------------------------

physicochemical_file = find_file_with_required_sheets(
    WORKING_DIRECTORY,
    required_sheets={PHYSICOCHEMICAL_SHEET_NAME},
    exclude_names={
        CONCENTRATION_FILE.name,
        OUTPUT_FILE.name,
        method_source_file.name
    }
)

if physicochemical_file is not None:

    print("File dei parametri fisico-chimici trovato:", physicochemical_file.name)

    physicochemical_df = pd.read_excel(
        physicochemical_file,
        sheet_name=PHYSICOCHEMICAL_SHEET_NAME,
        dtype={CAS_COLUMN: str}
    )

else:

    print(
        f"Nessun file con il foglio '{PHYSICOCHEMICAL_SHEET_NAME}' "
        "trovato in /content."
    )
    print(
        "Se disponi già di un file (anche solo parziale, con dati per "
        "alcuni analiti) caricalo ora. Se non lo carichi (premi "
        "'Annulla'/'Cancel upload'), tutti i parametri verranno "
        "ricercati automaticamente online."
    )

    uploaded_physico = files.upload()

    if len(uploaded_physico) > 0:

        physico_filename = next(iter(uploaded_physico))

        physicochemical_df = pd.read_excel(
            physico_filename,
            sheet_name=PHYSICOCHEMICAL_SHEET_NAME,
            dtype={CAS_COLUMN: str}
        )

        print("File caricato:", physico_filename)

    else:

        print("Nessun file caricato: si procede con la ricerca completa.")

        physicochemical_df = pd.DataFrame(
            columns=[
                CAS_COLUMN, HENRY_VALUE_COLUMN, HENRY_UNIT_COLUMN,
                LOGP_COLUMN, VOLATILITY_CLASS_COLUMN
            ]
        )

for column in [HENRY_VALUE_COLUMN, HENRY_UNIT_COLUMN, LOGP_COLUMN, VOLATILITY_CLASS_COLUMN]:
    if column not in physicochemical_df.columns:
        physicochemical_df[column] = np.nan

physicochemical_df[CAS_COLUMN] = (
    physicochemical_df[CAS_COLUMN].apply(normalize_cas)
)


# ---------------------------------------------------------------------
# 7f. Determinazione degli analiti mancanti
# ---------------------------------------------------------------------

known_cas = set(physicochemical_df[CAS_COLUMN].dropna())

analytes_to_search = analytes_needing_physico.loc[
    ~analytes_needing_physico[CAS_COLUMN].isin(known_cas)
].reset_index(drop=True)

print()
print(f"Analiti con parametri già disponibili: {len(known_cas)}")
print(f"Analiti da ricercare online: {len(analytes_to_search)}")


# ---------------------------------------------------------------------
# 7g. Ricerca online per gli analiti mancanti
# ---------------------------------------------------------------------

new_physico_records = []

for index, row in analytes_to_search.iterrows():

    analyte_cas = row[CAS_COLUMN]
    analyte_iupac = row[IUPAC_COLUMN]

    print()
    print("=" * 72)
    print(f"[{index + 1}/{len(analytes_to_search)}] {analyte_iupac} — {analyte_cas}")

    analyte_pubchem = get_pubchem_identity_for_physico(analyte_cas)

    result = None
    last_error = None

    for attempt in range(1, PHYSICOCHEMICAL_MAX_RETRIES + 1):
        try:
            if attempt > 1:
                print(f"Nuovo tentativo API ({attempt}/{PHYSICOCHEMICAL_MAX_RETRIES})...")

            result = search_physicochemical_online(
                analyte_cas, analyte_iupac, analyte_pubchem
            )
            break

        except Exception as error:
            last_error = error
            print(f"Errore nella chiamata API: {error}")
            time.sleep(PHYSICOCHEMICAL_PAUSE_SECONDS)

    if result is None:
        print(f"Impossibile ottenere dati per questo composto: {last_error}")
        new_physico_records.append({
            CAS_COLUMN: analyte_cas,
            IUPAC_COLUMN: analyte_iupac,
            HENRY_VALUE_COLUMN: np.nan,
            HENRY_UNIT_COLUMN: None,
            LOGP_COLUMN: np.nan,
            VOLATILITY_CLASS_COLUMN: None,
            "Source_name": None, "Source_url": None,
            "Confidence": "low", "Review_required": True,
            "Notes": f"Errore API: {last_error}",
            "Retrieved_on": str(date.today())
        })
        time.sleep(PHYSICOCHEMICAL_PAUSE_SECONDS)
        continue

    print("Henry:", result.henry_constant_value, result.henry_constant_unit)
    print("LogP:", result.logp_value, "| Volatility_class:", result.volatility_class)

    new_physico_records.append({
        CAS_COLUMN: analyte_cas,
        IUPAC_COLUMN: analyte_iupac,
        HENRY_VALUE_COLUMN: result.henry_constant_value,
        HENRY_UNIT_COLUMN: result.henry_constant_unit,
        LOGP_COLUMN: result.logp_value,
        VOLATILITY_CLASS_COLUMN: result.volatility_class,
        "Source_name": result.source_name,
        "Source_url": result.source_url,
        "Confidence": result.confidence,
        "Review_required": result.review_required,
        "Notes": result.notes,
        "Retrieved_on": str(date.today())
    })

    time.sleep(PHYSICOCHEMICAL_PAUSE_SECONDS)


# ---------------------------------------------------------------------
# 7h. Unione, salvataggio e download condizionato
# ---------------------------------------------------------------------

new_physico_df = pd.DataFrame(new_physico_records)
physico_new_added = len(new_physico_df) > 0

if physico_new_added:
    physicochemical_df = pd.concat(
        [physicochemical_df, new_physico_df], ignore_index=True, sort=False
    ).drop_duplicates(subset=[CAS_COLUMN], keep="last").reset_index(drop=True)

    # ---------------------------------------------------------------------
# 7h-bis. COLONNE DERIVATE RICHIESTE DAL RESTO DELLA CELLA
# ---------------------------------------------------------------------

physicochemical_df["Volatility_class_normalized"] = (
    physicochemical_df[VOLATILITY_CLASS_COLUMN]
    .apply(normalize_category_text)
)

physicochemical_df["Henry_dimensionless"] = (
    physicochemical_df.apply(
        lambda row: convert_henry_to_dimensionless(
            row[HENRY_VALUE_COLUMN],
            row[HENRY_UNIT_COLUMN],
            equilibrium_temperature_k
        ),
        axis=1
    )
)

print(
    "Composti con parametri fisico-chimici disponibili:",
    len(physicochemical_df)
)

physicochemical_source_name = (
    physicochemical_file.name
    if physicochemical_file is not None
    else PHYSICOCHEMICAL_CACHE_FILE.name
)

with pd.ExcelWriter(PHYSICOCHEMICAL_CACHE_FILE, engine="xlsxwriter") as writer:
    physicochemical_df.to_excel(writer, sheet_name=PHYSICOCHEMICAL_SHEET_NAME, index=False)

if physico_new_added:
    print()
    print(f"Aggiunti {len(new_physico_df)} nuovi composti. Download in corso...")
    files.download(str(PHYSICOCHEMICAL_CACHE_FILE))
else:
    print()
    print("Nessun nuovo composto aggiunto: download non ripetuto.")

# =====================================================================
# 8. ABBINAMENTO CON LE STIME DI CONCENTRAZIONE
# =====================================================================

air_df = matrix_df.merge(
    physicochemical_df[
        [
            CAS_COLUMN,
            HENRY_VALUE_COLUMN,
            HENRY_UNIT_COLUMN,
            "Henry_dimensionless",
            LOGP_COLUMN,
            "Volatility_class_normalized"
        ]
    ],
    on=CAS_COLUMN,
    how="left"
)
if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE == "headspace":
    raise SystemExit()
# =====================================================================
# 9. METODO DI CONVERSIONE UTILIZZATO PER COMPOSTO
# =====================================================================

air_df["Transfer_method"] = air_df.apply(
    lambda row: classify_transfer_method(
        row["Henry_dimensionless"],
        row[LOGP_COLUMN],
        row["Volatility_class_normalized"]
    ),
    axis=1
)

transfer_method_labels = {
    "rigoroso": (
        "Rigoroso (costante di Henry + logP, "
        "correzione acqua/lipide completa)"
    ),
    "acqua_solo": (
        "Approssimato (solo costante di Henry, fase lipidica non "
        "corretta: logP mancante — possibile SOVRAstima per composti "
        "lipofili, che il modello non riconosce come trattenuti "
        "nella fase grassa)"
    ),
    "statistico": (
        "Statistico (prior di volatilità, nessuna costante di Henry — "
        "valori provvisori non validati)"
    ),
    "non_calcolabile": (
        "Non calcolabile: mancano sia la costante di Henry sia "
        "un livello di volatilità riconosciuto"
    ),
}

air_df["Transfer_method_label"] = air_df["Transfer_method"].map(
    transfer_method_labels
)


# =====================================================================
# 10. CALCOLO DI log10(K_MATRICE->ARIA) E RELATIVA INCERTEZZA
# =====================================================================

lipid_density = DEFAULT_LIPID_DENSITY_KG_M3

specific_volume_water = water_fraction / WATER_DENSITY_KG_M3
specific_volume_lipid = lipid_fraction / lipid_density


def compute_log10_k_matrix_air(row):
    """
    Calcola log10(K_matrice->aria) e la relativa deviazione standard
    (log10), in base al metodo di conversione disponibile per la
    riga (composto).
    """

    method = row["Transfer_method"]

    if method == "rigoroso":

        k_ow = 10 ** row[LOGP_COLUMN]

        denominator = (
            specific_volume_water
            + k_ow * specific_volume_lipid
        )

        if denominator <= 0 or row["Henry_dimensionless"] <= 0:
            return np.nan, np.nan

        log10_k = (
            math.log10(row["Henry_dimensionless"])
            - math.log10(denominator)
        )

        return log10_k, RIGOROUS_RESIDUAL_LOG_SD

    if method == "acqua_solo":

        if (
            specific_volume_water <= 0
            or row["Henry_dimensionless"] <= 0
        ):
            return np.nan, np.nan

        log10_k = (
            math.log10(row["Henry_dimensionless"])
            - math.log10(specific_volume_water)
        )

        return log10_k, WATER_ONLY_RESIDUAL_LOG_SD

    if method == "statistico":

        prior = VOLATILITY_CLASS_PRIOR_TABLE[
            row["Volatility_class_normalized"]
        ]

        return prior["mu"], prior["sigma"]

    return np.nan, np.nan


k_results = air_df.apply(
    compute_log10_k_matrix_air,
    axis=1,
    result_type="expand"
)

air_df["Log10_K_matrix_air"] = k_results[0]
air_df["Log10_K_matrix_air_sd"] = k_results[1]


# =====================================================================
# 11. CONCENTRAZIONE STIMATA IN ARIA (MEDIANA E INTERVALLO CREDIBILE)
# =====================================================================

data_available = (
    matrix_df[MATRIX_CONCENTRATION_UNIT_COLUMN].astype(str).str.strip()
    == REQUIRED_MATRIX_CONCENTRATION_UNIT
) if False else (
    air_df[MATRIX_CONCENTRATION_UNIT_COLUMN].astype(str).str.strip()
    == REQUIRED_MATRIX_CONCENTRATION_UNIT
)

data_available = (
    data_available
    & air_df[MATRIX_LOG_CONCENTRATION_COLUMN].notna()
    & air_df[MATRIX_LOG_SD_COLUMN].notna()
    & air_df["Log10_K_matrix_air"].notna()
    & air_df["Log10_K_matrix_air_sd"].notna()
)

air_df["Air_concentration_calculable"] = np.where(
    data_available,
    "Sì",
    "No"
)

air_df["Estimated_log10_air_concentration"] = np.where(
    data_available,

    air_df[MATRIX_LOG_CONCENTRATION_COLUMN]
    + air_df["Log10_K_matrix_air"],

    np.nan
)

air_df["Estimated_log10_air_sd"] = np.where(
    data_available,

    np.sqrt(
        air_df[MATRIX_LOG_SD_COLUMN] ** 2
        + air_df["Log10_K_matrix_air_sd"] ** 2
    ),

    np.nan
)

z_score = norm.ppf(
    0.5 + CREDIBLE_INTERVAL_PROBABILITY / 2.0
)

air_df["Estimated_air_concentration_median_ug_m3"] = np.where(
    air_df["Estimated_log10_air_concentration"].notna(),
    10 ** air_df["Estimated_log10_air_concentration"],
    np.nan
)

air_df["Estimated_air_concentration_low_ug_m3"] = np.where(
    air_df["Estimated_log10_air_concentration"].notna(),

    10 ** (
        air_df["Estimated_log10_air_concentration"]
        - z_score * air_df["Estimated_log10_air_sd"]
    ),

    np.nan
)

air_df["Estimated_air_concentration_high_ug_m3"] = np.where(
    air_df["Estimated_log10_air_concentration"].notna(),

    10 ** (
        air_df["Estimated_log10_air_concentration"]
        + z_score * air_df["Estimated_log10_air_sd"]
    ),

    np.nan
)
if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE == "headspace":
    raise SystemExit()
# =====================================================================
# 12. CONFRONTO CON LA SOGLIA OLFATTIVA
# =====================================================================

has_threshold = (
    air_df[THRESHOLD_COLUMN].notna()
    & (air_df[THRESHOLD_COLUMN] > 0)
    & air_df["Estimated_log10_air_concentration"].notna()
)

log10_threshold = np.log10(
    air_df[THRESHOLD_COLUMN].where(
        air_df[THRESHOLD_COLUMN] > 0
    )
)

air_df["Log10_ratio_to_odor_threshold"] = np.where(
    has_threshold,
    air_df["Estimated_log10_air_concentration"] - log10_threshold,
    np.nan
)

z_threshold = (
    air_df["Estimated_log10_air_concentration"] - log10_threshold
) / air_df["Estimated_log10_air_sd"]

air_df["Probability_above_odor_threshold"] = np.where(
    has_threshold,
    norm.cdf(z_threshold),
    np.nan
)


# =====================================================================
# 13. NOTE DI ELABORAZIONE E TABELLA DI REVISIONE
# =====================================================================

air_df["Processing_note"] = ""

air_df.loc[
    air_df[MATRIX_CONCENTRATION_UNIT_COLUMN].astype(str).str.strip()
    != REQUIRED_MATRIX_CONCENTRATION_UNIT,
    "Processing_note"
] = (
    "Unità della concentrazione nel campione diversa da "
    f"'{REQUIRED_MATRIX_CONCENTRATION_UNIT}': riga non convertita"
)

air_df.loc[
    (air_df["Processing_note"] == "")
    & (air_df["Transfer_method"] == "non_calcolabile"),
    "Processing_note"
] = (
    "Nessuna costante di Henry né livello di volatilità "
    "riconosciuto per questo composto"
)

air_df.loc[
    (air_df["Processing_note"] == "")
    & (air_df["Transfer_method"] == "acqua_solo"),
    "Processing_note"
] = (
    "LogP mancante: correzione per il contenuto lipidico non "
    "applicata (possibile SOVRAstima per composti lipofili, "
    "non riconosciuti come trattenuti nella fase grassa)"
)

air_df.loc[
    (air_df["Processing_note"] == "")
    & (air_df["Transfer_method"] == "statistico"),
    "Processing_note"
] = (
    "Coefficiente di ripartizione stimato da prior di volatilità "
    "non validati: interpretare con cautela"
)

air_df.loc[
    (air_df["Processing_note"] == "")
    & (air_df[MATRIX_LOG_CONCENTRATION_COLUMN].isna()),
    "Processing_note"
] = "Concentrazione nel campione non calcolabile (vedi cella 4)"

air_df.loc[
    (air_df["Processing_note"] == "")
    & (air_df[THRESHOLD_COLUMN].isna()),
    "Processing_note"
] = "Soglia olfattiva in aria non disponibile per il confronto"


# =====================================================================
# 14. ORDINE DELLE COLONNE
# =====================================================================

output_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    IUPAC_COLUMN,
    CAS_COLUMN,
    MATRIX_LOG_CONCENTRATION_COLUMN,
    MATRIX_LOG_SD_COLUMN,
    MATRIX_CONCENTRATION_UNIT_COLUMN,
    HENRY_VALUE_COLUMN,
    HENRY_UNIT_COLUMN,
    "Henry_dimensionless",
    LOGP_COLUMN,
    "Volatility_class_normalized",
    "Transfer_method_label",
    "Log10_K_matrix_air",
    "Log10_K_matrix_air_sd",
    "Air_concentration_calculable",
    "Estimated_log10_air_concentration",
    "Estimated_log10_air_sd",
    "Estimated_air_concentration_median_ug_m3",
    "Estimated_air_concentration_low_ug_m3",
    "Estimated_air_concentration_high_ug_m3",
    THRESHOLD_COLUMN,
    "Log10_ratio_to_odor_threshold",
    "Probability_above_odor_threshold",
    "Processing_note"
]

output_columns = [
    column
    for column in output_columns
    if column in air_df.columns
]

air_df = air_df[output_columns]

air_df = air_df.sort_values(
    by=[SAMPLE_COLUMN, COMPOUND_COLUMN],
    ascending=[True, True]
).reset_index(drop=True)


# =====================================================================
# 15. TABELLA DI REVISIONE MANUALE
# =====================================================================

review_mask = (
    (air_df["Air_concentration_calculable"] == "No")
    | (air_df["Processing_note"] != "")
)

review_df = air_df.loc[review_mask].copy()


# =====================================================================
# 16. TABELLA DEI PARAMETRI DEL METODO E DEI PRIOR UTILIZZATI
# =====================================================================

method_parameters_df = pd.DataFrame(
    {
        "Parametro": [
            "File del metodo analitico",
            "Contenuto d'acqua [%]",
            "Contenuto lipidico [%]",
            "Densità acqua assunta [kg/m3]",
            "Densità lipide assunta [kg/m3]",
            "Temperatura di equilibrio [°C]",
            "Massa campione [g] (informativo)",
            "Volume headspace [mL] (informativo)",
            "Densità campione [g/mL] (informativo)",
            "Diluizione (informativo)",
            "Fibra SPME (informativo)",
            "Splittaggio (informativo, non usato nel calcolo)",
            "Detector (informativo, non usato nel calcolo)",
        ],
        "Valore": [
            method_source_file.name,
            water_fraction * 100.0,
            lipid_fraction * 100.0,
            WATER_DENSITY_KG_M3,
            lipid_density,
            method_parameters["equilibrium_temperature_c"],
            method_parameters["sample_mass_g"],
            method_parameters["headspace_volume_ml"],
            method_parameters["sample_density_g_ml"],
            method_parameters["dilution_text"],
            method_parameters["spme_fiber"],
            method_parameters["split_ratio"],
            method_parameters["detector_type"],
        ]
    }
)

volatility_priors_df = pd.DataFrame(
    [
        {
            "Volatility_class": level,
            "Log10_K_prior_mu": values["mu"],
            "Log10_K_prior_sigma": values["sigma"]
        }
        for level, values in VOLATILITY_CLASS_PRIOR_TABLE.items()
    ]
)

volatility_priors_df["Nota"] = (
    "Valori provvisori, non validati sperimentalmente"
)


# =====================================================================
# 17. TABELLA DI RIEPILOGO
# =====================================================================

summary_df = pd.DataFrame(
    {
        "Indicatore": [
            "File dei parametri fisico-chimici utilizzato",
            "Combinazioni campione-composto elaborate",
            "Concentrazioni in aria calcolabili",
            "Conversioni con metodo rigoroso (Henry + logP)",
            "Conversioni con solo Henry (lipide non corretto)",
            "Conversioni con prior statistico di volatilità",
            "Confronti con soglia olfattiva disponibili",
            "Record da revisionare",
            "Livello dell'intervallo credibile"
        ],
        "Valore": [
            physicochemical_source_name,
            len(air_df),
            int(
                (air_df["Air_concentration_calculable"] == "Sì")
                .sum()
            ),
            int(
                (air_df["Transfer_method_label"]
                 == transfer_method_labels["rigoroso"]).sum()
            ),
            int(
                (air_df["Transfer_method_label"]
                 == transfer_method_labels["acqua_solo"]).sum()
            ),
            int(
                (air_df["Transfer_method_label"]
                 == transfer_method_labels["statistico"]).sum()
            ),
            int(
                air_df["Probability_above_odor_threshold"]
                .notna()
                .sum()
            ),
            len(review_df),
            f"{int(CREDIBLE_INTERVAL_PROBABILITY * 100)}%"
        ]
    }
)
if "COMPOSITION_MODE" in globals() and COMPOSITION_MODE == "headspace":
    raise SystemExit()

# =====================================================================
# 18. SALVATAGGIO CON XLSXWRITER
# =====================================================================

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="xlsxwriter"
) as writer:

    air_df.to_excel(
        writer,
        sheet_name="Air_Concentration_Estimates",
        index=False
    )

    method_parameters_df.to_excel(
        writer,
        sheet_name="Method_Parameters_Used",
        index=False
    )

    volatility_priors_df.to_excel(
        writer,
        sheet_name="Volatility_Priors_Used",
        index=False
    )

    review_df.to_excel(
        writer,
        sheet_name="Manual_Review",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Processing_Summary",
        index=False
    )

    workbook = writer.book

    header_format = workbook.add_format(
        {
            "bold": True,
            "bg_color": "#D9EAF7",
            "border": 1,
            "text_wrap": True,
            "valign": "top"
        }
    )

    wrap_format = workbook.add_format(
        {
            "text_wrap": True,
            "valign": "top"
        }
    )

    decimal_format = workbook.add_format(
        {
            "num_format": "0.000",
            "valign": "top"
        }
    )

    scientific_format = workbook.add_format(
        {
            "num_format": "0.000E+00",
            "valign": "top"
        }
    )

    percent_format = workbook.add_format(
        {
            "num_format": "0.0%",
            "valign": "top"
        }
    )

    dataframe_by_sheet = {
        "Air_Concentration_Estimates": air_df,
        "Method_Parameters_Used": method_parameters_df,
        "Volatility_Priors_Used": volatility_priors_df,
        "Manual_Review": review_df,
        "Processing_Summary": summary_df
    }

    for sheet_name, dataframe in dataframe_by_sheet.items():

        worksheet = writer.sheets[sheet_name]

        worksheet.freeze_panes(1, 0)
        worksheet.set_default_row(28)

        if len(dataframe.columns) > 0:
            worksheet.autofilter(
                0,
                0,
                max(len(dataframe), 1),
                len(dataframe.columns) - 1
            )

        for column_index, column_name in enumerate(
            dataframe.columns
        ):

            worksheet.write(
                0,
                column_index,
                column_name,
                header_format
            )

            width = calculate_column_width(
                dataframe,
                column_name
            )

            worksheet.set_column(
                column_index,
                column_index,
                width,
                wrap_format
            )

    air_ws = writer.sheets["Air_Concentration_Estimates"]

    for column_name in [
        "Estimated_air_concentration_median_ug_m3",
        "Estimated_air_concentration_low_ug_m3",
        "Estimated_air_concentration_high_ug_m3",
        THRESHOLD_COLUMN
    ]:
        if column_name in air_df.columns:

            column_index = air_df.columns.get_loc(column_name)

            air_ws.set_column(
                column_index,
                column_index,
                20,
                scientific_format
            )

    for column_name in [
        MATRIX_LOG_CONCENTRATION_COLUMN,
        MATRIX_LOG_SD_COLUMN,
        "Log10_K_matrix_air",
        "Log10_K_matrix_air_sd",
        "Estimated_log10_air_concentration",
        "Estimated_log10_air_sd",
        "Log10_ratio_to_odor_threshold"
    ]:
        if column_name in air_df.columns:

            column_index = air_df.columns.get_loc(column_name)

            air_ws.set_column(
                column_index,
                column_index,
                18,
                decimal_format
            )

    if "Probability_above_odor_threshold" in air_df.columns:

        column_index = air_df.columns.get_loc(
            "Probability_above_odor_threshold"
        )

        air_ws.set_column(
            column_index,
            column_index,
            16,
            percent_format
        )

    if (
        "Log10_ratio_to_odor_threshold" in air_df.columns
        and len(air_df) > 0
    ):

        ratio_col = air_df.columns.get_loc(
            "Log10_ratio_to_odor_threshold"
        )

        air_ws.conditional_format(
            1,
            ratio_col,
            len(air_df),
            ratio_col,
            {
                "type": "3_color_scale",
                "min_color": "#63BE7B",
                "mid_color": "#FFEB84",
                "max_color": "#F8696B"
            }
        )


# =====================================================================
# 19. RISULTATO E DOWNLOAD
# =====================================================================

print()
print("=" * 72)
print("CONVERSIONE CAMPIONE->ARIA COMPLETATA")
print("=" * 72)

print("File creato:", OUTPUT_FILE.name)
print("Righe elaborate:", len(air_df))

print(
    "Concentrazioni in aria calcolabili:",
    int(
        (air_df["Air_concentration_calculable"] == "Sì")
        .sum()
    )
)

print(
    "Confronti con soglia disponibili:",
    int(
        air_df["Probability_above_odor_threshold"]
        .notna()
        .sum()
    )
)

print(
    "Record da controllare manualmente:",
    len(review_df)
)

print()
print(
    "PROMEMORIA: la conversione campione->aria si basa su un modello "
    "di equilibrio a due fasi (acqua+lipide) con parametri in parte "
    "provvisori (in particolare per i composti privi di costante di "
    "Henry). Non è una misura diretta della concentrazione in aria."
)

print()
print("Anteprima dei risultati:")

preview_columns = [
    SAMPLE_COLUMN,
    COMPOUND_COLUMN,
    CAS_COLUMN,
    "Transfer_method_label",
    "Estimated_air_concentration_median_ug_m3",
    "Estimated_air_concentration_low_ug_m3",
    "Estimated_air_concentration_high_ug_m3",
    THRESHOLD_COLUMN,
    "Log10_ratio_to_odor_threshold",
    "Probability_above_odor_threshold",
    "Processing_note"
]

preview_columns = [
    column
    for column in preview_columns
    if column in air_df.columns
]

display(
    air_df[preview_columns].head(30)
)

files.download(
    str(OUTPUT_FILE)
)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 60.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
RECUPERO DELLE STIME DI CONCENTRAZIONE NEL CAMPIONE
È stata utilizzata la variabile estimate_df presente nella memoria della sessione.
Righe disponibili: 8

PARAMETRI DEL METODO ANALITICO
File del metodo analitico utilizzato: GCMS_Areas.xlsx

Parametri riconosciuti:
 - Contenuto d'acqua: 4.0 %
 - Contenuto lipidico: 75.0 %
 - Temperatura di equilibrio: 40.0 °C
 - Massa campione (informativo): 1.0 g
 - Volume headspace (informativo): 19.0 mL
 - Densità campione (informativo): 1.1 g/mL
 - Diluizione (informativo): puro (100%)
 - Fibra SPME (informativo): gray, trifasica

Saving Physicochemical_Cache (1).xlsx to Physicochemical_Cache (1).xlsx
File caricato: Physicochemical_Cache (1).xlsx

Analiti con parametri già disponibili: 4
Analiti da ricercare online: 0
Composti con parametri fisico-chimici disponibili: 4

Nessun nuovo composto aggiunto: download non ripetuto.

CONVERSIONE CAMPIONE->ARIA COMPLETATA
File creato: GCMS_Air_Concentration_Estimates.xlsx
Righe elaborate: 8
Concentrazioni in aria calcolabili: 8
Confronti con soglia disponibili: 8
Record da controllare manualmente: 0

PROMEMORIA: la conversione campione->aria si basa su un modello di equilibrio a due fasi (acqua+lipide) con parametri in parte provvisori (in particolare per i composti privi di costante di Henry). Non è una misura diretta della concentrazione in aria.

Anteprima dei risultati:


,Sample_ID,GCMS_column_name,CAS,Transfer_method_label,Estimated_air_concentration_median_ug_m3,Estimated_air_concentration_low_ug_m3,Estimated_air_concentration_high_ug_m3,Odor_threshold_air_ug_m3,Log10_ratio_to_odor_threshold,Probability_above_odor_threshold,Processing_note
0,Cacao_01,"2,3,5-trimethylpyrazine",14667-55-1,"Rigoroso (costante di Henry + logP, correzione...",0.849356,6.129487e-05,11769.432276,50.000000,-1.769880,0.241058,
1,Cacao_01,2-methoxyphenol,90-05-1,"Rigoroso (costante di Henry + logP, correzione...",0.016410,3.145330e-04,0.856128,0.084000,-0.709177,0.248504,
2,Cacao_01,linalool,78-70-6,"Rigoroso (costante di Henry + logP, correzione...",0.003408,1.606952e-06,7.228521,3.200000,-2.972624,0.070800,
3,Cacao_01,phenol,108-95-2,"Rigoroso (costante di Henry + logP, correzione...",0.002497,1.802245e-07,34.601123,23.094479,-3.966056,0.057614,
4,Cacao_02,"2,3,5-trimethylpyrazine",14667-55-1,"Rigoroso (costante di Henry + logP, correzione...",0.942291,6.800272e-05,13057.009826,50.000000,-1.724785,0.246673,
5,Cacao_02,2-methoxyphenol,90-05-1,"Rigoroso (costante di Henry + logP, correzione...",0.013346,2.558668e-04,0.696103,0.084000,-0.798935,0.222071,
6,Cacao_02,linalool,78-70-6,"Rigoroso (costante di Henry + logP, correzione...",0.003919,1.847648e-06,8.312557,3.200000,-2.911973,0.074954,
7,Cacao_02,phenol,108-95-2,"Rigoroso (costante di Henry + logP, correzione...",0.003410,2.460523e-07,47.252848,23.094479,-3.830780,0.064083,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>